In [ ]:
# Open Avatar Studio V1.1 | Continuity First + Speech Speed | Long Video | 1/3 - Talking Avatar
# ONE notebook, THREE independent code cells. / 一个 Notebook，三个独立 Cell。
# No fixed total script length, narration duration, or final frame count limit.
# 无固定总文案长度、总时长或总帧数上限；单片段仍有资源保护限制。
# Full LP run: enable all three stages, GPU T4 x2 + Internet On, Save Version -> Save & Run All.
# Ditto direct: only Stage 1 is needed; Cell 3 can recover interrupted final assembly.
# Cross-session recovery: keep the SAME private Dataset; rerun unfinished stage.
# Stage 1 recovery needs the ORIGINAL source files and generation settings.
# Stage 2/3 recovery only needs the private Dataset (API reads pinned history) + required models.
# Add Input: paste the full HTTPS URL (not just the slug).
# Ditto: https://www.kaggle.com/datasets/danielsundl/open-avatar-ditto
# Qwen (text modes): https://www.kaggle.com/datasets/danielsundl/open-avatar-qwen3-tts
# LivePortrait: https://www.kaggle.com/datasets/danielsundl/open-avatar-liveportrait
# Demo: https://www.kaggle.com/datasets/danielsundl/open-avatar-demo-assets
# Bounded inference runtime embedded; do NOT attach the Runtime Dataset.
# Copy to a PRIVATE notebook for personal inputs and token. Review model terms before running.
# Public template: all RUN_STAGE switches False, no personal paths or credentials.
# ===== USER CONFIGURATION / 用户配置 =====
RUN_STAGE_1 = True  # True: run/resume; False: fully skip.
ALLOW_NONCOMMERCIAL_RESEARCH = True  # True only for authorized research under model terms.
INPUT_MODE = 'demo_audio'  # demo_audio | demo_text | audio | text
LANGUAGE = 'Chinese'  # demo_text: English/Chinese; text: English/Chinese/Auto
DEMO_PORTRAIT = 'portrait_01.png'  # portrait_01.png | portrait_02.png
DEMO_AUDIO = 'demo_audio.wav'  # or native_blink_long_test.wav if expanded Demo Assets mounted
PORTRAIT = ''  # custom: exact input file path
AUDIO = ''  # audio: exact input audio path; no fixed total duration cap
SCRIPT = ''  # text: exact UTF-8 .txt file path; no fixed total character cap
PERSIST_DATASET = ''  # strongly recommended: owner/oas-checkpoints-unique-task (PRIVATE)
REFERENCE_AUDIO = ''  # text: authorized reference voice file, 1-30s
REFERENCE_TEXT = ''  # optional reference transcript .txt path (1-1000 chars), blank=x-vector only
TEXT_SEGMENT_CHARACTERS = 2400  # 600-4000 per durable text unit, NOT total. A unit may contain multiple TTS chunks.
TTS_CHUNK_CHARACTERS = 600  # 80-800 per Qwen call. 600 is the continuity-first default.
DITTO_SEGMENT_SECONDS = 600  # 10-600s per Ditto run / outer audio unit. 600s is unverified on real GPU.
DITTO_WARMUP_SECONDS = 1.2  # 0-3s of inference-only preroll; cropped away, no speech lost.
SMOOTH_SEAMS = True  # quiet-only optical-flow bridges, preserve active speech and frame count.
SPEECH_SPEED = 0.9  # 0.5-2.0; <1 slower, >1 faster. Pitch-preserving tempo BEFORE Ditto; 1.0 bypasses filter.
# V1.1 final has a new build identity. Use a NEW private Dataset for new jobs.
# Complete existing V1.0 or V1.1 RC journals with their matching Notebook.
EXPRESSION = 0.70  # 0.0-1.0 expression strength
BLINK_MODE = 'liveportrait'  # liveportrait | ditto_template | ditto_native
DITTO_EYE_OPEN_FRAMES = 50  # template only; 0=random, positive=open frames
SEED = 42  # base seed; derived deterministically per segment
# Example format: yourusername/oas-checkpoints-my-long-video
# Create if absent; same task resumes automatically. Keep EVERY historical version until verified final.
# Latest Dataset version includes oas-progress.json and only the newest artifact.
# Earlier checkpoints are in immutable historical versions; do not delete old versions.
# 每片段每阶段一个新版本，最新清单索引历史文件；不能只保留最新版本。
# For remote persistence: Internet On + KAGGLE_API_TOKEN in Secrets. Never paste token into code.
# Blank = local journal only (NOT durable across container loss). Never run concurrent writers.
STAGE1_RESTORE_MANIFEST = ''  # optional exact oas-progress.json path; remote manifest selects original Dataset
EXPECTED_TASK_ID = ''  # optional 64-hex long-task ID, prevents wrong-task writes
STAGE1_MAX_NEW_SEGMENTS = 0  # 0=all remaining; positive=limit newly committed segments this run, NOT total
STAGE1_MAX_RUN_SECONDS = 0  # 0=unlimited; positive=soft pause budget checked BETWEEN segments only
REMOTE_VERIFY_WAIT_SECONDS = 600  # retry deadline for one Dataset version to become downloadable
REMOTE_COMMAND_TIMEOUT_SECONDS = 3600  # timeout per upload/download command; raise if necessary
CHECK_NETWORK = True
MONITOR_RESOURCES = True
WORK_ROOT = '/kaggle/working/open-avatar-studio-v11'
# ===== END USER CONFIGURATION / DO NOT EDIT BELOW =====

if type(RUN_STAGE_1) is not bool:
    raise ValueError("RUN_STAGE_1 must be True or False")
if RUN_STAGE_1:
    # Generated, content-verified V1.1 runtime. Do not edit the embedded payload.
    import base64 as _b64, hashlib as _hash, io as _io, json as _json
    import os as _os, shutil as _shutil, stat as _stat, sys as _sys
    import tempfile as _tempfile, zipfile as _zipfile
    from pathlib import Path as _Path

    _OAS_SHA = 'f0ddb13b6d236d1724f5bd9b8137a561405a3a22b9abb34758c0f5a771f3c6cc'
    _OAS_BUNDLE = '...'

    def _oas_bootstrap():
        data = _b64.b64decode(_OAS_BUNDLE, validate=True)
        if _hash.sha256(data).hexdigest() != _OAS_SHA:
            raise RuntimeError('V1.1 bundle SHA256 mismatch')
        with _zipfile.ZipFile(_io.BytesIO(data)) as z:
            manifest = _json.loads(z.read('runtime_manifest.json'))
            if manifest['schema'] != 'oas-runtime-bundle-v01' or manifest['release'] != '1.1.0':
                raise RuntimeError('Wrong runtime release')
            names = set(manifest['files']) | {'runtime_manifest.json'}
            if len(z.infolist()) != len(names) or set(z.namelist()) != names:
                raise RuntimeError('Unexpected runtime archive members')
            for m in z.infolist():
                if '/' in m.filename or '\\' in m.filename or m.filename in ('.','..') or _stat.S_IFMT(m.external_attr >> 16) not in (0,_stat.S_IFREG):
                    raise RuntimeError('Unsafe runtime archive member')
            expected = dict(manifest['files'])
            mbytes = z.read('runtime_manifest.json')
            expected['runtime_manifest.json'] = {'size':len(mbytes),'sha256':_hash.sha256(mbytes).hexdigest()}
            parent = _Path('/tmp/open-avatar-studio');parent.mkdir(parents=True,exist_ok=True)
            folder = parent/('runtime-'+_OAS_SHA[:16])
            def verify(path):
                if path.is_symlink() or not path.is_dir() or set(x.name for x in path.iterdir()) != names:
                    raise RuntimeError('Unexpected runtime directory contents: '+str(path))
                for n,rec in expected.items():
                    f=path/n
                    if f.is_symlink() or not f.is_file() or f.stat().st_size != rec['size'] or _hash.sha256(f.read_bytes()).hexdigest() != rec['sha256']:
                        raise RuntimeError('Runtime file mismatch: '+n)
            if not folder.exists():
                with _tempfile.TemporaryDirectory(dir=parent,prefix='.runtime-') as tmp:
                    stage=_Path(tmp)/'payload';stage.mkdir()
                    for n in names:
                        (stage/n).write_bytes(z.read(n))
                    verify(stage)
                    _os.rename(stage,folder)
            verify(folder)
        print('RUNTIME_BUNDLE_VERIFIED:',_OAS_SHA,flush=True)
        print('RUNTIME_FILES_VERIFIED:',len(expected),flush=True)
        return folder

    _OAS_ROOT = _oas_bootstrap()

    _sys.dont_write_bytecode = True
    for _name in ['resource_manager','auxiliary_manager','environment_manager','checkpoint_manager','demo_assets_builder','stage1_engine','stage23_engine','runtime_support','pipeline','continuity']:
        if _name in _sys.modules:
            _loaded = getattr(_sys.modules[_name],'__file__',None)
            if not _loaded or _Path(_loaded).resolve() != (_OAS_ROOT/(_name+'.py')).resolve():
                raise RuntimeError('Old module already loaded: '+_name+'. Start a NEW notebook Session.')
    if str(_OAS_ROOT) not in _sys.path:
        _sys.path.insert(0,str(_OAS_ROOT))
    import pipeline as oas

    """Optional private, versioned Kaggle persistence. No model or runtime changes."""
    import hashlib
    import json
    import os
    import re
    import shutil
    import subprocess
    import tempfile
    import time
    import uuid
    import zipfile
    from pathlib import Path


    class PersistenceError(RuntimeError):
        pass


    def digest(path):
        h = hashlib.sha256()
        size = 0
        with Path(path).open('rb') as f:
            for block in iter(lambda: f.read(1024*1024), b''):
                size += len(block)
                h.update(block)
        return {'bytes': size, 'sha256': h.hexdigest()}


    def validate_destination(ref):
        if not isinstance(ref, str) or not re.fullmatch(r'[a-z0-9][a-z0-9_-]{0,49}/[a-z0-9][a-z0-9-]{5,49}', ref):
            raise PersistenceError('PERSIST_DATASET must be a personal owner/slug (lowercase, slug 6-50 characters).')
        # Dedicated name only; no accidental overwrite of model/demo resources.
        if not ref.split('/')[1].startswith('oas-checkpoints-'):
            raise PersistenceError('PERSIST_DATASET slug must start with oas-checkpoints-. Use one dedicated task-specific name.')
        return ref


    class PrivateDatasetStore:
        manifest_name = "oas-progress.json"

        def __init__(self, ref, task_id, token, *, cli=None, wait_seconds=300, command_seconds=900,
                     run_command=subprocess.run, sleep=time.sleep, clock=time.monotonic):
            self.ref = validate_destination(ref)
            if not re.fullmatch('[0-9a-f]{64}', task_id):
                raise PersistenceError('Invalid task identity')
            if not isinstance(token, str) or not token.strip():
                raise PersistenceError('KAGGLE_API_TOKEN secret is empty or missing')
            self.task_id = task_id
            self.nonce = uuid.uuid4().hex
            self.cli = cli or shutil.which('kaggle')
            if not self.cli:
                raise PersistenceError('Kaggle CLI not installed. Use a Kaggle image with a token-capable CLI.')
            self.wait_seconds = wait_seconds
            self.command_seconds = command_seconds
            self._run, self._sleep, self._clock = run_command, sleep, clock
            self._tmp = tempfile.TemporaryDirectory(prefix='oas-kaggle-auth-')
            self._env = dict(os.environ)
            for key in ('KAGGLE_USERNAME','KAGGLE_KEY','KAGGLE_API_TOKEN','KAGGLE_CONFIG_DIR'):
                self._env.pop(key, None)
            self._env.update(KAGGLE_API_TOKEN=token, KAGGLE_CONFIG_DIR=self._tmp.name)
            self.version = 0
            self.last_receipt = None
            self._manifest = None
            self._records = {}
            self._preserved = {}  # Exact remote bytes of earlier checkpoints on cross-session recovery.

        def close(self):
            self._env.pop('KAGGLE_API_TOKEN', None)
            self._tmp.cleanup()

        def command(self, args, *, timeout=None):
            # Never include stdout/stderr in errors: CLI may include signed URLs.
            try:
                r = self._run([self.cli, 'datasets', *args], env=self._env,
                              stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
                              stderr=subprocess.PIPE, timeout=timeout or self.command_seconds,
                              check=False)
            except Exception:
                raise PersistenceError('KAGGLE_API_COMMAND_FAILED: '+args[0]) from None
            if r.returncode != 0:
                raise PersistenceError('KAGGLE_API_COMMAND_FAILED: '+args[0]+' exit='+str(r.returncode))
            return r

        @staticmethod
        def _privacy_flag(data):
            # Only explicit booleans (or exact text booleans from older clients).
            # Never accept the truthiness of a nonempty string or infer private from access.
            flags = []
            for k in ('isPrivate', 'is_private'):
                if k in data:
                    v = data[k]
                    if isinstance(v, bool):
                        flags.append(v)
                    elif isinstance(v, str) and v.strip().lower() in ('true', 'false'):
                        flags.append(v.strip().lower() == 'true')
                    elif v is not None:
                        raise PersistenceError('REMOTE_PRIVACY_INVALID: unrecognized '+k+' value type')
            if len(set(flags)) > 1:
                raise PersistenceError('REMOTE_PRIVACY_CONFLICT: inconsistent privacy fields')
            return flags[0] if flags else None

        def _metadata_privacy(self, meta):
            if not isinstance(meta, dict):
                raise PersistenceError('REMOTE_METADATA_INVALID: expected a JSON object')
            # Kaggle CLI can serialize the entire API response: {"info": DatasetInfo}.
            # Flat DatasetInfo and legacy upload-style metadata are also supported.
            nodes = [meta]
            if isinstance(meta.get('info'), dict):
                nodes.append(meta['info'])
            elif 'info' in meta and meta['info'] is not None:
                raise PersistenceError('REMOTE_METADATA_INVALID: unexpected info structure')
            owner, slug = self.ref.split('/')
            flags = []
            for node in nodes:
                if node.get('errorMessage') or node.get('error_message'):
                    raise PersistenceError('REMOTE_METADATA_ERROR: server returned an error')
                for key in ('ref', 'id'):
                    v = node.get(key)
                    # id may be numeric in API responses; only a path-like id is a ref.
                    if isinstance(v, str) and '/' in v and v.lower() != self.ref:
                        raise PersistenceError('REMOTE_METADATA_IDENTITY_MISMATCH: '+key)
                for key in ('datasetSlug', 'dataset_slug'):
                    v = node.get(key)
                    if isinstance(v, str) and v and v.lower() not in (slug, self.ref):
                        raise PersistenceError('REMOTE_METADATA_IDENTITY_MISMATCH: '+key)
                for key in ('ownerUser', 'owner_user', 'ownerSlug', 'owner_slug'):
                    v = node.get(key)
                    if isinstance(v, str) and v and v.lower() != owner:
                        raise PersistenceError('REMOTE_METADATA_IDENTITY_MISMATCH: '+key)
                v = self._privacy_flag(node)
                if v is not None:
                    flags.append(v)
            if len(set(flags)) > 1:
                raise PersistenceError('REMOTE_PRIVACY_CONFLICT: metadata and info disagree')
            return flags[0] if flags else None

        def _query_dataset_privacy(self):
            # Read-only fallback to the official SDK GetDataset endpoint. Run in the
            # same isolated token environment as CLI calls. Do not expose API output,
            # exception text, access tokens, signed URLs or user metadata in logs.
            import sys
            script = "import json, os, sys\nfrom kagglesdk import KaggleClient\nfrom kagglesdk.datasets.types.dataset_api_service import ApiGetDatasetRequest\nowner, slug = sys.argv[1].split('/')\nreq = ApiGetDatasetRequest()\nreq.owner_slug = owner\nreq.dataset_slug = slug\nwith KaggleClient(api_token=os.environ['KAGGLE_API_TOKEN']) as client:\n    item = client.datasets.dataset_api_client.get_dataset(req)\n    result = {'ref': item.ref, 'isPrivate': item.is_private}\nprint('OAS_PRIVATE_RESULT:' + json.dumps(result))\n"
            try:
                r = self._run([sys.executable, '-c', script, self.ref], env=self._env,
                              stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
                              stderr=subprocess.PIPE, timeout=60, check=False)
                if r.returncode != 0:
                    raise PersistenceError('REMOTE_PRIVACY_SDK_FAILED: exit='+str(r.returncode))
                raw = r.stdout.decode('utf-8', errors='replace') if isinstance(r.stdout, bytes) else str(r.stdout)
                matches = [x[len('OAS_PRIVATE_RESULT:'):] for x in raw.splitlines() if x.startswith('OAS_PRIVATE_RESULT:')]
                if len(matches) != 1:
                    raise PersistenceError('REMOTE_PRIVACY_SDK_FAILED: missing result')
                result = json.loads(matches[0])
            except PersistenceError:
                raise
            except Exception as exc:
                raise PersistenceError('REMOTE_PRIVACY_SDK_FAILED: '+type(exc).__name__) from None
            if not isinstance(result, dict) or result.get('ref') != self.ref:
                raise PersistenceError('REMOTE_METADATA_IDENTITY_MISMATCH: SDK ref')
            return self._privacy_flag(result)

        def _private(self):
            # Fail closed. Never upload to a public Dataset, or merely assume that
            # an accessible Dataset must be private. Handle nested API responses.
            with tempfile.TemporaryDirectory(prefix='oas-remote-meta-') as d:
                self.command(['metadata', self.ref, '-p', d], timeout=60)
                p = Path(d)/'dataset-metadata.json'
                try:
                    meta = json.loads(p.read_text(encoding='utf-8-sig'))
                except Exception:
                    raise PersistenceError('REMOTE_METADATA_INVALID: metadata JSON unavailable') from None
                privacy = self._metadata_privacy(meta)
                if privacy is None:
                    print('REMOTE_PRIVACY_METADATA_FIELD_MISSING: trying read-only SDK GetDataset', flush=True)
                    privacy = self._query_dataset_privacy()
                if privacy is False:
                    raise PersistenceError('Remote dataset privacy check: REMOTE_DATASET_PUBLIC; set Visibility to Private before retrying')
                if privacy is not True:
                    raise PersistenceError('Remote dataset privacy not explicitly confirmed; refusing to upload')

        def _assert_latest(self, expected):
            _, record=self._read_latest_manifest()
            if record!=expected:
                raise PersistenceError('REMOTE_VERSION_CHANGED: another version appeared; stop and inspect the latest Dataset before retrying. Do not run concurrent writers.')

        def _download_record(self, version, name, expected, destination=None):
            """Verify an exact version; optionally retain a verified copy, never unzip into a directory."""
            with tempfile.TemporaryDirectory(prefix='oas-remote-check-') as d:
                self.command(['download','-d',f'{self.ref}/{version}','-f',name,'-p',d,'-o','-q'])
                p=Path(d)/name
                if p.is_file() and not p.is_symlink():
                    actual=digest(p)
                    if actual!=expected:
                        raise PersistenceError('Remote byte verification failed: '+name)
                    if destination is not None: shutil.copyfile(p,destination)
                else:
                    wrapped=Path(d)/(name+'.zip')
                    if not wrapped.is_file():
                        raise PersistenceError('Remote file unavailable: '+name)
                    try:
                        with zipfile.ZipFile(wrapped) as z:
                            members=z.infolist()
                            if len(members)!=1 or members[0].filename!=name or members[0].file_size!=expected['bytes']:
                                raise PersistenceError('Unexpected remote archive wrapper: '+name)
                            h=hashlib.sha256();size=0
                            # A private scratch file, not a path derived from ZIP members.
                            staged=Path(d)/'verified_payload'
                            with z.open(members[0]) as f, staged.open('wb') as out:
                                for block in iter(lambda:f.read(1024*1024),b''):
                                    size+=len(block);h.update(block);out.write(block)
                            actual={'bytes':size,'sha256':h.hexdigest()}
                            if actual!=expected:
                                raise PersistenceError('Remote byte verification failed: '+name)
                            if destination is not None: shutil.copyfile(staged,destination)
                    except (OSError,zipfile.BadZipFile):
                        raise PersistenceError('Unreadable remote file: '+name) from None
                if destination is not None and digest(destination)!=expected:
                    raise PersistenceError('Verified copy changed: '+name)

        def _verify(self, version, manifest_record, files):
            deadline = self._clock()+self.wait_seconds
            while True:
                try:
                    self._download_record(version,self.manifest_name,manifest_record)
                    self._private()
                    for name,record in files.items():
                        self._download_record(version,name,record)
                    # A pinned version can be ready before the latest alias advances.
                    latest, observed = self._read_latest_manifest()
                    if observed != manifest_record:
                        validate_progress(latest,self.ref,self.task_id)
                        if latest['version'] < version:
                            raise PersistenceError('REMOTE_LATEST_ALIAS_PENDING')
                        raise PersistenceError('REMOTE_VERSION_CHANGED: another version appeared; stop and inspect the latest Dataset. Do not run concurrent writers.')
                    return
                except PersistenceError as exc:
                    reason = str(exc)
                    # Retry publication delays, not privacy, hash, identity, malformed
                    # manifest or concurrent-writer failures. Never log raw CLI output.
                    pending = reason.startswith(('KAGGLE_API_COMMAND_FAILED: download',
                                                  'Remote file unavailable:',
                                                  'REMOTE_LATEST_ALIAS_PENDING'))
                    if not pending:
                        raise PersistenceError('REMOTE_VERSION_VERIFY_FAILED: '+self.ref+'/'+str(version)+'; '+reason) from None
                    if self._clock() >= deadline:
                        raise PersistenceError('REMOTE_VERSION_NOT_VERIFIED: '+self.ref+'/'+str(version)+'; last reason: '+reason) from None
                    print('REMOTE_VERSION_WAIT:',self.ref,'VERSION:',version,'REASON:',reason,flush=True)
                    self._sleep(min(10,max(0,deadline-self._clock())))

        def assert_current(self):
            if self.version<1 or self._manifest is None:
                raise PersistenceError('No verified remote version')
            self._private()
            self._download_record(self.version,self.manifest_name,self._manifest)
            self._assert_latest(self._manifest)


    # Optional demo extension; the frozen runtime and original asset manifest stay unchanged.
    import hashlib
    import shutil
    import tempfile
    import wave
    from pathlib import Path
    import pipeline as oas
    import resource_manager as rm
    from demo_assets_builder import verify as verify_base_demo

    DEMO_AUDIO_FILES = ('demo_audio.wav', 'native_blink_long_test.wav')
    _DEMO_BASE = {'portrait_01.png','portrait_02.png','reference.wav','reference.txt',
                  'demo_audio.wav','script_en.txt','script_zh.txt','demo_config.json',
                  'PORTRAIT_LICENSE.txt','ASSET_NOTICES.md','asset_manifest.json'}
    _DEMO_LONG_RECORDS = {'native_blink_long_test.txt': {'size': 461, 'sha256': '88a5c2c85a00d960770a6016d1bc5f473cf67c320af682f8ef6a1f044c00d9fb'}, 'native_blink_long_test.wav': {'size': 1523172, 'sha256': 'fc8dbce194c574aa24653d7350823492050f26d3a4b0eae498864253a3c1ecf4'}}


    def verify_demo_mount(input_root='/kaggle/input'):
        root = rm.locate(input_root, 'open-avatar-demo-assets')
        actual = {p.relative_to(root).as_posix() for p in root.rglob('*') if p.is_file()}
        base = _DEMO_BASE | ({'dataset-metadata.json'} if 'dataset-metadata.json' in actual else set())
        extras = actual - base
        missing = _DEMO_BASE - actual
        if missing or extras not in (set(), set(_DEMO_LONG_RECORDS)):
            raise RuntimeError('DEMO_ASSET_SCOPE_MISMATCH: missing='+str(sorted(missing))+
                               '; extra='+str(sorted(extras))+'; attach original or official expanded Demo Assets')
        if not extras:
            verify_base_demo(root, mounted=True)
            return root, False
        # The older verifier intentionally rejects extra files. Verify an isolated copy
        # of the unchanged core, then verify both optional files against release pins.
        # Nothing is written to the mounted Dataset or to the frozen runtime.
        with tempfile.TemporaryDirectory(prefix='oas-demo-base-') as tmp:
            projection = Path(tmp)
            for name in sorted(base):
                shutil.copyfile(root/name, projection/name)
            verify_base_demo(projection, mounted=True)
        for name, expected in _DEMO_LONG_RECORDS.items():
            path = root/name
            if path.stat().st_size != expected['size'] or hashlib.sha256(path.read_bytes()).hexdigest() != expected['sha256']:
                raise RuntimeError('DEMO_LONG_ASSET_INTEGRITY_MISMATCH: '+name)
            print('DEMO_LONG_ASSET_VERIFIED:',name,flush=True)
        with wave.open(str(root/'native_blink_long_test.wav'),'rb') as w:
            duration = w.getnframes()/w.getframerate()
            if (w.getframerate(),w.getnchannels(),w.getsampwidth(),w.getcomptype()) != (24000,1,2,'NONE') or not 30 < duration <= 60:
                raise RuntimeError('DEMO_LONG_AUDIO_INVALID')
        print('DEMO_LONG_AUDIO_SECONDS:',round(duration,3),flush=True)
        return root, True


    def prepare_demo_job(mode, portrait='portrait_01.png', language='English',
                         demo_audio='demo_audio.wav', input_root='/kaggle/input', **opts):
        if portrait not in ('portrait_01.png','portrait_02.png'):
            raise ValueError('DEMO_PORTRAIT must be portrait_01.png or portrait_02.png')
        if mode == 'audio' and demo_audio not in DEMO_AUDIO_FILES:
            raise ValueError('DEMO_AUDIO must be demo_audio.wav or native_blink_long_test.wav')
        if mode not in ('audio','text') or (mode == 'text' and language not in ('English','Chinese')):
            raise ValueError('Unsupported demo mode/language; demo_text uses English or Chinese')
        root, has_long = verify_demo_mount(input_root)
        if mode == 'audio':
            if demo_audio == 'native_blink_long_test.wav' and not has_long:
                raise RuntimeError('DEMO_LONG_AUDIO_NOT_MOUNTED: select the expanded Demo Assets version, or use demo_audio.wav')
            print('DEMO_AUDIO_SELECTED:',demo_audio,flush=True)
            return oas.prepare_job(mode='audio',portrait=root/portrait,audio=root/demo_audio,**opts)
        return oas.prepare_job(mode='text',portrait=root/portrait,
                               script=root/('script_zh.txt' if language=='Chinese' else 'script_en.txt'),
                               reference_audio=root/'reference.wav',reference_text=root/'reference.txt',
                               language=language,**opts)


    def resolve_workflow(blink_mode, open_frames, export_checkpoint):
        if blink_mode not in ('liveportrait', 'ditto_template', 'ditto_native'):
            raise ValueError("BLINK_MODE must be liveportrait, ditto_template or ditto_native")
        if type(open_frames) is not int or open_frames < 0:
            raise ValueError('DITTO_EYE_OPEN_FRAMES must be a nonnegative integer')
        if export_checkpoint != 'auto' and type(export_checkpoint) is not bool:
            raise ValueError("EXPORT_CHECKPOINT must be 'auto', True or False")
        use_lp = blink_mode == 'liveportrait'
        export = use_lp if export_checkpoint == 'auto' else export_checkpoint
        if use_lp and not export:
            raise ValueError('LivePortrait needs a Stage 1 checkpoint; set EXPORT_CHECKPOINT to auto or True')
        return use_lp, export

    def configure_eye_mode(job, blink_mode, open_frames):
        # Identical to the tested Ditto-only experiment for the two direct modes.
        if blink_mode == 'liveportrait':
            if not job['config']['blink']['enabled']:
                raise ValueError('Invalid LivePortrait task configuration')
            return job  # preserve the exact original identity and model parameters
        if job['config']['blink']['enabled']:
            raise ValueError('Direct Ditto must not enable LivePortrait')
        ditto = job['config']['ditto']
        ditto['drive_eye'] = True
        if blink_mode == 'ditto_native':
            ditto['delta_eye_arr'] = None
            ditto['delta_eye_open_n'] = -1
        else:
            ditto.pop('delta_eye_arr', None)  # retain the pinned model's default template, if available
            ditto['delta_eye_open_n'] = open_frames
        job['identity'] = cp.make_identity(job['inputs'], job['config'], job['identity']['revisions'])
        job['task_id'] = cp.task_id(job['identity'])
        oas.verify_bound(job['identity'])
        return job

    # V1.1 continuity build: task identity changes with runtime and planning.
    SCHEDULER_SHA = '03b674ff888a2c7d8b36a1310b22b0ea9f3bb3a4c3d32583e724bb0c5d6dcf91'

    SEGMENT_WORKER_SOURCE = '"""A persistent stage-specific worker, with one verified runtime environment per stage.\nThe coordinator owns Kaggle credentials; this subprocess receives none.\n"""\nimport json, sys, traceback, shutil\nfrom pathlib import Path\nroot,specfile,stage=sys.argv[1:]\nsys.dont_write_bytecode=True;sys.path.insert(0,root)\nimport pipeline as oas\nimport checkpoint_manager as cp\nspec=json.loads(Path(specfile).read_text())\nwork=Path(spec[\'work\']);result=Path(spec[\'result\'])\nstudio=oas.Studio(work=work,input_root=spec[\'input_root\'],cache=spec[\'cache\'],accept_noncommercial_research=spec[\'accept\'],check_network=spec[\'check_network\'])\nfor line in sys.stdin:\n    cmd=json.loads(line)\n    if cmd.get(\'stop\'):break\n    tid=cmd[\'unit_id\']\n    try:\n        with oas.support.monitor(stage,None,gpu=stage!=\'stage3\',enabled=spec[\'monitor\']):\n            if stage==\'stage1\':\n                job=cmd[\'job\'];job[\'inputs\']={k:Path(v) for k,v in job[\'inputs\'].items()}\n                assert job[\'task_id\']==tid\n                studio.stage1(job,export=False)\n            else:\n                through=\'stage1\' if stage==\'stage2\' else \'stage2\'\n                studio.resume_archive(cmd[\'archive\'],tid,through)\n                if stage==\'stage2\':studio.stage2(tid,export=False)\n                else:studio.stage3(tid,export=False)\n            archive=studio.export(tid,stage)\n        cp.atomic_json(result,{\'unit_id\':tid,\'stage\':stage,\'archive\':str(archive),\'status\':\'OK\'})\n    except Exception:\n        traceback.print_exc();cp.atomic_json(result,{\'unit_id\':tid,\'stage\':stage,\'status\':\'FAILED\'});sys.exit(1)\n'

    """Open Avatar Studio 1.0: versioned, resumable long-task orchestration.
    The embedded 1.1 runtime is used ONLY as a bounded segment engine.
    No total text, duration or frame-count cap is imposed by this scheduler.
    """
    import copy
    import hashlib
    import json
    import math
    import os
    import re
    import shutil
    import subprocess
    import sys
    import tempfile
    import time
    import wave
    import zipfile
    from fractions import Fraction
    from pathlib import Path

    import checkpoint_manager as cp
    import environment_manager as em
    import pipeline as oas
    import stage1_engine as s1
    import continuity as cont

    SCHEMA_V1 = 'oas-long-task-v1'
    PROGRESS_NAME = 'oas-progress.json'
    STAGES_V1 = ('stage1','stage2','stage3')
    RATE = 24000
    FPS = 25
    FRAME_SAMPLES = RATE // FPS
    ENGINE_SHA = 'f0ddb13b6d236d1724f5bd9b8137a561405a3a22b9abb34758c0f5a771f3c6cc'
    # A separate scheduler build fingerprint is injected by the notebook builder.


    def ensure(ok, message):
        if not ok:
            raise RuntimeError(message)


    def canonical(obj):
        return json.dumps(obj, sort_keys=True, ensure_ascii=False, separators=(',',':'), allow_nan=False).encode('utf-8')


    def sha_json(obj):
        return hashlib.sha256(canonical(obj)).hexdigest()


    def write_json(path, obj):
        # The legacy per-segment manifest size limits must not cap a long task index.
        path = Path(path); path.parent.mkdir(parents=True,exist_ok=True)
        with tempfile.NamedTemporaryFile(dir=path.parent, prefix='.oas-',delete=False) as f:
            p=Path(f.name)
            try:
                f.write(canonical(obj)+b'\n'); f.flush(); os.fsync(f.fileno()); f.close()
                os.replace(p,path)
            finally:
                p.unlink(missing_ok=True)


    def read_json(path):
        p=Path(path)
        ensure(p.is_file() and not p.is_symlink(),'Missing regular progress file: '+str(p))
        return json.loads(p.read_text(encoding='utf-8'))


    def ishex(v):
        return isinstance(v,str) and re.fullmatch(r'[0-9a-f]{64}',v) is not None


    def artifact_record(path):
        r=cp.file_record(path); ensure(r['bytes']>0,'Empty artifact'); return r


    def check_record(r):
        ensure(isinstance(r,dict) and set(r)=={'bytes','sha256'} and type(r['bytes']) is int and r['bytes']>0 and ishex(r['sha256']), 'Invalid artifact digest')


    def text_chunks(text, limit):
        """Linear scan with punctuation-aware boundaries; only whitespace is trimmed.
        No total length check; never drop any non-whitespace character.
        """
        ensure(type(limit) is int and 80<=limit<=6000,'Invalid text partition limit (80-6000)')
        text=text.strip(); ensure(bool(text),'Script is empty')
        pos=0; out=[]
        while pos<len(text):
            end=min(len(text),pos+limit)
            if end<len(text):
                window=text[pos:end]; floor=max(1,limit//2)
                for pat in (r'[\u3002\uff01\uff1f.!?;\uff1b\n]+',r'[\uff0c\u3001\uff1a,:]+',r'\s+'):
                    choices=[m.end() for m in re.finditer(pat,window) if m.end()>=floor]
                    if choices:
                        end=pos+choices[-1];break
            chunk=text[pos:end].strip()
            if chunk: out.append(chunk)
            pos=end
        ensure(bool(out),'No speakable text chunks')
        return out


    def audio_chunks(source, folder, seconds, speech_speed=1.0):
        """Decode once as raw PCM; bounded RAM, frame-aligned quiet cuts.
        A one-second lookahead prevents an invalid sub-second final segment.
        Works beyond RIFF/WAV's aggregate 4GiB limit. Every sample is retained.
        """
        import numpy as np
        ensure(type(seconds) is int and 10<=seconds<=600,'DITTO_SEGMENT_SECONDS must be 10-600')
        info=s1.read_media(source)
        tracks=[x for x in info['streams'] if x['codec_type']=='audio']
        ensure(len(tracks)==1,'Exactly one audio track required')
        duration=s1.stream_duration(tracks[0],info)
        speed=s1.validate_speech_speed(speech_speed)
        ensure(duration>=1,'Narration must be at least one second')
        maximum=seconds*RATE; lookahead=min(10*RATE,maximum//3)
        folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)
        count=0; total=0; records=[]; buf=b''; eof=False
        with tempfile.TemporaryFile() as err:
            proc=subprocess.Popen(['ffmpeg','-nostdin','-v','error','-xerror','-i',str(source),'-map','0:a:0','-vn','-ac','1','-ar',str(RATE),*s1.speech_speed_filter(speed),'-f','s16le','pipe:1'],stdout=subprocess.PIPE,stderr=err)
            try:
                while True:
                    want=2*(maximum+lookahead)-len(buf)
                    if not eof and want>0:
                        part=proc.stdout.read(want)
                        buf+=part
                        if len(part)<want: eof=True
                    ensure(len(buf)%2==0,'Unaligned PCM stream')
                    n=len(buf)//2
                    if n==0 and eof:break
                    if eof and n<=maximum:
                        ensure(n>=RATE,'Decoded narration is shorter than one second')
                        end=n
                    else:
                        hi=min(maximum,n-lookahead)//FRAME_SAMPLES*FRAME_SAMPLES
                        lo=max(RATE,hi-5*RATE);lo=(lo+FRAME_SAMPLES-1)//FRAME_SAMPLES*FRAME_SAMPLES
                        ensure(lo<=hi,'Cannot partition audio with a valid tail')
                        signal=np.frombuffer(buf,dtype='<i2')
                        def energy(i):
                            x=signal[max(0,i-4*FRAME_SAMPLES):min(n,i+4*FRAME_SAMPLES)].astype(np.float64)
                            return (float(np.mean(x*x)),-i)
                        end=min(range(lo,hi+1,FRAME_SAMPLES),key=energy)
                    count+=1;total+=end
                    path=folder/f'part_{count:08d}.wav'
                    with wave.open(str(path),'wb') as w:
                        w.setnchannels(1);w.setsampwidth(2);w.setframerate(RATE);w.writeframes(buf[:2*end])
                    records.append((path,end));buf=buf[2*end:]
                ensure(proc.wait()==0,'FFmpeg audio decode failed; source may be incomplete')
            finally:
                if proc.poll() is None:
                    proc.kill();proc.wait()
                proc.stdout.close()
        ensure(records and total>=RATE and all(n%FRAME_SAMPLES==0 for _,n in records[:-1]),'Invalid audio partition')
        if speed != 1.0:
            print('SPEECH_SPEED_APPLIED:',speed,'INPUT_SECONDS:',duration,'OUTPUT_SECONDS:',total/RATE,'SCOPE: continuous input before partition',flush=True)
        print('AUDIO_PLAN:',count,'SEGMENTS, EXACT_PCM_SAMPLES:',total,'SECONDS:',total/RATE,flush=True)
        return records


    def input_paths(mode, language, demo_portrait, demo_audio, portrait, audio, script, reference_audio, reference_text, input_root):
        ensure(mode in ('demo_audio','demo_text','audio','text'),'Invalid INPUT_MODE')
        if mode.startswith('demo_'):
            root,has_long=verify_demo_mount(input_root)
            ensure(demo_portrait in ('portrait_01.png','portrait_02.png'),'Invalid DEMO_PORTRAIT')
            portrait=root/demo_portrait
            if mode=='demo_audio':
                ensure(demo_audio in DEMO_AUDIO_FILES,'Invalid DEMO_AUDIO')
                ensure(demo_audio!='native_blink_long_test.wav' or has_long,'Long demo audio not mounted')
                audio=root/demo_audio;mode='audio'
            else:
                ensure(language in ('English','Chinese'),'Demo text language must be English or Chinese')
                script=root/('script_zh.txt' if language=='Chinese' else 'script_en.txt')
                reference_audio=root/'reference.wav';reference_text=root/'reference.txt';mode='text'
        paths={'portrait':s1.input_file(portrait)}
        s1.inspect_photo(paths['portrait'])
        if mode=='audio': paths['audio']=s1.input_file(audio)
        else:
            ensure(language in ('English','Chinese','Auto'),'Invalid LANGUAGE')
            paths.update(script=s1.input_file(script),reference_audio=s1.input_file(reference_audio))
            s1.inspect_audio(paths['reference_audio'],30)
            if reference_text: paths['reference_text']=s1.input_file(reference_text)
        return mode,paths


    def make_plan(folder, *, mode, paths, language, blink_mode, expression, seed, open_frames, chunk_chars, segment_seconds, text_segment_chars=2400, warmup_seconds=1.2, smooth_seams=True, speech_speed=1.0):
        use_lp,_=resolve_workflow(blink_mode,open_frames,True)
        speed=s1.validate_speech_speed(speech_speed)
        print('SPEECH_SPEED:',speed,'MODE:',mode,flush=True)
        ensure(type(seed) is int and 0<=seed<2**32,'Invalid SEED')
        ensure(type(chunk_chars) is int and 80<=chunk_chars<=800,'TTS_CHUNK_CHARACTERS must be 80-800')
        ensure(type(text_segment_chars) is int and max(600,chunk_chars)<=text_segment_chars<=4000,'TEXT_SEGMENT_CHARACTERS must be 600-4000 and >= TTS_CHUNK_CHARACTERS')
        ensure(type(warmup_seconds) in (int,float) and math.isfinite(warmup_seconds) and 0<=warmup_seconds<=3,'DITTO_WARMUP_SECONDS must be 0-3')
        ensure(type(smooth_seams) is bool,'SMOOTH_SEAMS must be True or False')
        ensure(type(segment_seconds) is int and 10<=segment_seconds<=600,'DITTO_SEGMENT_SECONDS must be 10-600')
        folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)
        config=dict(mode=mode,language=language if mode=='text' else None,blink_mode=blink_mode,expression=expression,seed=seed,open_frames=open_frames,chunk_chars=chunk_chars,segment_seconds=segment_seconds,text_segment_chars=text_segment_chars,warmup_seconds=warmup_seconds,smooth_seams=smooth_seams,speech_speed=speed)
        inputs={k:artifact_record(v) for k,v in paths.items()}
        # The plan identity contains source hashes and behavior settings, never paths or Secrets.
        base=dict(engine_sha=ENGINE_SHA,scheduler_sha=SCHEDULER_SHA,inputs=inputs,config=config)
        base_id=sha_json(base)
        if mode=='text':
            text=paths['script'].read_text(encoding='utf-8-sig')
            chunks=text_chunks(text,text_segment_chars)
            print('SCRIPT_CHARACTERS:',len(text.strip()),'TASK_SEGMENTS:',len(chunks),'TTS_TARGET_CHARACTERS:',chunk_chars,flush=True)
            parts=[]
            for i,chunk in enumerate(chunks,1):
                p=folder/f'part_{i:08d}.txt';p.write_text(chunk,encoding='utf-8');parts.append((p,None))
        else:
            parts=audio_chunks(paths['audio'],folder,segment_seconds,speech_speed=speed)
        jobs=[];units=[]
        for i,(part,samples) in enumerate(parts,1):
            # The old runtime's duration/character limits apply ONLY to this small unit.
            opts=dict(portrait=paths['portrait'],language=language,seed=(seed if mode=='text' else (seed+i-1)%2**32),expression=expression,blink_enabled=use_lp,
                      max_script_characters=text_segment_chars,tts_chunk_characters=chunk_chars,max_video_seconds=(1200 if mode=='text' else 600),ditto_segment_seconds=segment_seconds,ditto_warmup_seconds=warmup_seconds,smooth_seams=smooth_seams,speech_speed=(speed if mode=='text' else 1.0))
            if mode=='text':
                job=oas.prepare_job(mode='text',script=part,reference_audio=paths['reference_audio'],reference_text=paths.get('reference_text'),**opts)
            else:
                job=oas.prepare_job(mode='audio',audio=part,**opts)
            job=configure_eye_mode(job,blink_mode,open_frames)
            job['config']['blink']['seed']=(seed+i-1)%2**32
            # For outer AUDIO partitions, use true adjacent audio as inference context.
            # Context is hashed with the unit; it is NOT included in the final narration.
            if mode=='audio' and warmup_seconds>0:
                pre=round(warmup_seconds*FPS)*FRAME_SAMPLES;post=min(round(warmup_seconds*FPS),12)*FRAME_SAMPLES
                for label,neighbor,amount,tail in [('context_before',i-2,pre,True),('context_after',i,post,False)]:
                    if amount and 0<=neighbor<len(parts):
                        with wave.open(str(parts[neighbor][0]),'rb') as w:
                            n=min(amount,w.getnframes())
                            if tail:w.setpos(w.getnframes()-n)
                            data=w.readframes(n)
                        ensure(len(data)==2*n and n>0,'Missing adjacent PCM context')
                        extra=folder/f'{label}_{i:08d}.pcm';extra.write_bytes(data);job['inputs'][label]=extra
            revisions=dict(job['identity']['revisions'],long_task_source=base_id,long_task_index=i)
            job['identity']=cp.make_identity(job['inputs'],job['config'],revisions)
            job['task_id']=cp.task_id(job['identity'])
            units.append(dict(index=i,unit_id=job['task_id'],samples=samples,source=artifact_record(part)))
            jobs.append(job)
        ensure({k:artifact_record(v) for k,v in paths.items()}==inputs,'Source changed while planning')
        plan=dict(base=base,units=units,use_lp=use_lp)
        return plan,jobs


    def expected_version(stage,index,n):
        return 1+STAGES_V1.index(stage)*n+index


    def artifact_name(stage,index):
        return f'{stage}_{index:08d}.bin'


    def route(plan):
        return STAGES_V1 if plan['use_lp'] else ('stage1',)


    def validate_progress(p, ref=None, expected_id='', expected_plan=None):
        ensure(isinstance(p,dict) and set(p)=={'schema','dataset','task_id','plan','version','records','final'},'Unrecognized V1.0 progress index')
        ensure(p['schema']==SCHEMA_V1 and p['dataset']==ref,'Wrong Dataset or journal schema')
        plan=p['plan'];tid=p['task_id']
        ensure(isinstance(plan,dict) and set(plan)=={'base','units','use_lp'} and ishex(tid) and sha_json(plan)==tid,'Task identity mismatch')
        ensure(not expected_id or tid==expected_id,'EXPECTED_TASK_ID mismatch')
        ensure(expected_plan is None or plan==expected_plan,'REMOTE_TASK_MISMATCH: source/config/segmentation differs; use original settings or a new Dataset')
        base=plan['base']; cfg=base['config']
        ensure(base['engine_sha']==ENGINE_SHA and base['scheduler_sha']==SCHEDULER_SHA,'Use the matching Notebook build (V1.0 or V1.1 RC checkpoints need their original notebook)')
        s1.validate_speech_speed(cfg['speech_speed'])
        ensure(type(plan['use_lp']) is bool and (cfg['blink_mode']=='liveportrait')==plan['use_lp'],'Invalid route')
        units=plan['units'];n=len(units)
        ensure(isinstance(units,list) and n>0,'Empty plan')
        for i,u in enumerate(units,1):
            ensure(isinstance(u,dict) and set(u)=={'index','unit_id','samples','source'} and u['index']==i and ishex(u['unit_id']),'Invalid segment plan')
            check_record(u['source'])
            if cfg['mode']=='audio':
                ensure(type(u['samples']) is int and u['samples']>=RATE and (i==n or u['samples']%FRAME_SAMPLES==0),'Invalid audio sample partition')
            else: ensure(cfg['mode']=='text' and u['samples'] is None,'Invalid text plan')
        ensure(len({u['unit_id'] for u in units})==n,'Duplicate unit identity')
        records=p['records'];ensure(isinstance(records,dict) and set(records)==set(STAGES_V1),'Invalid stage index')
        count=0
        for stage in STAGES_V1:
            entries=records[stage]
            ensure(isinstance(entries,list) and len(entries)<=n,'Invalid stage count')
            ensure(stage in route(plan) or not entries,'Unexpected stage for direct mode')
            if stage!='stage1' and entries:
                ensure(len(records[STAGES_V1[STAGES_V1.index(stage)-1]])==n,'Stage order violated')
            for i,e in enumerate(entries,1):
                ensure(isinstance(e,dict) and set(e)=={'version','name','record','unit_id'} and e['version']==expected_version(stage,i,n) and e['name']==artifact_name(stage,i) and e['unit_id']==units[i-1]['unit_id'],'Invalid history record')
                check_record(e['record'])
            count+=len(entries)
        if p['final'] is not None:
            f=p['final'];ensure(all(len(records[s])==n for s in route(plan)),'Final published before all segments completed')
            ensure(isinstance(f,dict) and set(f)=={'version','name','record','media'} and f['version']==count+2 and f['name']==f'oas_{tid}_final.mp4','Invalid final record')
            check_record(f['record']); ensure(isinstance(f['media'],dict),'Missing final media report')
        ensure(type(p['version']) is int and p['version']==1+count+(p['final'] is not None),'Invalid Dataset version sequence')
        return p


    def new_progress(plan,ref):
        p=dict(schema=SCHEMA_V1,dataset=ref,task_id=sha_json(plan),plan=plan,version=1,records={s:[] for s in STAGES_V1},final=None)
        return validate_progress(p,ref)


    def advance(p,stage,source,media=None):
        """Append one immutable artifact reference; never modify previous records."""
        validate_progress(p,p['dataset']);q=copy.deepcopy(p);n=len(q['plan']['units'])
        ensure(q['final'] is None,'Task already complete')
        q['version']+=1
        rec=artifact_record(source)
        if stage=='final':
            e=dict(version=q['version'],name=f"oas_{q['task_id']}_final.mp4",record=rec,media=media)
            q['final']=e
        else:
            ensure(stage in route(q['plan']),'Invalid stage')
            i=len(q['records'][stage])+1
            ensure(i<=n,'Stage already complete')
            e=dict(version=q['version'],name=artifact_name(stage,i),record=rec,unit_id=q['plan']['units'][i-1]['unit_id'])
            q['records'][stage].append(e)
        validate_progress(q,q['dataset'])
        return q,e


    def latest_entry(p):
        if p['final']: return p['final']
        for stage in reversed(STAGES_V1):
            if p['records'][stage]: return p['records'][stage][-1]
        return None


    def print_progress(p):
        n=len(p['plan']['units']); stages=route(p['plan'])
        nxt=next((f'{s}:{len(p["records"][s])+1}' for s in stages if len(p['records'][s])<n),'final' if p['final'] is None else 'complete')
        print('V1_PROGRESS:',p['task_id'],'VERSION:',p['version'],'TOTAL_SEGMENTS:',n,'DONE:',{s:len(p['records'][s]) for s in stages},'NEXT:',nxt,flush=True)


    class LocalJournal:
        """Local fallback. A complete journal directory is required for local recovery."""
        def __init__(self,folder,plan=None,expected_id=''):
            self.folder=Path(folder).resolve();self.folder.mkdir(parents=True,exist_ok=True)
            self.path=self.folder/PROGRESS_NAME
            if self.path.exists():self.p=validate_progress(read_json(self.path),None,expected_id,plan)
            else:
                ensure(plan is not None,'No local journal; specify remote Dataset or exact local progress path')
                self.p=new_progress(plan,None);ensure(not expected_id or self.p['task_id']==expected_id,'EXPECTED_TASK_ID mismatch');write_json(self.path,self.p)
            last=latest_entry(self.p)
            if last:self._verify(last)
            print_progress(self.p)
        def _path(self,e):return self.folder/'versions'/str(e['version'])/e['name']
        def _verify(self,e):
            p=self._path(e);ensure(artifact_record(p)==e['record'],'Missing or corrupt local checkpoint: '+str(p));return p
        def assert_current(self):ensure(read_json(self.path)==self.p,'Local journal changed; do not run concurrent writers')
        def commit(self,stage,source,media=None):
            self.assert_current();q,e=advance(self.p,stage,source,media)
            dst=self._path(e);dst.parent.mkdir(parents=True,exist_ok=True)
            # If killed after file copy but before index commit, retry from that last verified index.
            tmp=dst.with_name(dst.name+'.partial');shutil.copyfile(source,tmp)
            ensure(artifact_record(tmp)==e['record'],'Local copy changed');os.replace(tmp,dst)
            write_json(self.path,q);self.p=q;print_progress(q)
            return e
        def fetch(self,e,dest):
            shutil.copyfile(self._verify(e),dest);ensure(artifact_record(dest)==e['record'],'Local readback mismatch')
            return Path(dest)
        def close(self):pass


    class RemoteJournal(PrivateDatasetStore):
        """One private Dataset, append-only Kaggle versions, rolling payloads.
        Prior versions are retained; latest index points to all historical artifacts.
        No cumulative copying of old media on each commit. Single writer only.
        """
        manifest_name=PROGRESS_NAME
        def __init__(self,ref,token,folder,plan=None,expected_id='',wait_seconds=600,command_seconds=3600,**kw):
            super().__init__(ref,'0'*64,token,wait_seconds=wait_seconds,command_seconds=command_seconds,**kw)
            self.folder=Path(folder);self.folder.mkdir(parents=True,exist_ok=True);self.path=self.folder/PROGRESS_NAME
            try:
                try:self._private()
                except PersistenceError as exc:
                    # A metadata command failure does NOT prove absence. Only attempt create-only.
                    if plan is None or not str(exc).startswith('KAGGLE_API_COMMAND_FAILED: metadata'):raise
                    self.p=new_progress(plan,ref);self.task_id=self.p['task_id']
                    ensure(not expected_id or self.task_id==expected_id,'EXPECTED_TASK_ID mismatch')
                    self._write_index(self.p,None,None,create=True)
                else:
                    p,rec=self._read_latest_manifest();validate_progress(p,ref,expected_id,plan)
                    self.task_id=p['task_id']
                    self._download_record(p['version'],self.manifest_name,rec)
                    last=latest_entry(p)
                    if last:self._download_record(last['version'],last['name'],last['record'])
                    self._private();self._assert_latest(rec)
                    self.p=p;self.version=p['version'];self._manifest=rec;write_json(self.path,p)
                ensure(not expected_id or self.p['task_id']==expected_id,'EXPECTED_TASK_ID mismatch')
                print('V1_REMOTE_CONNECTED:',self.ref,'VERSION:',self.p['version'],flush=True);print_progress(self.p)
            except BaseException:
                self.close();raise
        @classmethod
        def open_secret(cls,ref,folder,plan=None,expected_id='',wait_seconds=600,command_seconds=3600):
            validate_destination(ref)
            try:
                from kaggle_secrets import UserSecretsClient
                token=UserSecretsClient().get_secret('KAGGLE_API_TOKEN')
            except Exception:raise PersistenceError('Add KAGGLE_API_TOKEN in Secrets and grant this PRIVATE Notebook access.') from None
            return cls(ref,token,folder,plan,expected_id,wait_seconds,command_seconds)
        def _read_latest_manifest(self):
            # No fixed total task-index size cap. Only a regular index, or a one-file ZIP wrapper.
            with tempfile.TemporaryDirectory(prefix='oas-v1-latest-') as d:
                self.command(['download','-d',self.ref,'-f',self.manifest_name,'-p',d,'-o','-q'])
                p=Path(d)/self.manifest_name
                if p.is_file() and not p.is_symlink():data=p.read_bytes()
                else:
                    try:
                        with zipfile.ZipFile(Path(d)/(self.manifest_name+'.zip')) as z:
                            items=z.infolist()
                            if len(items)!=1 or items[0].filename!=self.manifest_name:raise PersistenceError('Unexpected progress ZIP wrapper')
                            data=z.read(items[0])
                    except (OSError,zipfile.BadZipFile):raise PersistenceError('Cannot read latest progress index') from None
                try:return json.loads(data),{'bytes':len(data),'sha256':hashlib.sha256(data).hexdigest()}
                except ValueError:raise PersistenceError('Malformed progress index') from None
        def _write_index(self,q,e,source,create=False):
            validate_progress(q,self.ref)
            if not create:self.assert_current()
            with tempfile.TemporaryDirectory(prefix='oas-v1-upload-') as d:
                d=Path(d);write_json(d/self.manifest_name,q);mrec=artifact_record(d/self.manifest_name);records={}
                if e is not None:
                    shutil.copyfile(source,d/e['name']);ensure(artifact_record(d/e['name'])==e['record'],'Artifact changed during upload staging');records[e['name']]=e['record']
                write_json(d/'dataset-metadata.json',{'id':self.ref,'title':self.ref.split('/')[1],'licenses':[{'name':'other'}],'description':'Private Open Avatar Studio V1.1 long-task recovery. Keep historical versions until final output is verified.'})
                print('V1_REMOTE_SAVE:',self.ref,'VERSION:',q['version'],'ARTIFACT:',e['name'] if e else 'plan',flush=True)
                if create:self.command(['create','-p',str(d),'-t','-r','skip','-q'])
                else:
                    # Explicitly never pass --delete-old-versions. Historical versions contain the media.
                    self.assert_current();self.command(['version','-p',str(d),'-m',f'OAS V1.1 {q["version"]} {e["name"]}','-t','-r','skip','-q'])
                self._verify(q['version'],mrec,records)
            self.p=q;self.version=q['version'];self._manifest=mrec;write_json(self.path,q)
            print('V1_REMOTE_VERIFIED:',f'https://www.kaggle.com/datasets/{self.ref}/versions/{self.version}',flush=True);print_progress(q)
        def commit(self,stage,source,media=None):
            q,e=advance(self.p,stage,source,media);self._write_index(q,e,source);return e
        def fetch(self,e,dest):
            self._private();self._download_record(e['version'],e['name'],e['record'],destination=dest);return Path(dest)

    class SegmentWorker:
        """Reuse one subprocess per stage; each stage has its own fresh cache.
        The frozen runtime's cross-stage GPU guard remains active inside each cache.
        """
        def __init__(self,stage,root,folder,input_root,accept,check_network,monitor):
            self.stage=stage;self.folder=Path(folder);self.folder.mkdir(parents=True,exist_ok=True)
            self.work=self.folder/'work';self.cache=self.folder/'cache';self.cache.mkdir()
            self.result=self.folder/'result.json'
            script=self.folder/'worker.py';script.write_text(SEGMENT_WORKER_SOURCE,encoding='utf-8')
            spec=self.folder/'spec.json';write_json(spec,dict(work=str(self.work),cache=str(self.cache),result=str(self.result),input_root=str(input_root),accept=accept,check_network=check_network,monitor=monitor))
            self.proc=subprocess.Popen([sys.executable,'-B',str(script),str(root),str(spec),stage],stdin=subprocess.PIPE,text=True,bufsize=1,env=em.worker_env(self.cache,cpu=stage=='stage3'),start_new_session=True)
        def __enter__(self):return self
        def __exit__(self,*exc):self.close()
        def run(self,unit_id,job=None,archive=None):
            self.result.unlink(missing_ok=True)
            cmd={'unit_id':unit_id}
            if job is not None:cmd['job']=dict(job,inputs={k:str(v) for k,v in job['inputs'].items()})
            if archive is not None:cmd['archive']=str(archive)
            self.proc.stdin.write(json.dumps(cmd)+'\n');self.proc.stdin.flush()
            while not self.result.is_file():
                ensure(self.proc.poll() is None,'Segment worker exited before commit; resume from last verified index')
                time.sleep(.25)
            r=read_json(self.result)
            ensure(r.get('status')=='OK' and r.get('unit_id')==unit_id and r.get('stage')==self.stage,'Segment inference failed; previous commits remain valid')
            result=Path(r['archive']).resolve()
            ensure(result.is_relative_to(self.work.resolve()) and result.is_file() and not result.is_symlink(),'Unexpected segment output')
            return result
        def release(self,unit_id,archive):
            # Only after journal commit. Clean this worker's own scratch media, never the journal.
            ensure(ishex(unit_id),'Invalid unit ID')
            shutil.rmtree(self.work/'tasks'/unit_id)
            Path(archive).unlink()
        def close(self):
            if self.proc.poll() is None:
                try:self.proc.stdin.write('{"stop":true}\n');self.proc.stdin.flush();self.proc.wait(timeout=10)
                except (OSError,subprocess.TimeoutExpired):
                    import signal
                    try:os.killpg(self.proc.pid,signal.SIGTERM)
                    except ProcessLookupError:pass
                    try:self.proc.wait(timeout=10)
                    except subprocess.TimeoutExpired:
                        try:os.killpg(self.proc.pid,signal.SIGKILL)
                        except ProcessLookupError:pass
                        self.proc.wait()
            if self.proc.stdin:self.proc.stdin.close()


    def stage_ready(journal,stage):
        p=journal.p;n=len(p['plan']['units'])
        if stage not in route(p['plan']):return False
        if stage=='stage1':return True
        previous=STAGES_V1[STAGES_V1.index(stage)-1]
        if len(p['records'][previous])!=n:
            print('V1_STAGE_BLOCKED:',stage,'WAITING_FOR:',previous,'COMPLETED:',len(p['records'][previous]),'OF',n,flush=True)
            return False
        return True


    def execute_stage(journal,stage,*,root,work,input_root,jobs=None,accept=False,check_network=True,monitor=True,max_new=0,max_seconds=0,worker_cls=SegmentWorker):
        ensure(type(max_new) is int and max_new>=0,'MAX_NEW_SEGMENTS must be nonnegative; 0 means unlimited')
        ensure(type(max_seconds) is int and max_seconds>=0,'MAX_STAGE_RUN_SECONDS must be nonnegative; 0 means unlimited')
        if not stage_ready(journal,stage):return 'BLOCKED'
        p=journal.p;n=len(p['plan']['units']);done=len(p['records'][stage])
        if done==n:
            print('V1_STAGE_ALREADY_COMPLETE:',stage,'SKIPPED_SEGMENTS:',n,flush=True);return 'COMPLETE'
        if stage in ('stage1','stage2'):ensure(accept is True,'Explicit model terms acceptance required before inference')
        if stage=='stage1':ensure(jobs is not None and len(jobs)==n,'Stage 1 recovery requires original source and settings')
        print('V1_STAGE_START:',stage,'SKIPPED_COMMITTED:',done,'REMAINING:',n-done,flush=True)
        start=time.monotonic();processed=0
        Path(work).mkdir(parents=True,exist_ok=True)
        with tempfile.TemporaryDirectory(prefix='oas-v1-'+stage+'-',dir=work) as tmp:
            with worker_cls(stage,root,tmp,input_root,accept,check_network,monitor) as worker:
                for i in range(done+1,n+1):
                    if processed and ((max_new and processed>=max_new) or (max_seconds and time.monotonic()-start>=max_seconds)):
                        print('V1_PAUSED:',stage,'NEXT_SEGMENT:',i,'REMOTE_OR_LOCAL_VERSION:',journal.p['version'],flush=True);return 'PAUSED'
                    journal.assert_current()
                    uid=p['plan']['units'][i-1]['unit_id'];prev=None
                    print('V1_SEGMENT_START:',stage,i,'/',n,flush=True)
                    if stage!='stage1':
                        previous=STAGES_V1[STAGES_V1.index(stage)-1];e=journal.p['records'][previous][i-1]
                        prev=journal.fetch(e,Path(tmp)/'incoming.bin')
                    try:
                        out=worker.run(uid,job=jobs[i-1] if stage=='stage1' else None,archive=prev)
                        # Verify full archive with the original checkpoint manager before the journal is advanced.
                        verifier=oas.Studio(work=worker.work,input_root=input_root)
                        ensure(verifier.store.completed(uid)==list(STAGES_V1[:STAGES_V1.index(stage)+1]),'Wrong segment checkpoint scope')
                        verifier.verify_archive_prefix(out,uid,stage)
                        journal.commit(stage,out)
                        worker.release(uid,out)
                        processed+=1
                        print('V1_SEGMENT_COMMITTED:',stage,i,'/',n,flush=True)
                    finally:
                        if prev is not None:Path(prev).unlink(missing_ok=True)
        print('V1_STAGE_COMPLETE:',stage,'SEGMENTS:',n,flush=True);return 'COMPLETE'


    def run_ffmpeg(args):
        # Long-task assembly is not constrained by an aggregate wall-clock timeout.
        subprocess.run(['ffmpeg','-nostdin','-hide_banner','-loglevel','error','-xerror','-y',*map(str,args)],check=True)


    def media_video_info(path):
        info=s1.read_media(path);videos=[s for s in info['streams'] if s['codec_type']=='video'];audios=[s for s in info['streams'] if s['codec_type']=='audio']
        ensure(len(videos)==1,'Expected one video stream');v=videos[0]
        count=v.get('nb_frames')
        if not str(count).isdigit():
            count=subprocess.check_output(['ffprobe','-v','error','-count_frames','-select_streams','v:0','-show_entries','stream=nb_read_frames','-of','default=nw=1:nk=1',str(path)],text=True).strip()
        return dict(frames=int(count),fps=Fraction(v['avg_frame_rate']),width=int(v['width']),height=int(v['height']),video_duration=s1.stream_duration(v,info),audio_duration=s1.stream_duration(audios[0],info) if len(audios)==1 else None,audio_tracks=len(audios),codec=v.get('codec_name'),pix_fmt=v.get('pix_fmt'),sar=v.get('sample_aspect_ratio'))


    def prepare_assembly_clip(video,narration,dest,raw_audio,geometry=None,expected_samples=None,copy_video=False):
        """Per-segment video CFR normalization + exact PCM timeline.
        Each segment ends on a frame boundary. Only <40ms silence padding may be added.
        Never concatenate independently encoded AAC packets (encoder delays accumulate).
        """
        with wave.open(str(narration),'rb') as w:
            ensure((w.getnchannels(),w.getsampwidth(),w.getframerate(),w.getcomptype())==(1,2,RATE,'NONE'),'Invalid normalized narration')
            samples=w.getnframes();ensure(samples>=RATE,'Invalid narration samples')
            if expected_samples is not None:ensure(samples==expected_samples,'Audio source sample count changed')
            frames=(samples+FRAME_SAMPLES-1)//FRAME_SAMPLES;pad=frames*FRAME_SAMPLES-samples
            v=media_video_info(video);g=(v['width'],v['height'])
            ensure(v['fps']==FPS and v['audio_tracks']==1 and all(x%2==0 for x in g),'Unexpected segment geometry/fps/audio')
            ensure(geometry is None or g==geometry,'Segment geometry changed; cannot assemble')
            ensure(abs(v['frames']-frames)<=5 and abs(v['video_duration']-samples/RATE)<=.5 and abs(v['audio_duration']-samples/RATE)<=.5,'Segment is truncated or out of sync')
            if copy_video and v['frames']==frames and v['codec']=='h264' and v['pix_fmt']=='yuv420p' and v['sar'] in (None,'1:1'):
                # One unit: preserve the already aligned H.264 picture, remux audio once.
                run_ffmpeg(['-i',video,'-map','0:v:0','-an','-c:v','copy','-video_track_timescale','25000',dest])
                print('V1_SINGLE_CLIP_VIDEO_COPY: no additional image re-encode',flush=True)
            else:
                run_ffmpeg(['-threads','1','-i',video,'-map','0:v:0','-an','-vf',f'fps={FPS},tpad=stop_mode=clone:stop_duration=0.24,trim=end_frame={frames},setpts=N/({FPS}*TB),setsar=1','-r',str(FPS),'-vsync','cfr','-c:v','libx264','-threads','2','-preset','veryfast','-crf','17','-pix_fmt','yuv420p','-video_track_timescale','25000',dest])
            aligned=media_video_info(dest)
            ensure(aligned['frames']==frames and aligned['fps']==FPS and (aligned['width'],aligned['height'])==g and aligned['audio_tracks']==0,'Aligned clip verification failed')
            copied=0
            while True:
                b=w.readframes(65536)
                if not b:break
                ensure(len(b)%2==0,'Invalid PCM');raw_audio.write(b);copied+=len(b)//2
            ensure(copied==samples,'Narration PCM truncated')
            raw_audio.write(b'\0'*(2*pad))
        return frames,samples,pad,g


    def publish_final(journal,*,work,output_root,input_root='/kaggle/input'):
        p=journal.p;n=len(p['plan']['units']);stage=route(p['plan'])[-1]
        if len(p['records'][stage])!=n:
            print('V1_FINAL_BLOCKED: complete all segments of',stage,'first',flush=True);return None
        output_root=Path(output_root);output_root.mkdir(parents=True,exist_ok=True)
        target=output_root/f'oas_{p["task_id"]}_final.mp4'
        if p['final'] is not None:
            if target.is_file() and not target.is_symlink() and artifact_record(target)==p['final']['record']:
                print('V1_FINAL_REUSED:',target,flush=True);return target
            with tempfile.TemporaryDirectory(prefix='oas-v1-final-fetch-',dir=output_root) as tmp:
                f=journal.fetch(p['final'],Path(tmp)/'final.mp4');os.replace(f,target)
            print('V1_FINAL_RESTORED:',target,flush=True);return target
        Path(work).mkdir(parents=True,exist_ok=True)
        with tempfile.TemporaryDirectory(prefix='oas-v1-assembly-',dir=work) as tmp:
            d=Path(tmp);geometry=None;frames=samples=padding=0;clips=[];frame_counts=[]
            with (d/'narration.s16le').open('wb') as raw:
                for i,e in enumerate(p['records'][stage],1):
                    journal.assert_current()
                    archive=journal.fetch(e,d/'incoming.bin');uid=e['unit_id']
                    studio=oas.Studio(work=d/'restore',input_root=input_root)
                    studio.restore(archive,uid)
                    ensure(studio.store.completed(uid)==list(STAGES_V1[:STAGES_V1.index(stage)+1]),'Wrong assembly checkpoint scope')
                    unit=studio.store.path(uid)
                    video=unit/stage/('final_video.mp4' if stage=='stage3' else 'ditto_video.mp4')
                    dest=d/f'clip_{i:08d}.mp4'
                    nf,ns,npad,geometry=prepare_assembly_clip(video,unit/'stage1'/'narration.wav',dest,raw,geometry,p['plan']['units'][i-1]['samples'],copy_video=(n==1))
                    frames+=nf;samples+=ns;padding+=npad;clips.append(dest.name);frame_counts.append(nf)
                    shutil.rmtree(unit);archive.unlink()
                    print('V1_ASSEMBLY_CLIP:',i,'/',n,'FRAMES:',nf,'PAD_SAMPLES:',npad,flush=True)
            ensure(frames*FRAME_SAMPLES==samples+padding and (d/'narration.s16le').stat().st_size==2*(samples+padding),'Final PCM timeline mismatch')
            seams=cont.smooth_joins([d/x for x in clips],frame_counts,d/'narration.s16le',d/'seam_patches',enabled=p['plan']['base']['config']['smooth_seams'])
            (d/'clips.ffconcat').write_text('ffconcat version 1.0\n'+''.join('file '+x+'\n' for x in clips),encoding='utf-8')
            # MP4 output is assembled using disk streams rather than storing all frames/samples in RAM.
            staged=d/'joined.mp4'
            run_ffmpeg(['-f','concat','-safe','1','-i',d/'clips.ffconcat','-f','s16le','-ar',str(RATE),'-ac','1','-i',d/'narration.s16le','-map','0:v:0','-map','1:a:0','-c:v','copy','-c:a','aac','-b:a','160k','-movflags','+faststart',staged])
            v=media_video_info(staged);duration=frames/FPS
            ensure(v['frames']==frames and v['fps']==FPS and (v['width'],v['height'])==geometry and v['audio_tracks']==1,'Final video frame/geometry verification failed')
            ensure(abs(v['video_duration']-duration)<=.05 and abs(v['audio_duration']-duration)<=.1,'Final audio/video timeline verification failed')
            # Full decode validates media integrity. No fixed aggregate duration limit.
            run_ffmpeg(['-threads','1','-i',staged,'-map','0:v:0','-map','0:a:0','-f','null','-'])
            media=dict(segments=n,seams=seams,speech_speed=p['plan']['base']['config']['speech_speed'],frames=frames,fps=FPS,width=geometry[0],height=geometry[1],original_pcm_samples=samples,padding_pcm_samples=padding,duration_seconds=duration,video_duration=v['video_duration'],audio_duration=v['audio_duration'])
            os.replace(staged,target)
            journal.commit('final',target,media)
            write_json(output_root/f'oas_{p["task_id"]}_final_report.json',media)
        print('V1_COMPLETE:',target,'DURATION:',duration,'SHA256:',artifact_record(target)['sha256'],flush=True)
        return target


    def assert_prefix(old,new):
        ensure(old['task_id']==new['task_id'] and new['version']>=old['version'],'Remote/local journal is older than the selected checkpoint')
        for stage in STAGES_V1:
            ensure(new['records'][stage][:len(old['records'][stage])]==old['records'][stage],'Journal history differs from selected checkpoint')
        ensure(old['final'] is None or new['final']==old['final'],'Final record history differs')


    def open_journal(work,ref='',plan=None,expected_id='',restore_manifest='',wait_seconds=600,command_seconds=3600):
        ensure(not expected_id or ishex(expected_id),'EXPECTED_TASK_ID must be blank or 64 hex characters')
        ensure(type(wait_seconds) is int and wait_seconds>0 and type(command_seconds) is int and command_seconds>0,'Remote timeouts must be positive integers')
        work=Path(work).resolve();work.mkdir(parents=True,exist_ok=True)
        old=None
        if restore_manifest:
            path=Path(restore_manifest).resolve(strict=True);old=read_json(path)
            validate_progress(old,old.get('dataset'),expected_id,plan)
            expected_id=old['task_id']
            if old['dataset']:
                ensure(not ref or ref==old['dataset'],'Restore manifest and PERSIST_DATASET disagree');ref=old['dataset']
            else:
                ensure(not ref,'Local journal cannot be promoted to remote automatically; use original local journal')
                ensure(path.name==PROGRESS_NAME,'Use exact oas-progress.json filename')
                dest=work/'journals'/expected_id
                if path.parent!=dest:
                    dest.mkdir(parents=True,exist_ok=True);marker=dest/PROGRESS_NAME
                    if marker.is_file():
                        present=validate_progress(read_json(marker),None,expected_id,plan);assert_prefix(old,present)
                    else:
                        for entry in [e for st in STAGES_V1 for e in old['records'][st]]+([old['final']] if old['final'] else []):
                            rel=Path('versions')/str(entry['version'])/entry['name'];src=path.parent/rel;dst=dest/rel
                            ensure(artifact_record(src)==entry['record'],'Local restore artifact missing or corrupt: '+str(rel))
                            dst.parent.mkdir(parents=True,exist_ok=True);shutil.copyfile(src,dst)
                            ensure(artifact_record(dst)==entry['record'],'Local restore copy changed')
                        write_json(marker,old)
                    return LocalJournal(dest,plan,expected_id)
                return LocalJournal(path.parent,plan,expected_id)
        if ref:
            validate_destination(ref)
            # Local snapshot is not authoritative; a newly started Session always reads latest remote index.
            folder=work/'remote-snapshots'/ref.replace('/','__')
            j=RemoteJournal.open_secret(ref,folder,plan,expected_id,wait_seconds,command_seconds)
            try:
                if old is not None:assert_prefix(old,j.p)
            except BaseException:
                j.close();raise
            return j
        if plan is None:
            ensure(expected_id,'Cold local recovery needs exact RESTORE_MANIFEST or EXPECTED_TASK_ID')
            folder=work/'journals'/expected_id
        else:
            tid=sha_json(plan);ensure(not expected_id or expected_id==tid,'EXPECTED_TASK_ID mismatch');folder=work/'journals'/tid
        print('V1_LOCAL_ONLY: /kaggle/working is NOT durable across full container loss. Configure PERSIST_DATASET for long jobs.',flush=True)
        return LocalJournal(folder,plan,expected_id)


    def choose_journal_for_cell(stage,work,ref,restore_manifest,expected_id,wait_seconds,command_seconds,active_ref='',active_manifest='',active_id=''):
        # Explicit selection is authoritative; never silently mix two jobs.
        if active_id:
            ensure(not expected_id or expected_id==active_id,'Active task and expected task differ')
            if ref and active_ref:ensure(ref==active_ref,'Active task and persistence destinations differ')
            if ref and active_manifest and not active_ref:raise RuntimeError('Cannot silently switch active local task to a new remote Dataset')
        return open_journal(work,ref or active_ref,expected_id=expected_id or active_id,restore_manifest=restore_manifest or (active_manifest if not (ref or active_ref) else ''),wait_seconds=wait_seconds,command_seconds=command_seconds)



    _OAS_V1_CONTEXT = None
    ensure(ALLOW_NONCOMMERCIAL_RESEARCH is True,'Review model terms and explicitly accept authorized noncommercial research before inference.')
    if PERSIST_DATASET:validate_destination(PERSIST_DATASET)
    _work=Path(WORK_ROOT).resolve();_work.mkdir(parents=True,exist_ok=True)
    with tempfile.TemporaryDirectory(prefix='oas-v1-input-plan-',dir=_work) as _plan_tmp:
        _mode,_paths=input_paths(INPUT_MODE,LANGUAGE,DEMO_PORTRAIT,DEMO_AUDIO,PORTRAIT,AUDIO,SCRIPT,REFERENCE_AUDIO,REFERENCE_TEXT,'/kaggle/input')
        _plan,_jobs=make_plan(_plan_tmp,mode=_mode,paths=_paths,language=LANGUAGE,blink_mode=BLINK_MODE,expression=EXPRESSION,seed=SEED,open_frames=DITTO_EYE_OPEN_FRAMES,chunk_chars=TTS_CHUNK_CHARACTERS,segment_seconds=DITTO_SEGMENT_SECONDS,text_segment_chars=TEXT_SEGMENT_CHARACTERS,warmup_seconds=DITTO_WARMUP_SECONDS,smooth_seams=SMOOTH_SEAMS,speech_speed=SPEECH_SPEED)
        TASK_ID=sha_json(_plan);print('V1_TASK_ID:',TASK_ID,'SEGMENTS:',len(_jobs),flush=True)
        _j=open_journal(_work,PERSIST_DATASET,_plan,EXPECTED_TASK_ID,STAGE1_RESTORE_MANIFEST,REMOTE_VERIFY_WAIT_SECONDS,REMOTE_COMMAND_TIMEOUT_SECONDS)
        try:
            _OAS_V1_CONTEXT=dict(task_id=_j.p['task_id'],dataset=_j.p['dataset'] or '',manifest=str(_j.path),work=str(_work))
            print('V1_PROGRESS_PATH:',_j.path,flush=True)
            _state=execute_stage(_j,'stage1',root=_OAS_ROOT,work=_work,input_root='/kaggle/input',jobs=_jobs,accept=ALLOW_NONCOMMERCIAL_RESEARCH,check_network=CHECK_NETWORK,monitor=MONITOR_RESOURCES,max_new=STAGE1_MAX_NEW_SEGMENTS,max_seconds=STAGE1_MAX_RUN_SECONDS)
            if _state=='COMPLETE' and not _j.p['plan']['use_lp']:
                publish_final(_j,work=_work,output_root='/kaggle/working')
            run_status=copy.deepcopy(_j.p)
        finally:_j.close()

else:
    _OAS_V1_CONTEXT = None
    print('V1_STAGE1_SKIPPED: no source/model initialization',flush=True)


In [1]:
# Open Avatar Studio V1.1 | Continuity First + Speech Speed | Long Video | 2/3 - Blink Enhancement
# GPU T4 x2 + Internet On. Attach LivePortrait model Dataset:
# https://www.kaggle.com/datasets/danielsundl/open-avatar-liveportrait
# Full run: enable RUN_STAGE_2, keep restore and destination fields blank.
# Cold recovery: Cell 1 False, set STAGE2_PERSIST_DATASET to SAME private task Dataset.
# Original portrait/audio/script and Qwen/Ditto Input are not required here.
# Restores historical Stage 1 archives using their pinned versions and SHA256.
# Local fallback: mount the COMPLETE saved journal directory, not just its JSON index.
# ===== USER CONFIGURATION / 用户配置 =====
RUN_STAGE_2 = True
STAGE2_PERSIST_DATASET = ''  # blank=inherit Cell 1/task context; cold run: same owner/oas-checkpoints-...
STAGE2_RESTORE_MANIFEST = ''  # optional exact trusted oas-progress.json path (NOT a legacy ZIP)
STAGE2_EXPECTED_TASK_ID = ''  # optional 64-hex long-task ID
STAGE2_ALLOW_NONCOMMERCIAL_RESEARCH = True  # None=inherit Cell 1; True only if authorized for standalone
STAGE2_MAX_NEW_SEGMENTS = 0  # positive=limit new commits this run; 0=all remaining
STAGE2_MAX_RUN_SECONDS = 0  # optional soft budget between segments; 0=no budget
STAGE2_REMOTE_VERIFY_WAIT_SECONDS = 600
STAGE2_REMOTE_COMMAND_TIMEOUT_SECONDS = 3600
STAGE2_CHECK_NETWORK = True
STAGE2_MONITOR_RESOURCES = True
STAGE2_WORK_ROOT = ''  # blank=inherit Cell 1/default
# ===== END USER CONFIGURATION / DO NOT EDIT BELOW =====

if type(RUN_STAGE_2) is not bool:
    raise ValueError("RUN_STAGE_2 must be True or False")
if RUN_STAGE_2:
    # Generated, content-verified V1.1 runtime. Do not edit the embedded payload.
    import base64 as _b64, hashlib as _hash, io as _io, json as _json
    import os as _os, shutil as _shutil, stat as _stat, sys as _sys
    import tempfile as _tempfile, zipfile as _zipfile
    from pathlib import Path as _Path

    _OAS_SHA = 'f0ddb13b6d236d1724f5bd9b8137a561405a3a22b9abb34758c0f5a771f3c6cc'
    _OAS_BUNDLE = '...'

    def _oas_bootstrap():
        data = _b64.b64decode(_OAS_BUNDLE, validate=True)
        if _hash.sha256(data).hexdigest() != _OAS_SHA:
            raise RuntimeError('V1.1 bundle SHA256 mismatch')
        with _zipfile.ZipFile(_io.BytesIO(data)) as z:
            manifest = _json.loads(z.read('runtime_manifest.json'))
            if manifest['schema'] != 'oas-runtime-bundle-v01' or manifest['release'] != '1.1.0':
                raise RuntimeError('Wrong runtime release')
            names = set(manifest['files']) | {'runtime_manifest.json'}
            if len(z.infolist()) != len(names) or set(z.namelist()) != names:
                raise RuntimeError('Unexpected runtime archive members')
            for m in z.infolist():
                if '/' in m.filename or '\\' in m.filename or m.filename in ('.','..') or _stat.S_IFMT(m.external_attr >> 16) not in (0,_stat.S_IFREG):
                    raise RuntimeError('Unsafe runtime archive member')
            expected = dict(manifest['files'])
            mbytes = z.read('runtime_manifest.json')
            expected['runtime_manifest.json'] = {'size':len(mbytes),'sha256':_hash.sha256(mbytes).hexdigest()}
            parent = _Path('/tmp/open-avatar-studio');parent.mkdir(parents=True,exist_ok=True)
            folder = parent/('runtime-'+_OAS_SHA[:16])
            def verify(path):
                if path.is_symlink() or not path.is_dir() or set(x.name for x in path.iterdir()) != names:
                    raise RuntimeError('Unexpected runtime directory contents: '+str(path))
                for n,rec in expected.items():
                    f=path/n
                    if f.is_symlink() or not f.is_file() or f.stat().st_size != rec['size'] or _hash.sha256(f.read_bytes()).hexdigest() != rec['sha256']:
                        raise RuntimeError('Runtime file mismatch: '+n)
            if not folder.exists():
                with _tempfile.TemporaryDirectory(dir=parent,prefix='.runtime-') as tmp:
                    stage=_Path(tmp)/'payload';stage.mkdir()
                    for n in names:
                        (stage/n).write_bytes(z.read(n))
                    verify(stage)
                    _os.rename(stage,folder)
            verify(folder)
        print('RUNTIME_BUNDLE_VERIFIED:',_OAS_SHA,flush=True)
        print('RUNTIME_FILES_VERIFIED:',len(expected),flush=True)
        return folder

    _OAS_ROOT = _oas_bootstrap()

    _sys.dont_write_bytecode = True
    for _name in ['resource_manager','auxiliary_manager','environment_manager','checkpoint_manager','demo_assets_builder','stage1_engine','stage23_engine','runtime_support','pipeline','continuity']:
        if _name in _sys.modules:
            _loaded = getattr(_sys.modules[_name],'__file__',None)
            if not _loaded or _Path(_loaded).resolve() != (_OAS_ROOT/(_name+'.py')).resolve():
                raise RuntimeError('Old module already loaded: '+_name+'. Start a NEW notebook Session.')
    if str(_OAS_ROOT) not in _sys.path:
        _sys.path.insert(0,str(_OAS_ROOT))
    import pipeline as oas

    """Optional private, versioned Kaggle persistence. No model or runtime changes."""
    import hashlib
    import json
    import os
    import re
    import shutil
    import subprocess
    import tempfile
    import time
    import uuid
    import zipfile
    from pathlib import Path


    class PersistenceError(RuntimeError):
        pass


    def digest(path):
        h = hashlib.sha256()
        size = 0
        with Path(path).open('rb') as f:
            for block in iter(lambda: f.read(1024*1024), b''):
                size += len(block)
                h.update(block)
        return {'bytes': size, 'sha256': h.hexdigest()}


    def validate_destination(ref):
        if not isinstance(ref, str) or not re.fullmatch(r'[a-z0-9][a-z0-9_-]{0,49}/[a-z0-9][a-z0-9-]{5,49}', ref):
            raise PersistenceError('PERSIST_DATASET must be a personal owner/slug (lowercase, slug 6-50 characters).')
        # Dedicated name only; no accidental overwrite of model/demo resources.
        if not ref.split('/')[1].startswith('oas-checkpoints-'):
            raise PersistenceError('PERSIST_DATASET slug must start with oas-checkpoints-. Use one dedicated task-specific name.')
        return ref


    class PrivateDatasetStore:
        manifest_name = "oas-progress.json"

        def __init__(self, ref, task_id, token, *, cli=None, wait_seconds=300, command_seconds=900,
                     run_command=subprocess.run, sleep=time.sleep, clock=time.monotonic):
            self.ref = validate_destination(ref)
            if not re.fullmatch('[0-9a-f]{64}', task_id):
                raise PersistenceError('Invalid task identity')
            if not isinstance(token, str) or not token.strip():
                raise PersistenceError('KAGGLE_API_TOKEN secret is empty or missing')
            self.task_id = task_id
            self.nonce = uuid.uuid4().hex
            self.cli = cli or shutil.which('kaggle')
            if not self.cli:
                raise PersistenceError('Kaggle CLI not installed. Use a Kaggle image with a token-capable CLI.')
            self.wait_seconds = wait_seconds
            self.command_seconds = command_seconds
            self._run, self._sleep, self._clock = run_command, sleep, clock
            self._tmp = tempfile.TemporaryDirectory(prefix='oas-kaggle-auth-')
            self._env = dict(os.environ)
            for key in ('KAGGLE_USERNAME','KAGGLE_KEY','KAGGLE_API_TOKEN','KAGGLE_CONFIG_DIR'):
                self._env.pop(key, None)
            self._env.update(KAGGLE_API_TOKEN=token, KAGGLE_CONFIG_DIR=self._tmp.name)
            self.version = 0
            self.last_receipt = None
            self._manifest = None
            self._records = {}
            self._preserved = {}  # Exact remote bytes of earlier checkpoints on cross-session recovery.

        def close(self):
            self._env.pop('KAGGLE_API_TOKEN', None)
            self._tmp.cleanup()

        def command(self, args, *, timeout=None):
            # Never include stdout/stderr in errors: CLI may include signed URLs.
            try:
                r = self._run([self.cli, 'datasets', *args], env=self._env,
                              stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
                              stderr=subprocess.PIPE, timeout=timeout or self.command_seconds,
                              check=False)
            except Exception:
                raise PersistenceError('KAGGLE_API_COMMAND_FAILED: '+args[0]) from None
            if r.returncode != 0:
                raise PersistenceError('KAGGLE_API_COMMAND_FAILED: '+args[0]+' exit='+str(r.returncode))
            return r

        @staticmethod
        def _privacy_flag(data):
            # Only explicit booleans (or exact text booleans from older clients).
            # Never accept the truthiness of a nonempty string or infer private from access.
            flags = []
            for k in ('isPrivate', 'is_private'):
                if k in data:
                    v = data[k]
                    if isinstance(v, bool):
                        flags.append(v)
                    elif isinstance(v, str) and v.strip().lower() in ('true', 'false'):
                        flags.append(v.strip().lower() == 'true')
                    elif v is not None:
                        raise PersistenceError('REMOTE_PRIVACY_INVALID: unrecognized '+k+' value type')
            if len(set(flags)) > 1:
                raise PersistenceError('REMOTE_PRIVACY_CONFLICT: inconsistent privacy fields')
            return flags[0] if flags else None

        def _metadata_privacy(self, meta):
            if not isinstance(meta, dict):
                raise PersistenceError('REMOTE_METADATA_INVALID: expected a JSON object')
            # Kaggle CLI can serialize the entire API response: {"info": DatasetInfo}.
            # Flat DatasetInfo and legacy upload-style metadata are also supported.
            nodes = [meta]
            if isinstance(meta.get('info'), dict):
                nodes.append(meta['info'])
            elif 'info' in meta and meta['info'] is not None:
                raise PersistenceError('REMOTE_METADATA_INVALID: unexpected info structure')
            owner, slug = self.ref.split('/')
            flags = []
            for node in nodes:
                if node.get('errorMessage') or node.get('error_message'):
                    raise PersistenceError('REMOTE_METADATA_ERROR: server returned an error')
                for key in ('ref', 'id'):
                    v = node.get(key)
                    # id may be numeric in API responses; only a path-like id is a ref.
                    if isinstance(v, str) and '/' in v and v.lower() != self.ref:
                        raise PersistenceError('REMOTE_METADATA_IDENTITY_MISMATCH: '+key)
                for key in ('datasetSlug', 'dataset_slug'):
                    v = node.get(key)
                    if isinstance(v, str) and v and v.lower() not in (slug, self.ref):
                        raise PersistenceError('REMOTE_METADATA_IDENTITY_MISMATCH: '+key)
                for key in ('ownerUser', 'owner_user', 'ownerSlug', 'owner_slug'):
                    v = node.get(key)
                    if isinstance(v, str) and v and v.lower() != owner:
                        raise PersistenceError('REMOTE_METADATA_IDENTITY_MISMATCH: '+key)
                v = self._privacy_flag(node)
                if v is not None:
                    flags.append(v)
            if len(set(flags)) > 1:
                raise PersistenceError('REMOTE_PRIVACY_CONFLICT: metadata and info disagree')
            return flags[0] if flags else None

        def _query_dataset_privacy(self):
            # Read-only fallback to the official SDK GetDataset endpoint. Run in the
            # same isolated token environment as CLI calls. Do not expose API output,
            # exception text, access tokens, signed URLs or user metadata in logs.
            import sys
            script = "import json, os, sys\nfrom kagglesdk import KaggleClient\nfrom kagglesdk.datasets.types.dataset_api_service import ApiGetDatasetRequest\nowner, slug = sys.argv[1].split('/')\nreq = ApiGetDatasetRequest()\nreq.owner_slug = owner\nreq.dataset_slug = slug\nwith KaggleClient(api_token=os.environ['KAGGLE_API_TOKEN']) as client:\n    item = client.datasets.dataset_api_client.get_dataset(req)\n    result = {'ref': item.ref, 'isPrivate': item.is_private}\nprint('OAS_PRIVATE_RESULT:' + json.dumps(result))\n"
            try:
                r = self._run([sys.executable, '-c', script, self.ref], env=self._env,
                              stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
                              stderr=subprocess.PIPE, timeout=60, check=False)
                if r.returncode != 0:
                    raise PersistenceError('REMOTE_PRIVACY_SDK_FAILED: exit='+str(r.returncode))
                raw = r.stdout.decode('utf-8', errors='replace') if isinstance(r.stdout, bytes) else str(r.stdout)
                matches = [x[len('OAS_PRIVATE_RESULT:'):] for x in raw.splitlines() if x.startswith('OAS_PRIVATE_RESULT:')]
                if len(matches) != 1:
                    raise PersistenceError('REMOTE_PRIVACY_SDK_FAILED: missing result')
                result = json.loads(matches[0])
            except PersistenceError:
                raise
            except Exception as exc:
                raise PersistenceError('REMOTE_PRIVACY_SDK_FAILED: '+type(exc).__name__) from None
            if not isinstance(result, dict) or result.get('ref') != self.ref:
                raise PersistenceError('REMOTE_METADATA_IDENTITY_MISMATCH: SDK ref')
            return self._privacy_flag(result)

        def _private(self):
            # Fail closed. Never upload to a public Dataset, or merely assume that
            # an accessible Dataset must be private. Handle nested API responses.
            with tempfile.TemporaryDirectory(prefix='oas-remote-meta-') as d:
                self.command(['metadata', self.ref, '-p', d], timeout=60)
                p = Path(d)/'dataset-metadata.json'
                try:
                    meta = json.loads(p.read_text(encoding='utf-8-sig'))
                except Exception:
                    raise PersistenceError('REMOTE_METADATA_INVALID: metadata JSON unavailable') from None
                privacy = self._metadata_privacy(meta)
                if privacy is None:
                    print('REMOTE_PRIVACY_METADATA_FIELD_MISSING: trying read-only SDK GetDataset', flush=True)
                    privacy = self._query_dataset_privacy()
                if privacy is False:
                    raise PersistenceError('Remote dataset privacy check: REMOTE_DATASET_PUBLIC; set Visibility to Private before retrying')
                if privacy is not True:
                    raise PersistenceError('Remote dataset privacy not explicitly confirmed; refusing to upload')

        def _assert_latest(self, expected):
            _, record=self._read_latest_manifest()
            if record!=expected:
                raise PersistenceError('REMOTE_VERSION_CHANGED: another version appeared; stop and inspect the latest Dataset before retrying. Do not run concurrent writers.')

        def _download_record(self, version, name, expected, destination=None):
            """Verify an exact version; optionally retain a verified copy, never unzip into a directory."""
            with tempfile.TemporaryDirectory(prefix='oas-remote-check-') as d:
                self.command(['download','-d',f'{self.ref}/{version}','-f',name,'-p',d,'-o','-q'])
                p=Path(d)/name
                if p.is_file() and not p.is_symlink():
                    actual=digest(p)
                    if actual!=expected:
                        raise PersistenceError('Remote byte verification failed: '+name)
                    if destination is not None: shutil.copyfile(p,destination)
                else:
                    wrapped=Path(d)/(name+'.zip')
                    if not wrapped.is_file():
                        raise PersistenceError('Remote file unavailable: '+name)
                    try:
                        with zipfile.ZipFile(wrapped) as z:
                            members=z.infolist()
                            if len(members)!=1 or members[0].filename!=name or members[0].file_size!=expected['bytes']:
                                raise PersistenceError('Unexpected remote archive wrapper: '+name)
                            h=hashlib.sha256();size=0
                            # A private scratch file, not a path derived from ZIP members.
                            staged=Path(d)/'verified_payload'
                            with z.open(members[0]) as f, staged.open('wb') as out:
                                for block in iter(lambda:f.read(1024*1024),b''):
                                    size+=len(block);h.update(block);out.write(block)
                            actual={'bytes':size,'sha256':h.hexdigest()}
                            if actual!=expected:
                                raise PersistenceError('Remote byte verification failed: '+name)
                            if destination is not None: shutil.copyfile(staged,destination)
                    except (OSError,zipfile.BadZipFile):
                        raise PersistenceError('Unreadable remote file: '+name) from None
                if destination is not None and digest(destination)!=expected:
                    raise PersistenceError('Verified copy changed: '+name)

        def _verify(self, version, manifest_record, files):
            deadline = self._clock()+self.wait_seconds
            while True:
                try:
                    self._download_record(version,self.manifest_name,manifest_record)
                    self._private()
                    for name,record in files.items():
                        self._download_record(version,name,record)
                    # A pinned version can be ready before the latest alias advances.
                    latest, observed = self._read_latest_manifest()
                    if observed != manifest_record:
                        validate_progress(latest,self.ref,self.task_id)
                        if latest['version'] < version:
                            raise PersistenceError('REMOTE_LATEST_ALIAS_PENDING')
                        raise PersistenceError('REMOTE_VERSION_CHANGED: another version appeared; stop and inspect the latest Dataset. Do not run concurrent writers.')
                    return
                except PersistenceError as exc:
                    reason = str(exc)
                    # Retry publication delays, not privacy, hash, identity, malformed
                    # manifest or concurrent-writer failures. Never log raw CLI output.
                    pending = reason.startswith(('KAGGLE_API_COMMAND_FAILED: download',
                                                  'Remote file unavailable:',
                                                  'REMOTE_LATEST_ALIAS_PENDING'))
                    if not pending:
                        raise PersistenceError('REMOTE_VERSION_VERIFY_FAILED: '+self.ref+'/'+str(version)+'; '+reason) from None
                    if self._clock() >= deadline:
                        raise PersistenceError('REMOTE_VERSION_NOT_VERIFIED: '+self.ref+'/'+str(version)+'; last reason: '+reason) from None
                    print('REMOTE_VERSION_WAIT:',self.ref,'VERSION:',version,'REASON:',reason,flush=True)
                    self._sleep(min(10,max(0,deadline-self._clock())))

        def assert_current(self):
            if self.version<1 or self._manifest is None:
                raise PersistenceError('No verified remote version')
            self._private()
            self._download_record(self.version,self.manifest_name,self._manifest)
            self._assert_latest(self._manifest)


    # Optional demo extension; the frozen runtime and original asset manifest stay unchanged.
    import hashlib
    import shutil
    import tempfile
    import wave
    from pathlib import Path
    import pipeline as oas
    import resource_manager as rm
    from demo_assets_builder import verify as verify_base_demo

    DEMO_AUDIO_FILES = ('demo_audio.wav', 'native_blink_long_test.wav')
    _DEMO_BASE = {'portrait_01.png','portrait_02.png','reference.wav','reference.txt',
                  'demo_audio.wav','script_en.txt','script_zh.txt','demo_config.json',
                  'PORTRAIT_LICENSE.txt','ASSET_NOTICES.md','asset_manifest.json'}
    _DEMO_LONG_RECORDS = {'native_blink_long_test.txt': {'size': 461, 'sha256': '88a5c2c85a00d960770a6016d1bc5f473cf67c320af682f8ef6a1f044c00d9fb'}, 'native_blink_long_test.wav': {'size': 1523172, 'sha256': 'fc8dbce194c574aa24653d7350823492050f26d3a4b0eae498864253a3c1ecf4'}}


    def verify_demo_mount(input_root='/kaggle/input'):
        root = rm.locate(input_root, 'open-avatar-demo-assets')
        actual = {p.relative_to(root).as_posix() for p in root.rglob('*') if p.is_file()}
        base = _DEMO_BASE | ({'dataset-metadata.json'} if 'dataset-metadata.json' in actual else set())
        extras = actual - base
        missing = _DEMO_BASE - actual
        if missing or extras not in (set(), set(_DEMO_LONG_RECORDS)):
            raise RuntimeError('DEMO_ASSET_SCOPE_MISMATCH: missing='+str(sorted(missing))+
                               '; extra='+str(sorted(extras))+'; attach original or official expanded Demo Assets')
        if not extras:
            verify_base_demo(root, mounted=True)
            return root, False
        # The older verifier intentionally rejects extra files. Verify an isolated copy
        # of the unchanged core, then verify both optional files against release pins.
        # Nothing is written to the mounted Dataset or to the frozen runtime.
        with tempfile.TemporaryDirectory(prefix='oas-demo-base-') as tmp:
            projection = Path(tmp)
            for name in sorted(base):
                shutil.copyfile(root/name, projection/name)
            verify_base_demo(projection, mounted=True)
        for name, expected in _DEMO_LONG_RECORDS.items():
            path = root/name
            if path.stat().st_size != expected['size'] or hashlib.sha256(path.read_bytes()).hexdigest() != expected['sha256']:
                raise RuntimeError('DEMO_LONG_ASSET_INTEGRITY_MISMATCH: '+name)
            print('DEMO_LONG_ASSET_VERIFIED:',name,flush=True)
        with wave.open(str(root/'native_blink_long_test.wav'),'rb') as w:
            duration = w.getnframes()/w.getframerate()
            if (w.getframerate(),w.getnchannels(),w.getsampwidth(),w.getcomptype()) != (24000,1,2,'NONE') or not 30 < duration <= 60:
                raise RuntimeError('DEMO_LONG_AUDIO_INVALID')
        print('DEMO_LONG_AUDIO_SECONDS:',round(duration,3),flush=True)
        return root, True


    def prepare_demo_job(mode, portrait='portrait_01.png', language='English',
                         demo_audio='demo_audio.wav', input_root='/kaggle/input', **opts):
        if portrait not in ('portrait_01.png','portrait_02.png'):
            raise ValueError('DEMO_PORTRAIT must be portrait_01.png or portrait_02.png')
        if mode == 'audio' and demo_audio not in DEMO_AUDIO_FILES:
            raise ValueError('DEMO_AUDIO must be demo_audio.wav or native_blink_long_test.wav')
        if mode not in ('audio','text') or (mode == 'text' and language not in ('English','Chinese')):
            raise ValueError('Unsupported demo mode/language; demo_text uses English or Chinese')
        root, has_long = verify_demo_mount(input_root)
        if mode == 'audio':
            if demo_audio == 'native_blink_long_test.wav' and not has_long:
                raise RuntimeError('DEMO_LONG_AUDIO_NOT_MOUNTED: select the expanded Demo Assets version, or use demo_audio.wav')
            print('DEMO_AUDIO_SELECTED:',demo_audio,flush=True)
            return oas.prepare_job(mode='audio',portrait=root/portrait,audio=root/demo_audio,**opts)
        return oas.prepare_job(mode='text',portrait=root/portrait,
                               script=root/('script_zh.txt' if language=='Chinese' else 'script_en.txt'),
                               reference_audio=root/'reference.wav',reference_text=root/'reference.txt',
                               language=language,**opts)


    def resolve_workflow(blink_mode, open_frames, export_checkpoint):
        if blink_mode not in ('liveportrait', 'ditto_template', 'ditto_native'):
            raise ValueError("BLINK_MODE must be liveportrait, ditto_template or ditto_native")
        if type(open_frames) is not int or open_frames < 0:
            raise ValueError('DITTO_EYE_OPEN_FRAMES must be a nonnegative integer')
        if export_checkpoint != 'auto' and type(export_checkpoint) is not bool:
            raise ValueError("EXPORT_CHECKPOINT must be 'auto', True or False")
        use_lp = blink_mode == 'liveportrait'
        export = use_lp if export_checkpoint == 'auto' else export_checkpoint
        if use_lp and not export:
            raise ValueError('LivePortrait needs a Stage 1 checkpoint; set EXPORT_CHECKPOINT to auto or True')
        return use_lp, export

    def configure_eye_mode(job, blink_mode, open_frames):
        # Identical to the tested Ditto-only experiment for the two direct modes.
        if blink_mode == 'liveportrait':
            if not job['config']['blink']['enabled']:
                raise ValueError('Invalid LivePortrait task configuration')
            return job  # preserve the exact original identity and model parameters
        if job['config']['blink']['enabled']:
            raise ValueError('Direct Ditto must not enable LivePortrait')
        ditto = job['config']['ditto']
        ditto['drive_eye'] = True
        if blink_mode == 'ditto_native':
            ditto['delta_eye_arr'] = None
            ditto['delta_eye_open_n'] = -1
        else:
            ditto.pop('delta_eye_arr', None)  # retain the pinned model's default template, if available
            ditto['delta_eye_open_n'] = open_frames
        job['identity'] = cp.make_identity(job['inputs'], job['config'], job['identity']['revisions'])
        job['task_id'] = cp.task_id(job['identity'])
        oas.verify_bound(job['identity'])
        return job

    # V1.1 continuity build: task identity changes with runtime and planning.
    SCHEDULER_SHA = '03b674ff888a2c7d8b36a1310b22b0ea9f3bb3a4c3d32583e724bb0c5d6dcf91'

    SEGMENT_WORKER_SOURCE = '"""A persistent stage-specific worker, with one verified runtime environment per stage.\nThe coordinator owns Kaggle credentials; this subprocess receives none.\n"""\nimport json, sys, traceback, shutil\nfrom pathlib import Path\nroot,specfile,stage=sys.argv[1:]\nsys.dont_write_bytecode=True;sys.path.insert(0,root)\nimport pipeline as oas\nimport checkpoint_manager as cp\nspec=json.loads(Path(specfile).read_text())\nwork=Path(spec[\'work\']);result=Path(spec[\'result\'])\nstudio=oas.Studio(work=work,input_root=spec[\'input_root\'],cache=spec[\'cache\'],accept_noncommercial_research=spec[\'accept\'],check_network=spec[\'check_network\'])\nfor line in sys.stdin:\n    cmd=json.loads(line)\n    if cmd.get(\'stop\'):break\n    tid=cmd[\'unit_id\']\n    try:\n        with oas.support.monitor(stage,None,gpu=stage!=\'stage3\',enabled=spec[\'monitor\']):\n            if stage==\'stage1\':\n                job=cmd[\'job\'];job[\'inputs\']={k:Path(v) for k,v in job[\'inputs\'].items()}\n                assert job[\'task_id\']==tid\n                studio.stage1(job,export=False)\n            else:\n                through=\'stage1\' if stage==\'stage2\' else \'stage2\'\n                studio.resume_archive(cmd[\'archive\'],tid,through)\n                if stage==\'stage2\':studio.stage2(tid,export=False)\n                else:studio.stage3(tid,export=False)\n            archive=studio.export(tid,stage)\n        cp.atomic_json(result,{\'unit_id\':tid,\'stage\':stage,\'archive\':str(archive),\'status\':\'OK\'})\n    except Exception:\n        traceback.print_exc();cp.atomic_json(result,{\'unit_id\':tid,\'stage\':stage,\'status\':\'FAILED\'});sys.exit(1)\n'

    """Open Avatar Studio 1.0: versioned, resumable long-task orchestration.
    The embedded 1.1 runtime is used ONLY as a bounded segment engine.
    No total text, duration or frame-count cap is imposed by this scheduler.
    """
    import copy
    import hashlib
    import json
    import math
    import os
    import re
    import shutil
    import subprocess
    import sys
    import tempfile
    import time
    import wave
    import zipfile
    from fractions import Fraction
    from pathlib import Path

    import checkpoint_manager as cp
    import environment_manager as em
    import pipeline as oas
    import stage1_engine as s1
    import continuity as cont

    SCHEMA_V1 = 'oas-long-task-v1'
    PROGRESS_NAME = 'oas-progress.json'
    STAGES_V1 = ('stage1','stage2','stage3')
    RATE = 24000
    FPS = 25
    FRAME_SAMPLES = RATE // FPS
    ENGINE_SHA = 'f0ddb13b6d236d1724f5bd9b8137a561405a3a22b9abb34758c0f5a771f3c6cc'
    # A separate scheduler build fingerprint is injected by the notebook builder.


    def ensure(ok, message):
        if not ok:
            raise RuntimeError(message)


    def canonical(obj):
        return json.dumps(obj, sort_keys=True, ensure_ascii=False, separators=(',',':'), allow_nan=False).encode('utf-8')


    def sha_json(obj):
        return hashlib.sha256(canonical(obj)).hexdigest()


    def write_json(path, obj):
        # The legacy per-segment manifest size limits must not cap a long task index.
        path = Path(path); path.parent.mkdir(parents=True,exist_ok=True)
        with tempfile.NamedTemporaryFile(dir=path.parent, prefix='.oas-',delete=False) as f:
            p=Path(f.name)
            try:
                f.write(canonical(obj)+b'\n'); f.flush(); os.fsync(f.fileno()); f.close()
                os.replace(p,path)
            finally:
                p.unlink(missing_ok=True)


    def read_json(path):
        p=Path(path)
        ensure(p.is_file() and not p.is_symlink(),'Missing regular progress file: '+str(p))
        return json.loads(p.read_text(encoding='utf-8'))


    def ishex(v):
        return isinstance(v,str) and re.fullmatch(r'[0-9a-f]{64}',v) is not None


    def artifact_record(path):
        r=cp.file_record(path); ensure(r['bytes']>0,'Empty artifact'); return r


    def check_record(r):
        ensure(isinstance(r,dict) and set(r)=={'bytes','sha256'} and type(r['bytes']) is int and r['bytes']>0 and ishex(r['sha256']), 'Invalid artifact digest')


    def text_chunks(text, limit):
        """Linear scan with punctuation-aware boundaries; only whitespace is trimmed.
        No total length check; never drop any non-whitespace character.
        """
        ensure(type(limit) is int and 80<=limit<=6000,'Invalid text partition limit (80-6000)')
        text=text.strip(); ensure(bool(text),'Script is empty')
        pos=0; out=[]
        while pos<len(text):
            end=min(len(text),pos+limit)
            if end<len(text):
                window=text[pos:end]; floor=max(1,limit//2)
                for pat in (r'[\u3002\uff01\uff1f.!?;\uff1b\n]+',r'[\uff0c\u3001\uff1a,:]+',r'\s+'):
                    choices=[m.end() for m in re.finditer(pat,window) if m.end()>=floor]
                    if choices:
                        end=pos+choices[-1];break
            chunk=text[pos:end].strip()
            if chunk: out.append(chunk)
            pos=end
        ensure(bool(out),'No speakable text chunks')
        return out


    def audio_chunks(source, folder, seconds, speech_speed=1.0):
        """Decode once as raw PCM; bounded RAM, frame-aligned quiet cuts.
        A one-second lookahead prevents an invalid sub-second final segment.
        Works beyond RIFF/WAV's aggregate 4GiB limit. Every sample is retained.
        """
        import numpy as np
        ensure(type(seconds) is int and 10<=seconds<=600,'DITTO_SEGMENT_SECONDS must be 10-600')
        info=s1.read_media(source)
        tracks=[x for x in info['streams'] if x['codec_type']=='audio']
        ensure(len(tracks)==1,'Exactly one audio track required')
        duration=s1.stream_duration(tracks[0],info)
        speed=s1.validate_speech_speed(speech_speed)
        ensure(duration>=1,'Narration must be at least one second')
        maximum=seconds*RATE; lookahead=min(10*RATE,maximum//3)
        folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)
        count=0; total=0; records=[]; buf=b''; eof=False
        with tempfile.TemporaryFile() as err:
            proc=subprocess.Popen(['ffmpeg','-nostdin','-v','error','-xerror','-i',str(source),'-map','0:a:0','-vn','-ac','1','-ar',str(RATE),*s1.speech_speed_filter(speed),'-f','s16le','pipe:1'],stdout=subprocess.PIPE,stderr=err)
            try:
                while True:
                    want=2*(maximum+lookahead)-len(buf)
                    if not eof and want>0:
                        part=proc.stdout.read(want)
                        buf+=part
                        if len(part)<want: eof=True
                    ensure(len(buf)%2==0,'Unaligned PCM stream')
                    n=len(buf)//2
                    if n==0 and eof:break
                    if eof and n<=maximum:
                        ensure(n>=RATE,'Decoded narration is shorter than one second')
                        end=n
                    else:
                        hi=min(maximum,n-lookahead)//FRAME_SAMPLES*FRAME_SAMPLES
                        lo=max(RATE,hi-5*RATE);lo=(lo+FRAME_SAMPLES-1)//FRAME_SAMPLES*FRAME_SAMPLES
                        ensure(lo<=hi,'Cannot partition audio with a valid tail')
                        signal=np.frombuffer(buf,dtype='<i2')
                        def energy(i):
                            x=signal[max(0,i-4*FRAME_SAMPLES):min(n,i+4*FRAME_SAMPLES)].astype(np.float64)
                            return (float(np.mean(x*x)),-i)
                        end=min(range(lo,hi+1,FRAME_SAMPLES),key=energy)
                    count+=1;total+=end
                    path=folder/f'part_{count:08d}.wav'
                    with wave.open(str(path),'wb') as w:
                        w.setnchannels(1);w.setsampwidth(2);w.setframerate(RATE);w.writeframes(buf[:2*end])
                    records.append((path,end));buf=buf[2*end:]
                ensure(proc.wait()==0,'FFmpeg audio decode failed; source may be incomplete')
            finally:
                if proc.poll() is None:
                    proc.kill();proc.wait()
                proc.stdout.close()
        ensure(records and total>=RATE and all(n%FRAME_SAMPLES==0 for _,n in records[:-1]),'Invalid audio partition')
        if speed != 1.0:
            print('SPEECH_SPEED_APPLIED:',speed,'INPUT_SECONDS:',duration,'OUTPUT_SECONDS:',total/RATE,'SCOPE: continuous input before partition',flush=True)
        print('AUDIO_PLAN:',count,'SEGMENTS, EXACT_PCM_SAMPLES:',total,'SECONDS:',total/RATE,flush=True)
        return records


    def input_paths(mode, language, demo_portrait, demo_audio, portrait, audio, script, reference_audio, reference_text, input_root):
        ensure(mode in ('demo_audio','demo_text','audio','text'),'Invalid INPUT_MODE')
        if mode.startswith('demo_'):
            root,has_long=verify_demo_mount(input_root)
            ensure(demo_portrait in ('portrait_01.png','portrait_02.png'),'Invalid DEMO_PORTRAIT')
            portrait=root/demo_portrait
            if mode=='demo_audio':
                ensure(demo_audio in DEMO_AUDIO_FILES,'Invalid DEMO_AUDIO')
                ensure(demo_audio!='native_blink_long_test.wav' or has_long,'Long demo audio not mounted')
                audio=root/demo_audio;mode='audio'
            else:
                ensure(language in ('English','Chinese'),'Demo text language must be English or Chinese')
                script=root/('script_zh.txt' if language=='Chinese' else 'script_en.txt')
                reference_audio=root/'reference.wav';reference_text=root/'reference.txt';mode='text'
        paths={'portrait':s1.input_file(portrait)}
        s1.inspect_photo(paths['portrait'])
        if mode=='audio': paths['audio']=s1.input_file(audio)
        else:
            ensure(language in ('English','Chinese','Auto'),'Invalid LANGUAGE')
            paths.update(script=s1.input_file(script),reference_audio=s1.input_file(reference_audio))
            s1.inspect_audio(paths['reference_audio'],30)
            if reference_text: paths['reference_text']=s1.input_file(reference_text)
        return mode,paths


    def make_plan(folder, *, mode, paths, language, blink_mode, expression, seed, open_frames, chunk_chars, segment_seconds, text_segment_chars=2400, warmup_seconds=1.2, smooth_seams=True, speech_speed=1.0):
        use_lp,_=resolve_workflow(blink_mode,open_frames,True)
        speed=s1.validate_speech_speed(speech_speed)
        print('SPEECH_SPEED:',speed,'MODE:',mode,flush=True)
        ensure(type(seed) is int and 0<=seed<2**32,'Invalid SEED')
        ensure(type(chunk_chars) is int and 80<=chunk_chars<=800,'TTS_CHUNK_CHARACTERS must be 80-800')
        ensure(type(text_segment_chars) is int and max(600,chunk_chars)<=text_segment_chars<=4000,'TEXT_SEGMENT_CHARACTERS must be 600-4000 and >= TTS_CHUNK_CHARACTERS')
        ensure(type(warmup_seconds) in (int,float) and math.isfinite(warmup_seconds) and 0<=warmup_seconds<=3,'DITTO_WARMUP_SECONDS must be 0-3')
        ensure(type(smooth_seams) is bool,'SMOOTH_SEAMS must be True or False')
        ensure(type(segment_seconds) is int and 10<=segment_seconds<=600,'DITTO_SEGMENT_SECONDS must be 10-600')
        folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)
        config=dict(mode=mode,language=language if mode=='text' else None,blink_mode=blink_mode,expression=expression,seed=seed,open_frames=open_frames,chunk_chars=chunk_chars,segment_seconds=segment_seconds,text_segment_chars=text_segment_chars,warmup_seconds=warmup_seconds,smooth_seams=smooth_seams,speech_speed=speed)
        inputs={k:artifact_record(v) for k,v in paths.items()}
        # The plan identity contains source hashes and behavior settings, never paths or Secrets.
        base=dict(engine_sha=ENGINE_SHA,scheduler_sha=SCHEDULER_SHA,inputs=inputs,config=config)
        base_id=sha_json(base)
        if mode=='text':
            text=paths['script'].read_text(encoding='utf-8-sig')
            chunks=text_chunks(text,text_segment_chars)
            print('SCRIPT_CHARACTERS:',len(text.strip()),'TASK_SEGMENTS:',len(chunks),'TTS_TARGET_CHARACTERS:',chunk_chars,flush=True)
            parts=[]
            for i,chunk in enumerate(chunks,1):
                p=folder/f'part_{i:08d}.txt';p.write_text(chunk,encoding='utf-8');parts.append((p,None))
        else:
            parts=audio_chunks(paths['audio'],folder,segment_seconds,speech_speed=speed)
        jobs=[];units=[]
        for i,(part,samples) in enumerate(parts,1):
            # The old runtime's duration/character limits apply ONLY to this small unit.
            opts=dict(portrait=paths['portrait'],language=language,seed=(seed if mode=='text' else (seed+i-1)%2**32),expression=expression,blink_enabled=use_lp,
                      max_script_characters=text_segment_chars,tts_chunk_characters=chunk_chars,max_video_seconds=(1200 if mode=='text' else 600),ditto_segment_seconds=segment_seconds,ditto_warmup_seconds=warmup_seconds,smooth_seams=smooth_seams,speech_speed=(speed if mode=='text' else 1.0))
            if mode=='text':
                job=oas.prepare_job(mode='text',script=part,reference_audio=paths['reference_audio'],reference_text=paths.get('reference_text'),**opts)
            else:
                job=oas.prepare_job(mode='audio',audio=part,**opts)
            job=configure_eye_mode(job,blink_mode,open_frames)
            job['config']['blink']['seed']=(seed+i-1)%2**32
            # For outer AUDIO partitions, use true adjacent audio as inference context.
            # Context is hashed with the unit; it is NOT included in the final narration.
            if mode=='audio' and warmup_seconds>0:
                pre=round(warmup_seconds*FPS)*FRAME_SAMPLES;post=min(round(warmup_seconds*FPS),12)*FRAME_SAMPLES
                for label,neighbor,amount,tail in [('context_before',i-2,pre,True),('context_after',i,post,False)]:
                    if amount and 0<=neighbor<len(parts):
                        with wave.open(str(parts[neighbor][0]),'rb') as w:
                            n=min(amount,w.getnframes())
                            if tail:w.setpos(w.getnframes()-n)
                            data=w.readframes(n)
                        ensure(len(data)==2*n and n>0,'Missing adjacent PCM context')
                        extra=folder/f'{label}_{i:08d}.pcm';extra.write_bytes(data);job['inputs'][label]=extra
            revisions=dict(job['identity']['revisions'],long_task_source=base_id,long_task_index=i)
            job['identity']=cp.make_identity(job['inputs'],job['config'],revisions)
            job['task_id']=cp.task_id(job['identity'])
            units.append(dict(index=i,unit_id=job['task_id'],samples=samples,source=artifact_record(part)))
            jobs.append(job)
        ensure({k:artifact_record(v) for k,v in paths.items()}==inputs,'Source changed while planning')
        plan=dict(base=base,units=units,use_lp=use_lp)
        return plan,jobs


    def expected_version(stage,index,n):
        return 1+STAGES_V1.index(stage)*n+index


    def artifact_name(stage,index):
        return f'{stage}_{index:08d}.bin'


    def route(plan):
        return STAGES_V1 if plan['use_lp'] else ('stage1',)


    def validate_progress(p, ref=None, expected_id='', expected_plan=None):
        ensure(isinstance(p,dict) and set(p)=={'schema','dataset','task_id','plan','version','records','final'},'Unrecognized V1.0 progress index')
        ensure(p['schema']==SCHEMA_V1 and p['dataset']==ref,'Wrong Dataset or journal schema')
        plan=p['plan'];tid=p['task_id']
        ensure(isinstance(plan,dict) and set(plan)=={'base','units','use_lp'} and ishex(tid) and sha_json(plan)==tid,'Task identity mismatch')
        ensure(not expected_id or tid==expected_id,'EXPECTED_TASK_ID mismatch')
        ensure(expected_plan is None or plan==expected_plan,'REMOTE_TASK_MISMATCH: source/config/segmentation differs; use original settings or a new Dataset')
        base=plan['base']; cfg=base['config']
        ensure(base['engine_sha']==ENGINE_SHA and base['scheduler_sha']==SCHEDULER_SHA,'Use the matching Notebook build (V1.0 or V1.1 RC checkpoints need their original notebook)')
        s1.validate_speech_speed(cfg['speech_speed'])
        ensure(type(plan['use_lp']) is bool and (cfg['blink_mode']=='liveportrait')==plan['use_lp'],'Invalid route')
        units=plan['units'];n=len(units)
        ensure(isinstance(units,list) and n>0,'Empty plan')
        for i,u in enumerate(units,1):
            ensure(isinstance(u,dict) and set(u)=={'index','unit_id','samples','source'} and u['index']==i and ishex(u['unit_id']),'Invalid segment plan')
            check_record(u['source'])
            if cfg['mode']=='audio':
                ensure(type(u['samples']) is int and u['samples']>=RATE and (i==n or u['samples']%FRAME_SAMPLES==0),'Invalid audio sample partition')
            else: ensure(cfg['mode']=='text' and u['samples'] is None,'Invalid text plan')
        ensure(len({u['unit_id'] for u in units})==n,'Duplicate unit identity')
        records=p['records'];ensure(isinstance(records,dict) and set(records)==set(STAGES_V1),'Invalid stage index')
        count=0
        for stage in STAGES_V1:
            entries=records[stage]
            ensure(isinstance(entries,list) and len(entries)<=n,'Invalid stage count')
            ensure(stage in route(plan) or not entries,'Unexpected stage for direct mode')
            if stage!='stage1' and entries:
                ensure(len(records[STAGES_V1[STAGES_V1.index(stage)-1]])==n,'Stage order violated')
            for i,e in enumerate(entries,1):
                ensure(isinstance(e,dict) and set(e)=={'version','name','record','unit_id'} and e['version']==expected_version(stage,i,n) and e['name']==artifact_name(stage,i) and e['unit_id']==units[i-1]['unit_id'],'Invalid history record')
                check_record(e['record'])
            count+=len(entries)
        if p['final'] is not None:
            f=p['final'];ensure(all(len(records[s])==n for s in route(plan)),'Final published before all segments completed')
            ensure(isinstance(f,dict) and set(f)=={'version','name','record','media'} and f['version']==count+2 and f['name']==f'oas_{tid}_final.mp4','Invalid final record')
            check_record(f['record']); ensure(isinstance(f['media'],dict),'Missing final media report')
        ensure(type(p['version']) is int and p['version']==1+count+(p['final'] is not None),'Invalid Dataset version sequence')
        return p


    def new_progress(plan,ref):
        p=dict(schema=SCHEMA_V1,dataset=ref,task_id=sha_json(plan),plan=plan,version=1,records={s:[] for s in STAGES_V1},final=None)
        return validate_progress(p,ref)


    def advance(p,stage,source,media=None):
        """Append one immutable artifact reference; never modify previous records."""
        validate_progress(p,p['dataset']);q=copy.deepcopy(p);n=len(q['plan']['units'])
        ensure(q['final'] is None,'Task already complete')
        q['version']+=1
        rec=artifact_record(source)
        if stage=='final':
            e=dict(version=q['version'],name=f"oas_{q['task_id']}_final.mp4",record=rec,media=media)
            q['final']=e
        else:
            ensure(stage in route(q['plan']),'Invalid stage')
            i=len(q['records'][stage])+1
            ensure(i<=n,'Stage already complete')
            e=dict(version=q['version'],name=artifact_name(stage,i),record=rec,unit_id=q['plan']['units'][i-1]['unit_id'])
            q['records'][stage].append(e)
        validate_progress(q,q['dataset'])
        return q,e


    def latest_entry(p):
        if p['final']: return p['final']
        for stage in reversed(STAGES_V1):
            if p['records'][stage]: return p['records'][stage][-1]
        return None


    def print_progress(p):
        n=len(p['plan']['units']); stages=route(p['plan'])
        nxt=next((f'{s}:{len(p["records"][s])+1}' for s in stages if len(p['records'][s])<n),'final' if p['final'] is None else 'complete')
        print('V1_PROGRESS:',p['task_id'],'VERSION:',p['version'],'TOTAL_SEGMENTS:',n,'DONE:',{s:len(p['records'][s]) for s in stages},'NEXT:',nxt,flush=True)


    class LocalJournal:
        """Local fallback. A complete journal directory is required for local recovery."""
        def __init__(self,folder,plan=None,expected_id=''):
            self.folder=Path(folder).resolve();self.folder.mkdir(parents=True,exist_ok=True)
            self.path=self.folder/PROGRESS_NAME
            if self.path.exists():self.p=validate_progress(read_json(self.path),None,expected_id,plan)
            else:
                ensure(plan is not None,'No local journal; specify remote Dataset or exact local progress path')
                self.p=new_progress(plan,None);ensure(not expected_id or self.p['task_id']==expected_id,'EXPECTED_TASK_ID mismatch');write_json(self.path,self.p)
            last=latest_entry(self.p)
            if last:self._verify(last)
            print_progress(self.p)
        def _path(self,e):return self.folder/'versions'/str(e['version'])/e['name']
        def _verify(self,e):
            p=self._path(e);ensure(artifact_record(p)==e['record'],'Missing or corrupt local checkpoint: '+str(p));return p
        def assert_current(self):ensure(read_json(self.path)==self.p,'Local journal changed; do not run concurrent writers')
        def commit(self,stage,source,media=None):
            self.assert_current();q,e=advance(self.p,stage,source,media)
            dst=self._path(e);dst.parent.mkdir(parents=True,exist_ok=True)
            # If killed after file copy but before index commit, retry from that last verified index.
            tmp=dst.with_name(dst.name+'.partial');shutil.copyfile(source,tmp)
            ensure(artifact_record(tmp)==e['record'],'Local copy changed');os.replace(tmp,dst)
            write_json(self.path,q);self.p=q;print_progress(q)
            return e
        def fetch(self,e,dest):
            shutil.copyfile(self._verify(e),dest);ensure(artifact_record(dest)==e['record'],'Local readback mismatch')
            return Path(dest)
        def close(self):pass


    class RemoteJournal(PrivateDatasetStore):
        """One private Dataset, append-only Kaggle versions, rolling payloads.
        Prior versions are retained; latest index points to all historical artifacts.
        No cumulative copying of old media on each commit. Single writer only.
        """
        manifest_name=PROGRESS_NAME
        def __init__(self,ref,token,folder,plan=None,expected_id='',wait_seconds=600,command_seconds=3600,**kw):
            super().__init__(ref,'0'*64,token,wait_seconds=wait_seconds,command_seconds=command_seconds,**kw)
            self.folder=Path(folder);self.folder.mkdir(parents=True,exist_ok=True);self.path=self.folder/PROGRESS_NAME
            try:
                try:self._private()
                except PersistenceError as exc:
                    # A metadata command failure does NOT prove absence. Only attempt create-only.
                    if plan is None or not str(exc).startswith('KAGGLE_API_COMMAND_FAILED: metadata'):raise
                    self.p=new_progress(plan,ref);self.task_id=self.p['task_id']
                    ensure(not expected_id or self.task_id==expected_id,'EXPECTED_TASK_ID mismatch')
                    self._write_index(self.p,None,None,create=True)
                else:
                    p,rec=self._read_latest_manifest();validate_progress(p,ref,expected_id,plan)
                    self.task_id=p['task_id']
                    self._download_record(p['version'],self.manifest_name,rec)
                    last=latest_entry(p)
                    if last:self._download_record(last['version'],last['name'],last['record'])
                    self._private();self._assert_latest(rec)
                    self.p=p;self.version=p['version'];self._manifest=rec;write_json(self.path,p)
                ensure(not expected_id or self.p['task_id']==expected_id,'EXPECTED_TASK_ID mismatch')
                print('V1_REMOTE_CONNECTED:',self.ref,'VERSION:',self.p['version'],flush=True);print_progress(self.p)
            except BaseException:
                self.close();raise
        @classmethod
        def open_secret(cls,ref,folder,plan=None,expected_id='',wait_seconds=600,command_seconds=3600):
            validate_destination(ref)
            try:
                from kaggle_secrets import UserSecretsClient
                token=UserSecretsClient().get_secret('KAGGLE_API_TOKEN')
            except Exception:raise PersistenceError('Add KAGGLE_API_TOKEN in Secrets and grant this PRIVATE Notebook access.') from None
            return cls(ref,token,folder,plan,expected_id,wait_seconds,command_seconds)
        def _read_latest_manifest(self):
            # No fixed total task-index size cap. Only a regular index, or a one-file ZIP wrapper.
            with tempfile.TemporaryDirectory(prefix='oas-v1-latest-') as d:
                self.command(['download','-d',self.ref,'-f',self.manifest_name,'-p',d,'-o','-q'])
                p=Path(d)/self.manifest_name
                if p.is_file() and not p.is_symlink():data=p.read_bytes()
                else:
                    try:
                        with zipfile.ZipFile(Path(d)/(self.manifest_name+'.zip')) as z:
                            items=z.infolist()
                            if len(items)!=1 or items[0].filename!=self.manifest_name:raise PersistenceError('Unexpected progress ZIP wrapper')
                            data=z.read(items[0])
                    except (OSError,zipfile.BadZipFile):raise PersistenceError('Cannot read latest progress index') from None
                try:return json.loads(data),{'bytes':len(data),'sha256':hashlib.sha256(data).hexdigest()}
                except ValueError:raise PersistenceError('Malformed progress index') from None
        def _write_index(self,q,e,source,create=False):
            validate_progress(q,self.ref)
            if not create:self.assert_current()
            with tempfile.TemporaryDirectory(prefix='oas-v1-upload-') as d:
                d=Path(d);write_json(d/self.manifest_name,q);mrec=artifact_record(d/self.manifest_name);records={}
                if e is not None:
                    shutil.copyfile(source,d/e['name']);ensure(artifact_record(d/e['name'])==e['record'],'Artifact changed during upload staging');records[e['name']]=e['record']
                write_json(d/'dataset-metadata.json',{'id':self.ref,'title':self.ref.split('/')[1],'licenses':[{'name':'other'}],'description':'Private Open Avatar Studio V1.1 long-task recovery. Keep historical versions until final output is verified.'})
                print('V1_REMOTE_SAVE:',self.ref,'VERSION:',q['version'],'ARTIFACT:',e['name'] if e else 'plan',flush=True)
                if create:self.command(['create','-p',str(d),'-t','-r','skip','-q'])
                else:
                    # Explicitly never pass --delete-old-versions. Historical versions contain the media.
                    self.assert_current();self.command(['version','-p',str(d),'-m',f'OAS V1.1 {q["version"]} {e["name"]}','-t','-r','skip','-q'])
                self._verify(q['version'],mrec,records)
            self.p=q;self.version=q['version'];self._manifest=mrec;write_json(self.path,q)
            print('V1_REMOTE_VERIFIED:',f'https://www.kaggle.com/datasets/{self.ref}/versions/{self.version}',flush=True);print_progress(q)
        def commit(self,stage,source,media=None):
            q,e=advance(self.p,stage,source,media);self._write_index(q,e,source);return e
        def fetch(self,e,dest):
            self._private();self._download_record(e['version'],e['name'],e['record'],destination=dest);return Path(dest)

    class SegmentWorker:
        """Reuse one subprocess per stage; each stage has its own fresh cache.
        The frozen runtime's cross-stage GPU guard remains active inside each cache.
        """
        def __init__(self,stage,root,folder,input_root,accept,check_network,monitor):
            self.stage=stage;self.folder=Path(folder);self.folder.mkdir(parents=True,exist_ok=True)
            self.work=self.folder/'work';self.cache=self.folder/'cache';self.cache.mkdir()
            self.result=self.folder/'result.json'
            script=self.folder/'worker.py';script.write_text(SEGMENT_WORKER_SOURCE,encoding='utf-8')
            spec=self.folder/'spec.json';write_json(spec,dict(work=str(self.work),cache=str(self.cache),result=str(self.result),input_root=str(input_root),accept=accept,check_network=check_network,monitor=monitor))
            self.proc=subprocess.Popen([sys.executable,'-B',str(script),str(root),str(spec),stage],stdin=subprocess.PIPE,text=True,bufsize=1,env=em.worker_env(self.cache,cpu=stage=='stage3'),start_new_session=True)
        def __enter__(self):return self
        def __exit__(self,*exc):self.close()
        def run(self,unit_id,job=None,archive=None):
            self.result.unlink(missing_ok=True)
            cmd={'unit_id':unit_id}
            if job is not None:cmd['job']=dict(job,inputs={k:str(v) for k,v in job['inputs'].items()})
            if archive is not None:cmd['archive']=str(archive)
            self.proc.stdin.write(json.dumps(cmd)+'\n');self.proc.stdin.flush()
            while not self.result.is_file():
                ensure(self.proc.poll() is None,'Segment worker exited before commit; resume from last verified index')
                time.sleep(.25)
            r=read_json(self.result)
            ensure(r.get('status')=='OK' and r.get('unit_id')==unit_id and r.get('stage')==self.stage,'Segment inference failed; previous commits remain valid')
            result=Path(r['archive']).resolve()
            ensure(result.is_relative_to(self.work.resolve()) and result.is_file() and not result.is_symlink(),'Unexpected segment output')
            return result
        def release(self,unit_id,archive):
            # Only after journal commit. Clean this worker's own scratch media, never the journal.
            ensure(ishex(unit_id),'Invalid unit ID')
            shutil.rmtree(self.work/'tasks'/unit_id)
            Path(archive).unlink()
        def close(self):
            if self.proc.poll() is None:
                try:self.proc.stdin.write('{"stop":true}\n');self.proc.stdin.flush();self.proc.wait(timeout=10)
                except (OSError,subprocess.TimeoutExpired):
                    import signal
                    try:os.killpg(self.proc.pid,signal.SIGTERM)
                    except ProcessLookupError:pass
                    try:self.proc.wait(timeout=10)
                    except subprocess.TimeoutExpired:
                        try:os.killpg(self.proc.pid,signal.SIGKILL)
                        except ProcessLookupError:pass
                        self.proc.wait()
            if self.proc.stdin:self.proc.stdin.close()


    def stage_ready(journal,stage):
        p=journal.p;n=len(p['plan']['units'])
        if stage not in route(p['plan']):return False
        if stage=='stage1':return True
        previous=STAGES_V1[STAGES_V1.index(stage)-1]
        if len(p['records'][previous])!=n:
            print('V1_STAGE_BLOCKED:',stage,'WAITING_FOR:',previous,'COMPLETED:',len(p['records'][previous]),'OF',n,flush=True)
            return False
        return True


    def execute_stage(journal,stage,*,root,work,input_root,jobs=None,accept=False,check_network=True,monitor=True,max_new=0,max_seconds=0,worker_cls=SegmentWorker):
        ensure(type(max_new) is int and max_new>=0,'MAX_NEW_SEGMENTS must be nonnegative; 0 means unlimited')
        ensure(type(max_seconds) is int and max_seconds>=0,'MAX_STAGE_RUN_SECONDS must be nonnegative; 0 means unlimited')
        if not stage_ready(journal,stage):return 'BLOCKED'
        p=journal.p;n=len(p['plan']['units']);done=len(p['records'][stage])
        if done==n:
            print('V1_STAGE_ALREADY_COMPLETE:',stage,'SKIPPED_SEGMENTS:',n,flush=True);return 'COMPLETE'
        if stage in ('stage1','stage2'):ensure(accept is True,'Explicit model terms acceptance required before inference')
        if stage=='stage1':ensure(jobs is not None and len(jobs)==n,'Stage 1 recovery requires original source and settings')
        print('V1_STAGE_START:',stage,'SKIPPED_COMMITTED:',done,'REMAINING:',n-done,flush=True)
        start=time.monotonic();processed=0
        Path(work).mkdir(parents=True,exist_ok=True)
        with tempfile.TemporaryDirectory(prefix='oas-v1-'+stage+'-',dir=work) as tmp:
            with worker_cls(stage,root,tmp,input_root,accept,check_network,monitor) as worker:
                for i in range(done+1,n+1):
                    if processed and ((max_new and processed>=max_new) or (max_seconds and time.monotonic()-start>=max_seconds)):
                        print('V1_PAUSED:',stage,'NEXT_SEGMENT:',i,'REMOTE_OR_LOCAL_VERSION:',journal.p['version'],flush=True);return 'PAUSED'
                    journal.assert_current()
                    uid=p['plan']['units'][i-1]['unit_id'];prev=None
                    print('V1_SEGMENT_START:',stage,i,'/',n,flush=True)
                    if stage!='stage1':
                        previous=STAGES_V1[STAGES_V1.index(stage)-1];e=journal.p['records'][previous][i-1]
                        prev=journal.fetch(e,Path(tmp)/'incoming.bin')
                    try:
                        out=worker.run(uid,job=jobs[i-1] if stage=='stage1' else None,archive=prev)
                        # Verify full archive with the original checkpoint manager before the journal is advanced.
                        verifier=oas.Studio(work=worker.work,input_root=input_root)
                        ensure(verifier.store.completed(uid)==list(STAGES_V1[:STAGES_V1.index(stage)+1]),'Wrong segment checkpoint scope')
                        verifier.verify_archive_prefix(out,uid,stage)
                        journal.commit(stage,out)
                        worker.release(uid,out)
                        processed+=1
                        print('V1_SEGMENT_COMMITTED:',stage,i,'/',n,flush=True)
                    finally:
                        if prev is not None:Path(prev).unlink(missing_ok=True)
        print('V1_STAGE_COMPLETE:',stage,'SEGMENTS:',n,flush=True);return 'COMPLETE'


    def run_ffmpeg(args):
        # Long-task assembly is not constrained by an aggregate wall-clock timeout.
        subprocess.run(['ffmpeg','-nostdin','-hide_banner','-loglevel','error','-xerror','-y',*map(str,args)],check=True)


    def media_video_info(path):
        info=s1.read_media(path);videos=[s for s in info['streams'] if s['codec_type']=='video'];audios=[s for s in info['streams'] if s['codec_type']=='audio']
        ensure(len(videos)==1,'Expected one video stream');v=videos[0]
        count=v.get('nb_frames')
        if not str(count).isdigit():
            count=subprocess.check_output(['ffprobe','-v','error','-count_frames','-select_streams','v:0','-show_entries','stream=nb_read_frames','-of','default=nw=1:nk=1',str(path)],text=True).strip()
        return dict(frames=int(count),fps=Fraction(v['avg_frame_rate']),width=int(v['width']),height=int(v['height']),video_duration=s1.stream_duration(v,info),audio_duration=s1.stream_duration(audios[0],info) if len(audios)==1 else None,audio_tracks=len(audios),codec=v.get('codec_name'),pix_fmt=v.get('pix_fmt'),sar=v.get('sample_aspect_ratio'))


    def prepare_assembly_clip(video,narration,dest,raw_audio,geometry=None,expected_samples=None,copy_video=False):
        """Per-segment video CFR normalization + exact PCM timeline.
        Each segment ends on a frame boundary. Only <40ms silence padding may be added.
        Never concatenate independently encoded AAC packets (encoder delays accumulate).
        """
        with wave.open(str(narration),'rb') as w:
            ensure((w.getnchannels(),w.getsampwidth(),w.getframerate(),w.getcomptype())==(1,2,RATE,'NONE'),'Invalid normalized narration')
            samples=w.getnframes();ensure(samples>=RATE,'Invalid narration samples')
            if expected_samples is not None:ensure(samples==expected_samples,'Audio source sample count changed')
            frames=(samples+FRAME_SAMPLES-1)//FRAME_SAMPLES;pad=frames*FRAME_SAMPLES-samples
            v=media_video_info(video);g=(v['width'],v['height'])
            ensure(v['fps']==FPS and v['audio_tracks']==1 and all(x%2==0 for x in g),'Unexpected segment geometry/fps/audio')
            ensure(geometry is None or g==geometry,'Segment geometry changed; cannot assemble')
            ensure(abs(v['frames']-frames)<=5 and abs(v['video_duration']-samples/RATE)<=.5 and abs(v['audio_duration']-samples/RATE)<=.5,'Segment is truncated or out of sync')
            if copy_video and v['frames']==frames and v['codec']=='h264' and v['pix_fmt']=='yuv420p' and v['sar'] in (None,'1:1'):
                # One unit: preserve the already aligned H.264 picture, remux audio once.
                run_ffmpeg(['-i',video,'-map','0:v:0','-an','-c:v','copy','-video_track_timescale','25000',dest])
                print('V1_SINGLE_CLIP_VIDEO_COPY: no additional image re-encode',flush=True)
            else:
                run_ffmpeg(['-threads','1','-i',video,'-map','0:v:0','-an','-vf',f'fps={FPS},tpad=stop_mode=clone:stop_duration=0.24,trim=end_frame={frames},setpts=N/({FPS}*TB),setsar=1','-r',str(FPS),'-vsync','cfr','-c:v','libx264','-threads','2','-preset','veryfast','-crf','17','-pix_fmt','yuv420p','-video_track_timescale','25000',dest])
            aligned=media_video_info(dest)
            ensure(aligned['frames']==frames and aligned['fps']==FPS and (aligned['width'],aligned['height'])==g and aligned['audio_tracks']==0,'Aligned clip verification failed')
            copied=0
            while True:
                b=w.readframes(65536)
                if not b:break
                ensure(len(b)%2==0,'Invalid PCM');raw_audio.write(b);copied+=len(b)//2
            ensure(copied==samples,'Narration PCM truncated')
            raw_audio.write(b'\0'*(2*pad))
        return frames,samples,pad,g


    def publish_final(journal,*,work,output_root,input_root='/kaggle/input'):
        p=journal.p;n=len(p['plan']['units']);stage=route(p['plan'])[-1]
        if len(p['records'][stage])!=n:
            print('V1_FINAL_BLOCKED: complete all segments of',stage,'first',flush=True);return None
        output_root=Path(output_root);output_root.mkdir(parents=True,exist_ok=True)
        target=output_root/f'oas_{p["task_id"]}_final.mp4'
        if p['final'] is not None:
            if target.is_file() and not target.is_symlink() and artifact_record(target)==p['final']['record']:
                print('V1_FINAL_REUSED:',target,flush=True);return target
            with tempfile.TemporaryDirectory(prefix='oas-v1-final-fetch-',dir=output_root) as tmp:
                f=journal.fetch(p['final'],Path(tmp)/'final.mp4');os.replace(f,target)
            print('V1_FINAL_RESTORED:',target,flush=True);return target
        Path(work).mkdir(parents=True,exist_ok=True)
        with tempfile.TemporaryDirectory(prefix='oas-v1-assembly-',dir=work) as tmp:
            d=Path(tmp);geometry=None;frames=samples=padding=0;clips=[];frame_counts=[]
            with (d/'narration.s16le').open('wb') as raw:
                for i,e in enumerate(p['records'][stage],1):
                    journal.assert_current()
                    archive=journal.fetch(e,d/'incoming.bin');uid=e['unit_id']
                    studio=oas.Studio(work=d/'restore',input_root=input_root)
                    studio.restore(archive,uid)
                    ensure(studio.store.completed(uid)==list(STAGES_V1[:STAGES_V1.index(stage)+1]),'Wrong assembly checkpoint scope')
                    unit=studio.store.path(uid)
                    video=unit/stage/('final_video.mp4' if stage=='stage3' else 'ditto_video.mp4')
                    dest=d/f'clip_{i:08d}.mp4'
                    nf,ns,npad,geometry=prepare_assembly_clip(video,unit/'stage1'/'narration.wav',dest,raw,geometry,p['plan']['units'][i-1]['samples'],copy_video=(n==1))
                    frames+=nf;samples+=ns;padding+=npad;clips.append(dest.name);frame_counts.append(nf)
                    shutil.rmtree(unit);archive.unlink()
                    print('V1_ASSEMBLY_CLIP:',i,'/',n,'FRAMES:',nf,'PAD_SAMPLES:',npad,flush=True)
            ensure(frames*FRAME_SAMPLES==samples+padding and (d/'narration.s16le').stat().st_size==2*(samples+padding),'Final PCM timeline mismatch')
            seams=cont.smooth_joins([d/x for x in clips],frame_counts,d/'narration.s16le',d/'seam_patches',enabled=p['plan']['base']['config']['smooth_seams'])
            (d/'clips.ffconcat').write_text('ffconcat version 1.0\n'+''.join('file '+x+'\n' for x in clips),encoding='utf-8')
            # MP4 output is assembled using disk streams rather than storing all frames/samples in RAM.
            staged=d/'joined.mp4'
            run_ffmpeg(['-f','concat','-safe','1','-i',d/'clips.ffconcat','-f','s16le','-ar',str(RATE),'-ac','1','-i',d/'narration.s16le','-map','0:v:0','-map','1:a:0','-c:v','copy','-c:a','aac','-b:a','160k','-movflags','+faststart',staged])
            v=media_video_info(staged);duration=frames/FPS
            ensure(v['frames']==frames and v['fps']==FPS and (v['width'],v['height'])==geometry and v['audio_tracks']==1,'Final video frame/geometry verification failed')
            ensure(abs(v['video_duration']-duration)<=.05 and abs(v['audio_duration']-duration)<=.1,'Final audio/video timeline verification failed')
            # Full decode validates media integrity. No fixed aggregate duration limit.
            run_ffmpeg(['-threads','1','-i',staged,'-map','0:v:0','-map','0:a:0','-f','null','-'])
            media=dict(segments=n,seams=seams,speech_speed=p['plan']['base']['config']['speech_speed'],frames=frames,fps=FPS,width=geometry[0],height=geometry[1],original_pcm_samples=samples,padding_pcm_samples=padding,duration_seconds=duration,video_duration=v['video_duration'],audio_duration=v['audio_duration'])
            os.replace(staged,target)
            journal.commit('final',target,media)
            write_json(output_root/f'oas_{p["task_id"]}_final_report.json',media)
        print('V1_COMPLETE:',target,'DURATION:',duration,'SHA256:',artifact_record(target)['sha256'],flush=True)
        return target


    def assert_prefix(old,new):
        ensure(old['task_id']==new['task_id'] and new['version']>=old['version'],'Remote/local journal is older than the selected checkpoint')
        for stage in STAGES_V1:
            ensure(new['records'][stage][:len(old['records'][stage])]==old['records'][stage],'Journal history differs from selected checkpoint')
        ensure(old['final'] is None or new['final']==old['final'],'Final record history differs')


    def open_journal(work,ref='',plan=None,expected_id='',restore_manifest='',wait_seconds=600,command_seconds=3600):
        ensure(not expected_id or ishex(expected_id),'EXPECTED_TASK_ID must be blank or 64 hex characters')
        ensure(type(wait_seconds) is int and wait_seconds>0 and type(command_seconds) is int and command_seconds>0,'Remote timeouts must be positive integers')
        work=Path(work).resolve();work.mkdir(parents=True,exist_ok=True)
        old=None
        if restore_manifest:
            path=Path(restore_manifest).resolve(strict=True);old=read_json(path)
            validate_progress(old,old.get('dataset'),expected_id,plan)
            expected_id=old['task_id']
            if old['dataset']:
                ensure(not ref or ref==old['dataset'],'Restore manifest and PERSIST_DATASET disagree');ref=old['dataset']
            else:
                ensure(not ref,'Local journal cannot be promoted to remote automatically; use original local journal')
                ensure(path.name==PROGRESS_NAME,'Use exact oas-progress.json filename')
                dest=work/'journals'/expected_id
                if path.parent!=dest:
                    dest.mkdir(parents=True,exist_ok=True);marker=dest/PROGRESS_NAME
                    if marker.is_file():
                        present=validate_progress(read_json(marker),None,expected_id,plan);assert_prefix(old,present)
                    else:
                        for entry in [e for st in STAGES_V1 for e in old['records'][st]]+([old['final']] if old['final'] else []):
                            rel=Path('versions')/str(entry['version'])/entry['name'];src=path.parent/rel;dst=dest/rel
                            ensure(artifact_record(src)==entry['record'],'Local restore artifact missing or corrupt: '+str(rel))
                            dst.parent.mkdir(parents=True,exist_ok=True);shutil.copyfile(src,dst)
                            ensure(artifact_record(dst)==entry['record'],'Local restore copy changed')
                        write_json(marker,old)
                    return LocalJournal(dest,plan,expected_id)
                return LocalJournal(path.parent,plan,expected_id)
        if ref:
            validate_destination(ref)
            # Local snapshot is not authoritative; a newly started Session always reads latest remote index.
            folder=work/'remote-snapshots'/ref.replace('/','__')
            j=RemoteJournal.open_secret(ref,folder,plan,expected_id,wait_seconds,command_seconds)
            try:
                if old is not None:assert_prefix(old,j.p)
            except BaseException:
                j.close();raise
            return j
        if plan is None:
            ensure(expected_id,'Cold local recovery needs exact RESTORE_MANIFEST or EXPECTED_TASK_ID')
            folder=work/'journals'/expected_id
        else:
            tid=sha_json(plan);ensure(not expected_id or expected_id==tid,'EXPECTED_TASK_ID mismatch');folder=work/'journals'/tid
        print('V1_LOCAL_ONLY: /kaggle/working is NOT durable across full container loss. Configure PERSIST_DATASET for long jobs.',flush=True)
        return LocalJournal(folder,plan,expected_id)


    def choose_journal_for_cell(stage,work,ref,restore_manifest,expected_id,wait_seconds,command_seconds,active_ref='',active_manifest='',active_id=''):
        # Explicit selection is authoritative; never silently mix two jobs.
        if active_id:
            ensure(not expected_id or expected_id==active_id,'Active task and expected task differ')
            if ref and active_ref:ensure(ref==active_ref,'Active task and persistence destinations differ')
            if ref and active_manifest and not active_ref:raise RuntimeError('Cannot silently switch active local task to a new remote Dataset')
        return open_journal(work,ref or active_ref,expected_id=expected_id or active_id,restore_manifest=restore_manifest or (active_manifest if not (ref or active_ref) else ''),wait_seconds=wait_seconds,command_seconds=command_seconds)



    _ctx=globals().get('_OAS_V1_CONTEXT') or {}
    _work=Path(STAGE2_WORK_ROOT or _ctx.get('work') or globals().get('WORK_ROOT','/kaggle/working/open-avatar-studio-v11')).resolve()
    _accept=globals().get('ALLOW_NONCOMMERCIAL_RESEARCH',False) if STAGE2_ALLOW_NONCOMMERCIAL_RESEARCH is None else STAGE2_ALLOW_NONCOMMERCIAL_RESEARCH
    ensure(type(_accept) is bool,'STAGE2_ALLOW_NONCOMMERCIAL_RESEARCH must be None, True or False')
    _ref=STAGE2_PERSIST_DATASET or _ctx.get('dataset') or globals().get('PERSIST_DATASET','')
    _j=choose_journal_for_cell('stage2',_work,_ref,STAGE2_RESTORE_MANIFEST,STAGE2_EXPECTED_TASK_ID,STAGE2_REMOTE_VERIFY_WAIT_SECONDS,STAGE2_REMOTE_COMMAND_TIMEOUT_SECONDS,_ctx.get('dataset',''),_ctx.get('manifest',''),_ctx.get('task_id',''))
    try:
        TASK_ID=_j.p['task_id'];_OAS_V1_CONTEXT=dict(task_id=TASK_ID,dataset=_j.p['dataset'] or '',manifest=str(_j.path),work=str(_work))
        print('V1_PROGRESS_PATH:',_j.path,flush=True)
        if _j.p['plan']['use_lp']:
            _state=execute_stage(_j,'stage2',root=_OAS_ROOT,work=_work,input_root='/kaggle/input',accept=_accept,check_network=STAGE2_CHECK_NETWORK,monitor=STAGE2_MONITOR_RESOURCES,max_new=STAGE2_MAX_NEW_SEGMENTS,max_seconds=STAGE2_MAX_RUN_SECONDS)
        else:print('V1_STAGE2_SKIPPED: direct Ditto route',flush=True)
        run_status=copy.deepcopy(_j.p)
    finally:_j.close()

else:
    print('V1_STAGE2_SKIPPED: RUN_STAGE_2=False',flush=True)


RUNTIME_BUNDLE_VERIFIED: f0ddb13b6d236d1724f5bd9b8137a561405a3a22b9abb34758c0f5a771f3c6cc
RUNTIME_FILES_VERIFIED: 23
V1_REMOTE_CONNECTED: elleea/oas-checkpoints-avatar-checkpoint-continous-07 VERSION: 2
V1_PROGRESS: c930780bad42318800a803f8037d1618494c8636455fc4f572d6d11ea547507e VERSION: 2 TOTAL_SEGMENTS: 1 DONE: {'stage1': 1, 'stage2': 0, 'stage3': 0} NEXT: stage2:1
V1_PROGRESS_PATH: /kaggle/working/open-avatar-studio-v11/remote-snapshots/elleea__oas-checkpoints-avatar-checkpoint-continous-07/oas-progress.json
V1_STAGE_START: stage2 SKIPPED_COMMITTED: 0 REMAINING: 1
V1_SEGMENT_START: stage2 1 / 1
[RESOURCE] 05:00:39 stage2 RAM_GiB: 1.22 WORKING_GiB: 0.67 LIMIT_GiB: 30.0 GPU_MiB: [[0, 0, 15360], [1, 0, 15360]] EVENTS: low 0; high 0; max 0; oom 0; oom_kill 0; oom_group_kill 0
CHECKPOINT_RESTORED: c81b3b2ec90f8d7e89c93496e200d26bd01262062cbda07a00da22993e6a3fab
TASK_ID: c81b3b2ec90f8d7e89c93496e200d26bd01262062cbda07a00da22993e6a3fab
COMPLETED_STAGES: ['stage1']
NEXT_STAGE: stage2
NETWO

From https://github.com/KwaiVGI/LivePortrait
 * branch            9b294b3d0536135442ea73cb01e6cb3ca7029dd3 -> FETCH_HEAD


END: LivePortrait_git_fetch exit= 0 seconds= 1.52
START: LivePortrait_git_checkout


HEAD is now at 9b294b3 docs: add AVTR-1 to community resources


END: LivePortrait_git_checkout exit= 0 seconds= 0.21
SOURCE_READY: LivePortrait 9b294b3d0536135442ea73cb01e6cb3ca7029dd3
START: lp_ort_probe
Traceback (most recent call last):
  File "<string>", line 2, in <module>
    import onnxruntime as ort
ModuleNotFoundError: No module named 'onnxruntime'
END: lp_ort_probe exit= 1 seconds= 0.13
START: lp_install_onnx_cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 21.0 MB/s  0:00:00
[RESOURCE] 05:01:03 stage2 RAM_GiB: 2.63 WORKING_GiB: 1.1 LIMIT_GiB: 30.0 GPU_MiB: [[0, 0, 15360], [1, 0, 15360]] EVENTS: low 0; high 0; max 0; oom 0; oom_kill 0; oom_group_kill 0
END: lp_install_onnx_cpu exit= 0 seconds= 3.32
START: lp_ort_final
LP_ONNX_CPU_OK: 1.23.2 ['AzureExecutionProvider', 'CPUExecutionProvider']
END: lp_ort_final exit= 0 seconds= 0.21
START: lp_cpu_probe
[RESOURCE] 05:01:06 stage2 RAM_GiB: 2.91 WORKING_GiB: 1.35 LIMIT_GiB: 30.0 GPU_MiB: [[0, 0, 15360], [1, 0, 15360]] EVENTS: low 0; high 0; max 0; oom 0; oom_kill 0; oom_group_kill 0

In [2]:
# Open Avatar Studio V1.1 | Continuity First + Speech Speed | Long Video | 3/3 - Final Composition + Assembly
# CPU-only: standalone Accelerator None; Internet On for private Dataset recovery/persistence.
# No model or original source Input needed here. Full GPU batch still owns its allocation.
# Full run: enable RUN_STAGE_3, keep restore and destination blank.
# Cold recovery: Cells 1 & 2 False, set SAME STAGE3_PERSIST_DATASET (or trusted progress path).
# Stage 3 commits each composited segment, then assembles all segments on disk.
# Direct Ditto route: can also recover final assembly from Stage 1 without LivePortrait.
# Final assembly may need considerable disk/time. If interrupted, all segment commits remain.
# ===== USER CONFIGURATION / 用户配置 =====
RUN_STAGE_3 = True
STAGE3_PERSIST_DATASET = ''  # blank=inherit task context; cold run: same private Dataset ref
STAGE3_RESTORE_MANIFEST = ''  # optional exact trusted oas-progress.json path (NOT a legacy ZIP)
STAGE3_EXPECTED_TASK_ID = ''
STAGE3_MAX_NEW_SEGMENTS = 0  # optional max newly composited segments per run; 0=all
STAGE3_MAX_RUN_SECONDS = 0  # soft budget between segments only, 0=unlimited; does not cap assembly
STAGE3_ASSEMBLE_FINAL = True  # False=prepare/commit segments only, later run Cell 3 with True
STAGE3_REMOTE_VERIFY_WAIT_SECONDS = 600
STAGE3_REMOTE_COMMAND_TIMEOUT_SECONDS = 3600
STAGE3_MONITOR_RESOURCES = True
STAGE3_WORK_ROOT = ''  # blank=inherit Cell 1/default
# ===== END USER CONFIGURATION / DO NOT EDIT BELOW =====

if type(RUN_STAGE_3) is not bool:
    raise ValueError("RUN_STAGE_3 must be True or False")
if RUN_STAGE_3:
    # Generated, content-verified V1.1 runtime. Do not edit the embedded payload.
    import base64 as _b64, hashlib as _hash, io as _io, json as _json
    import os as _os, shutil as _shutil, stat as _stat, sys as _sys
    import tempfile as _tempfile, zipfile as _zipfile
    from pathlib import Path as _Path

    _OAS_SHA = 'f0ddb13b6d236d1724f5bd9b8137a561405a3a22b9abb34758c0f5a771f3c6cc'
    _OAS_BUNDLE = '...'

    def _oas_bootstrap():
        data = _b64.b64decode(_OAS_BUNDLE, validate=True)
        if _hash.sha256(data).hexdigest() != _OAS_SHA:
            raise RuntimeError('V1.1 bundle SHA256 mismatch')
        with _zipfile.ZipFile(_io.BytesIO(data)) as z:
            manifest = _json.loads(z.read('runtime_manifest.json'))
            if manifest['schema'] != 'oas-runtime-bundle-v01' or manifest['release'] != '1.1.0':
                raise RuntimeError('Wrong runtime release')
            names = set(manifest['files']) | {'runtime_manifest.json'}
            if len(z.infolist()) != len(names) or set(z.namelist()) != names:
                raise RuntimeError('Unexpected runtime archive members')
            for m in z.infolist():
                if '/' in m.filename or '\\' in m.filename or m.filename in ('.','..') or _stat.S_IFMT(m.external_attr >> 16) not in (0,_stat.S_IFREG):
                    raise RuntimeError('Unsafe runtime archive member')
            expected = dict(manifest['files'])
            mbytes = z.read('runtime_manifest.json')
            expected['runtime_manifest.json'] = {'size':len(mbytes),'sha256':_hash.sha256(mbytes).hexdigest()}
            parent = _Path('/tmp/open-avatar-studio');parent.mkdir(parents=True,exist_ok=True)
            folder = parent/('runtime-'+_OAS_SHA[:16])
            def verify(path):
                if path.is_symlink() or not path.is_dir() or set(x.name for x in path.iterdir()) != names:
                    raise RuntimeError('Unexpected runtime directory contents: '+str(path))
                for n,rec in expected.items():
                    f=path/n
                    if f.is_symlink() or not f.is_file() or f.stat().st_size != rec['size'] or _hash.sha256(f.read_bytes()).hexdigest() != rec['sha256']:
                        raise RuntimeError('Runtime file mismatch: '+n)
            if not folder.exists():
                with _tempfile.TemporaryDirectory(dir=parent,prefix='.runtime-') as tmp:
                    stage=_Path(tmp)/'payload';stage.mkdir()
                    for n in names:
                        (stage/n).write_bytes(z.read(n))
                    verify(stage)
                    _os.rename(stage,folder)
            verify(folder)
        print('RUNTIME_BUNDLE_VERIFIED:',_OAS_SHA,flush=True)
        print('RUNTIME_FILES_VERIFIED:',len(expected),flush=True)
        return folder

    _OAS_ROOT = _oas_bootstrap()

    _sys.dont_write_bytecode = True
    for _name in ['resource_manager','auxiliary_manager','environment_manager','checkpoint_manager','demo_assets_builder','stage1_engine','stage23_engine','runtime_support','pipeline','continuity']:
        if _name in _sys.modules:
            _loaded = getattr(_sys.modules[_name],'__file__',None)
            if not _loaded or _Path(_loaded).resolve() != (_OAS_ROOT/(_name+'.py')).resolve():
                raise RuntimeError('Old module already loaded: '+_name+'. Start a NEW notebook Session.')
    if str(_OAS_ROOT) not in _sys.path:
        _sys.path.insert(0,str(_OAS_ROOT))
    import pipeline as oas

    """Optional private, versioned Kaggle persistence. No model or runtime changes."""
    import hashlib
    import json
    import os
    import re
    import shutil
    import subprocess
    import tempfile
    import time
    import uuid
    import zipfile
    from pathlib import Path


    class PersistenceError(RuntimeError):
        pass


    def digest(path):
        h = hashlib.sha256()
        size = 0
        with Path(path).open('rb') as f:
            for block in iter(lambda: f.read(1024*1024), b''):
                size += len(block)
                h.update(block)
        return {'bytes': size, 'sha256': h.hexdigest()}


    def validate_destination(ref):
        if not isinstance(ref, str) or not re.fullmatch(r'[a-z0-9][a-z0-9_-]{0,49}/[a-z0-9][a-z0-9-]{5,49}', ref):
            raise PersistenceError('PERSIST_DATASET must be a personal owner/slug (lowercase, slug 6-50 characters).')
        # Dedicated name only; no accidental overwrite of model/demo resources.
        if not ref.split('/')[1].startswith('oas-checkpoints-'):
            raise PersistenceError('PERSIST_DATASET slug must start with oas-checkpoints-. Use one dedicated task-specific name.')
        return ref


    class PrivateDatasetStore:
        manifest_name = "oas-progress.json"

        def __init__(self, ref, task_id, token, *, cli=None, wait_seconds=300, command_seconds=900,
                     run_command=subprocess.run, sleep=time.sleep, clock=time.monotonic):
            self.ref = validate_destination(ref)
            if not re.fullmatch('[0-9a-f]{64}', task_id):
                raise PersistenceError('Invalid task identity')
            if not isinstance(token, str) or not token.strip():
                raise PersistenceError('KAGGLE_API_TOKEN secret is empty or missing')
            self.task_id = task_id
            self.nonce = uuid.uuid4().hex
            self.cli = cli or shutil.which('kaggle')
            if not self.cli:
                raise PersistenceError('Kaggle CLI not installed. Use a Kaggle image with a token-capable CLI.')
            self.wait_seconds = wait_seconds
            self.command_seconds = command_seconds
            self._run, self._sleep, self._clock = run_command, sleep, clock
            self._tmp = tempfile.TemporaryDirectory(prefix='oas-kaggle-auth-')
            self._env = dict(os.environ)
            for key in ('KAGGLE_USERNAME','KAGGLE_KEY','KAGGLE_API_TOKEN','KAGGLE_CONFIG_DIR'):
                self._env.pop(key, None)
            self._env.update(KAGGLE_API_TOKEN=token, KAGGLE_CONFIG_DIR=self._tmp.name)
            self.version = 0
            self.last_receipt = None
            self._manifest = None
            self._records = {}
            self._preserved = {}  # Exact remote bytes of earlier checkpoints on cross-session recovery.

        def close(self):
            self._env.pop('KAGGLE_API_TOKEN', None)
            self._tmp.cleanup()

        def command(self, args, *, timeout=None):
            # Never include stdout/stderr in errors: CLI may include signed URLs.
            try:
                r = self._run([self.cli, 'datasets', *args], env=self._env,
                              stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
                              stderr=subprocess.PIPE, timeout=timeout or self.command_seconds,
                              check=False)
            except Exception:
                raise PersistenceError('KAGGLE_API_COMMAND_FAILED: '+args[0]) from None
            if r.returncode != 0:
                raise PersistenceError('KAGGLE_API_COMMAND_FAILED: '+args[0]+' exit='+str(r.returncode))
            return r

        @staticmethod
        def _privacy_flag(data):
            # Only explicit booleans (or exact text booleans from older clients).
            # Never accept the truthiness of a nonempty string or infer private from access.
            flags = []
            for k in ('isPrivate', 'is_private'):
                if k in data:
                    v = data[k]
                    if isinstance(v, bool):
                        flags.append(v)
                    elif isinstance(v, str) and v.strip().lower() in ('true', 'false'):
                        flags.append(v.strip().lower() == 'true')
                    elif v is not None:
                        raise PersistenceError('REMOTE_PRIVACY_INVALID: unrecognized '+k+' value type')
            if len(set(flags)) > 1:
                raise PersistenceError('REMOTE_PRIVACY_CONFLICT: inconsistent privacy fields')
            return flags[0] if flags else None

        def _metadata_privacy(self, meta):
            if not isinstance(meta, dict):
                raise PersistenceError('REMOTE_METADATA_INVALID: expected a JSON object')
            # Kaggle CLI can serialize the entire API response: {"info": DatasetInfo}.
            # Flat DatasetInfo and legacy upload-style metadata are also supported.
            nodes = [meta]
            if isinstance(meta.get('info'), dict):
                nodes.append(meta['info'])
            elif 'info' in meta and meta['info'] is not None:
                raise PersistenceError('REMOTE_METADATA_INVALID: unexpected info structure')
            owner, slug = self.ref.split('/')
            flags = []
            for node in nodes:
                if node.get('errorMessage') or node.get('error_message'):
                    raise PersistenceError('REMOTE_METADATA_ERROR: server returned an error')
                for key in ('ref', 'id'):
                    v = node.get(key)
                    # id may be numeric in API responses; only a path-like id is a ref.
                    if isinstance(v, str) and '/' in v and v.lower() != self.ref:
                        raise PersistenceError('REMOTE_METADATA_IDENTITY_MISMATCH: '+key)
                for key in ('datasetSlug', 'dataset_slug'):
                    v = node.get(key)
                    if isinstance(v, str) and v and v.lower() not in (slug, self.ref):
                        raise PersistenceError('REMOTE_METADATA_IDENTITY_MISMATCH: '+key)
                for key in ('ownerUser', 'owner_user', 'ownerSlug', 'owner_slug'):
                    v = node.get(key)
                    if isinstance(v, str) and v and v.lower() != owner:
                        raise PersistenceError('REMOTE_METADATA_IDENTITY_MISMATCH: '+key)
                v = self._privacy_flag(node)
                if v is not None:
                    flags.append(v)
            if len(set(flags)) > 1:
                raise PersistenceError('REMOTE_PRIVACY_CONFLICT: metadata and info disagree')
            return flags[0] if flags else None

        def _query_dataset_privacy(self):
            # Read-only fallback to the official SDK GetDataset endpoint. Run in the
            # same isolated token environment as CLI calls. Do not expose API output,
            # exception text, access tokens, signed URLs or user metadata in logs.
            import sys
            script = "import json, os, sys\nfrom kagglesdk import KaggleClient\nfrom kagglesdk.datasets.types.dataset_api_service import ApiGetDatasetRequest\nowner, slug = sys.argv[1].split('/')\nreq = ApiGetDatasetRequest()\nreq.owner_slug = owner\nreq.dataset_slug = slug\nwith KaggleClient(api_token=os.environ['KAGGLE_API_TOKEN']) as client:\n    item = client.datasets.dataset_api_client.get_dataset(req)\n    result = {'ref': item.ref, 'isPrivate': item.is_private}\nprint('OAS_PRIVATE_RESULT:' + json.dumps(result))\n"
            try:
                r = self._run([sys.executable, '-c', script, self.ref], env=self._env,
                              stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
                              stderr=subprocess.PIPE, timeout=60, check=False)
                if r.returncode != 0:
                    raise PersistenceError('REMOTE_PRIVACY_SDK_FAILED: exit='+str(r.returncode))
                raw = r.stdout.decode('utf-8', errors='replace') if isinstance(r.stdout, bytes) else str(r.stdout)
                matches = [x[len('OAS_PRIVATE_RESULT:'):] for x in raw.splitlines() if x.startswith('OAS_PRIVATE_RESULT:')]
                if len(matches) != 1:
                    raise PersistenceError('REMOTE_PRIVACY_SDK_FAILED: missing result')
                result = json.loads(matches[0])
            except PersistenceError:
                raise
            except Exception as exc:
                raise PersistenceError('REMOTE_PRIVACY_SDK_FAILED: '+type(exc).__name__) from None
            if not isinstance(result, dict) or result.get('ref') != self.ref:
                raise PersistenceError('REMOTE_METADATA_IDENTITY_MISMATCH: SDK ref')
            return self._privacy_flag(result)

        def _private(self):
            # Fail closed. Never upload to a public Dataset, or merely assume that
            # an accessible Dataset must be private. Handle nested API responses.
            with tempfile.TemporaryDirectory(prefix='oas-remote-meta-') as d:
                self.command(['metadata', self.ref, '-p', d], timeout=60)
                p = Path(d)/'dataset-metadata.json'
                try:
                    meta = json.loads(p.read_text(encoding='utf-8-sig'))
                except Exception:
                    raise PersistenceError('REMOTE_METADATA_INVALID: metadata JSON unavailable') from None
                privacy = self._metadata_privacy(meta)
                if privacy is None:
                    print('REMOTE_PRIVACY_METADATA_FIELD_MISSING: trying read-only SDK GetDataset', flush=True)
                    privacy = self._query_dataset_privacy()
                if privacy is False:
                    raise PersistenceError('Remote dataset privacy check: REMOTE_DATASET_PUBLIC; set Visibility to Private before retrying')
                if privacy is not True:
                    raise PersistenceError('Remote dataset privacy not explicitly confirmed; refusing to upload')

        def _assert_latest(self, expected):
            _, record=self._read_latest_manifest()
            if record!=expected:
                raise PersistenceError('REMOTE_VERSION_CHANGED: another version appeared; stop and inspect the latest Dataset before retrying. Do not run concurrent writers.')

        def _download_record(self, version, name, expected, destination=None):
            """Verify an exact version; optionally retain a verified copy, never unzip into a directory."""
            with tempfile.TemporaryDirectory(prefix='oas-remote-check-') as d:
                self.command(['download','-d',f'{self.ref}/{version}','-f',name,'-p',d,'-o','-q'])
                p=Path(d)/name
                if p.is_file() and not p.is_symlink():
                    actual=digest(p)
                    if actual!=expected:
                        raise PersistenceError('Remote byte verification failed: '+name)
                    if destination is not None: shutil.copyfile(p,destination)
                else:
                    wrapped=Path(d)/(name+'.zip')
                    if not wrapped.is_file():
                        raise PersistenceError('Remote file unavailable: '+name)
                    try:
                        with zipfile.ZipFile(wrapped) as z:
                            members=z.infolist()
                            if len(members)!=1 or members[0].filename!=name or members[0].file_size!=expected['bytes']:
                                raise PersistenceError('Unexpected remote archive wrapper: '+name)
                            h=hashlib.sha256();size=0
                            # A private scratch file, not a path derived from ZIP members.
                            staged=Path(d)/'verified_payload'
                            with z.open(members[0]) as f, staged.open('wb') as out:
                                for block in iter(lambda:f.read(1024*1024),b''):
                                    size+=len(block);h.update(block);out.write(block)
                            actual={'bytes':size,'sha256':h.hexdigest()}
                            if actual!=expected:
                                raise PersistenceError('Remote byte verification failed: '+name)
                            if destination is not None: shutil.copyfile(staged,destination)
                    except (OSError,zipfile.BadZipFile):
                        raise PersistenceError('Unreadable remote file: '+name) from None
                if destination is not None and digest(destination)!=expected:
                    raise PersistenceError('Verified copy changed: '+name)

        def _verify(self, version, manifest_record, files):
            deadline = self._clock()+self.wait_seconds
            while True:
                try:
                    self._download_record(version,self.manifest_name,manifest_record)
                    self._private()
                    for name,record in files.items():
                        self._download_record(version,name,record)
                    # A pinned version can be ready before the latest alias advances.
                    latest, observed = self._read_latest_manifest()
                    if observed != manifest_record:
                        validate_progress(latest,self.ref,self.task_id)
                        if latest['version'] < version:
                            raise PersistenceError('REMOTE_LATEST_ALIAS_PENDING')
                        raise PersistenceError('REMOTE_VERSION_CHANGED: another version appeared; stop and inspect the latest Dataset. Do not run concurrent writers.')
                    return
                except PersistenceError as exc:
                    reason = str(exc)
                    # Retry publication delays, not privacy, hash, identity, malformed
                    # manifest or concurrent-writer failures. Never log raw CLI output.
                    pending = reason.startswith(('KAGGLE_API_COMMAND_FAILED: download',
                                                  'Remote file unavailable:',
                                                  'REMOTE_LATEST_ALIAS_PENDING'))
                    if not pending:
                        raise PersistenceError('REMOTE_VERSION_VERIFY_FAILED: '+self.ref+'/'+str(version)+'; '+reason) from None
                    if self._clock() >= deadline:
                        raise PersistenceError('REMOTE_VERSION_NOT_VERIFIED: '+self.ref+'/'+str(version)+'; last reason: '+reason) from None
                    print('REMOTE_VERSION_WAIT:',self.ref,'VERSION:',version,'REASON:',reason,flush=True)
                    self._sleep(min(10,max(0,deadline-self._clock())))

        def assert_current(self):
            if self.version<1 or self._manifest is None:
                raise PersistenceError('No verified remote version')
            self._private()
            self._download_record(self.version,self.manifest_name,self._manifest)
            self._assert_latest(self._manifest)


    # Optional demo extension; the frozen runtime and original asset manifest stay unchanged.
    import hashlib
    import shutil
    import tempfile
    import wave
    from pathlib import Path
    import pipeline as oas
    import resource_manager as rm
    from demo_assets_builder import verify as verify_base_demo

    DEMO_AUDIO_FILES = ('demo_audio.wav', 'native_blink_long_test.wav')
    _DEMO_BASE = {'portrait_01.png','portrait_02.png','reference.wav','reference.txt',
                  'demo_audio.wav','script_en.txt','script_zh.txt','demo_config.json',
                  'PORTRAIT_LICENSE.txt','ASSET_NOTICES.md','asset_manifest.json'}
    _DEMO_LONG_RECORDS = {'native_blink_long_test.txt': {'size': 461, 'sha256': '88a5c2c85a00d960770a6016d1bc5f473cf67c320af682f8ef6a1f044c00d9fb'}, 'native_blink_long_test.wav': {'size': 1523172, 'sha256': 'fc8dbce194c574aa24653d7350823492050f26d3a4b0eae498864253a3c1ecf4'}}


    def verify_demo_mount(input_root='/kaggle/input'):
        root = rm.locate(input_root, 'open-avatar-demo-assets')
        actual = {p.relative_to(root).as_posix() for p in root.rglob('*') if p.is_file()}
        base = _DEMO_BASE | ({'dataset-metadata.json'} if 'dataset-metadata.json' in actual else set())
        extras = actual - base
        missing = _DEMO_BASE - actual
        if missing or extras not in (set(), set(_DEMO_LONG_RECORDS)):
            raise RuntimeError('DEMO_ASSET_SCOPE_MISMATCH: missing='+str(sorted(missing))+
                               '; extra='+str(sorted(extras))+'; attach original or official expanded Demo Assets')
        if not extras:
            verify_base_demo(root, mounted=True)
            return root, False
        # The older verifier intentionally rejects extra files. Verify an isolated copy
        # of the unchanged core, then verify both optional files against release pins.
        # Nothing is written to the mounted Dataset or to the frozen runtime.
        with tempfile.TemporaryDirectory(prefix='oas-demo-base-') as tmp:
            projection = Path(tmp)
            for name in sorted(base):
                shutil.copyfile(root/name, projection/name)
            verify_base_demo(projection, mounted=True)
        for name, expected in _DEMO_LONG_RECORDS.items():
            path = root/name
            if path.stat().st_size != expected['size'] or hashlib.sha256(path.read_bytes()).hexdigest() != expected['sha256']:
                raise RuntimeError('DEMO_LONG_ASSET_INTEGRITY_MISMATCH: '+name)
            print('DEMO_LONG_ASSET_VERIFIED:',name,flush=True)
        with wave.open(str(root/'native_blink_long_test.wav'),'rb') as w:
            duration = w.getnframes()/w.getframerate()
            if (w.getframerate(),w.getnchannels(),w.getsampwidth(),w.getcomptype()) != (24000,1,2,'NONE') or not 30 < duration <= 60:
                raise RuntimeError('DEMO_LONG_AUDIO_INVALID')
        print('DEMO_LONG_AUDIO_SECONDS:',round(duration,3),flush=True)
        return root, True


    def prepare_demo_job(mode, portrait='portrait_01.png', language='English',
                         demo_audio='demo_audio.wav', input_root='/kaggle/input', **opts):
        if portrait not in ('portrait_01.png','portrait_02.png'):
            raise ValueError('DEMO_PORTRAIT must be portrait_01.png or portrait_02.png')
        if mode == 'audio' and demo_audio not in DEMO_AUDIO_FILES:
            raise ValueError('DEMO_AUDIO must be demo_audio.wav or native_blink_long_test.wav')
        if mode not in ('audio','text') or (mode == 'text' and language not in ('English','Chinese')):
            raise ValueError('Unsupported demo mode/language; demo_text uses English or Chinese')
        root, has_long = verify_demo_mount(input_root)
        if mode == 'audio':
            if demo_audio == 'native_blink_long_test.wav' and not has_long:
                raise RuntimeError('DEMO_LONG_AUDIO_NOT_MOUNTED: select the expanded Demo Assets version, or use demo_audio.wav')
            print('DEMO_AUDIO_SELECTED:',demo_audio,flush=True)
            return oas.prepare_job(mode='audio',portrait=root/portrait,audio=root/demo_audio,**opts)
        return oas.prepare_job(mode='text',portrait=root/portrait,
                               script=root/('script_zh.txt' if language=='Chinese' else 'script_en.txt'),
                               reference_audio=root/'reference.wav',reference_text=root/'reference.txt',
                               language=language,**opts)


    def resolve_workflow(blink_mode, open_frames, export_checkpoint):
        if blink_mode not in ('liveportrait', 'ditto_template', 'ditto_native'):
            raise ValueError("BLINK_MODE must be liveportrait, ditto_template or ditto_native")
        if type(open_frames) is not int or open_frames < 0:
            raise ValueError('DITTO_EYE_OPEN_FRAMES must be a nonnegative integer')
        if export_checkpoint != 'auto' and type(export_checkpoint) is not bool:
            raise ValueError("EXPORT_CHECKPOINT must be 'auto', True or False")
        use_lp = blink_mode == 'liveportrait'
        export = use_lp if export_checkpoint == 'auto' else export_checkpoint
        if use_lp and not export:
            raise ValueError('LivePortrait needs a Stage 1 checkpoint; set EXPORT_CHECKPOINT to auto or True')
        return use_lp, export

    def configure_eye_mode(job, blink_mode, open_frames):
        # Identical to the tested Ditto-only experiment for the two direct modes.
        if blink_mode == 'liveportrait':
            if not job['config']['blink']['enabled']:
                raise ValueError('Invalid LivePortrait task configuration')
            return job  # preserve the exact original identity and model parameters
        if job['config']['blink']['enabled']:
            raise ValueError('Direct Ditto must not enable LivePortrait')
        ditto = job['config']['ditto']
        ditto['drive_eye'] = True
        if blink_mode == 'ditto_native':
            ditto['delta_eye_arr'] = None
            ditto['delta_eye_open_n'] = -1
        else:
            ditto.pop('delta_eye_arr', None)  # retain the pinned model's default template, if available
            ditto['delta_eye_open_n'] = open_frames
        job['identity'] = cp.make_identity(job['inputs'], job['config'], job['identity']['revisions'])
        job['task_id'] = cp.task_id(job['identity'])
        oas.verify_bound(job['identity'])
        return job

    # V1.1 continuity build: task identity changes with runtime and planning.
    SCHEDULER_SHA = '03b674ff888a2c7d8b36a1310b22b0ea9f3bb3a4c3d32583e724bb0c5d6dcf91'

    SEGMENT_WORKER_SOURCE = '"""A persistent stage-specific worker, with one verified runtime environment per stage.\nThe coordinator owns Kaggle credentials; this subprocess receives none.\n"""\nimport json, sys, traceback, shutil\nfrom pathlib import Path\nroot,specfile,stage=sys.argv[1:]\nsys.dont_write_bytecode=True;sys.path.insert(0,root)\nimport pipeline as oas\nimport checkpoint_manager as cp\nspec=json.loads(Path(specfile).read_text())\nwork=Path(spec[\'work\']);result=Path(spec[\'result\'])\nstudio=oas.Studio(work=work,input_root=spec[\'input_root\'],cache=spec[\'cache\'],accept_noncommercial_research=spec[\'accept\'],check_network=spec[\'check_network\'])\nfor line in sys.stdin:\n    cmd=json.loads(line)\n    if cmd.get(\'stop\'):break\n    tid=cmd[\'unit_id\']\n    try:\n        with oas.support.monitor(stage,None,gpu=stage!=\'stage3\',enabled=spec[\'monitor\']):\n            if stage==\'stage1\':\n                job=cmd[\'job\'];job[\'inputs\']={k:Path(v) for k,v in job[\'inputs\'].items()}\n                assert job[\'task_id\']==tid\n                studio.stage1(job,export=False)\n            else:\n                through=\'stage1\' if stage==\'stage2\' else \'stage2\'\n                studio.resume_archive(cmd[\'archive\'],tid,through)\n                if stage==\'stage2\':studio.stage2(tid,export=False)\n                else:studio.stage3(tid,export=False)\n            archive=studio.export(tid,stage)\n        cp.atomic_json(result,{\'unit_id\':tid,\'stage\':stage,\'archive\':str(archive),\'status\':\'OK\'})\n    except Exception:\n        traceback.print_exc();cp.atomic_json(result,{\'unit_id\':tid,\'stage\':stage,\'status\':\'FAILED\'});sys.exit(1)\n'

    """Open Avatar Studio 1.0: versioned, resumable long-task orchestration.
    The embedded 1.1 runtime is used ONLY as a bounded segment engine.
    No total text, duration or frame-count cap is imposed by this scheduler.
    """
    import copy
    import hashlib
    import json
    import math
    import os
    import re
    import shutil
    import subprocess
    import sys
    import tempfile
    import time
    import wave
    import zipfile
    from fractions import Fraction
    from pathlib import Path

    import checkpoint_manager as cp
    import environment_manager as em
    import pipeline as oas
    import stage1_engine as s1
    import continuity as cont

    SCHEMA_V1 = 'oas-long-task-v1'
    PROGRESS_NAME = 'oas-progress.json'
    STAGES_V1 = ('stage1','stage2','stage3')
    RATE = 24000
    FPS = 25
    FRAME_SAMPLES = RATE // FPS
    ENGINE_SHA = 'f0ddb13b6d236d1724f5bd9b8137a561405a3a22b9abb34758c0f5a771f3c6cc'
    # A separate scheduler build fingerprint is injected by the notebook builder.


    def ensure(ok, message):
        if not ok:
            raise RuntimeError(message)


    def canonical(obj):
        return json.dumps(obj, sort_keys=True, ensure_ascii=False, separators=(',',':'), allow_nan=False).encode('utf-8')


    def sha_json(obj):
        return hashlib.sha256(canonical(obj)).hexdigest()


    def write_json(path, obj):
        # The legacy per-segment manifest size limits must not cap a long task index.
        path = Path(path); path.parent.mkdir(parents=True,exist_ok=True)
        with tempfile.NamedTemporaryFile(dir=path.parent, prefix='.oas-',delete=False) as f:
            p=Path(f.name)
            try:
                f.write(canonical(obj)+b'\n'); f.flush(); os.fsync(f.fileno()); f.close()
                os.replace(p,path)
            finally:
                p.unlink(missing_ok=True)


    def read_json(path):
        p=Path(path)
        ensure(p.is_file() and not p.is_symlink(),'Missing regular progress file: '+str(p))
        return json.loads(p.read_text(encoding='utf-8'))


    def ishex(v):
        return isinstance(v,str) and re.fullmatch(r'[0-9a-f]{64}',v) is not None


    def artifact_record(path):
        r=cp.file_record(path); ensure(r['bytes']>0,'Empty artifact'); return r


    def check_record(r):
        ensure(isinstance(r,dict) and set(r)=={'bytes','sha256'} and type(r['bytes']) is int and r['bytes']>0 and ishex(r['sha256']), 'Invalid artifact digest')


    def text_chunks(text, limit):
        """Linear scan with punctuation-aware boundaries; only whitespace is trimmed.
        No total length check; never drop any non-whitespace character.
        """
        ensure(type(limit) is int and 80<=limit<=6000,'Invalid text partition limit (80-6000)')
        text=text.strip(); ensure(bool(text),'Script is empty')
        pos=0; out=[]
        while pos<len(text):
            end=min(len(text),pos+limit)
            if end<len(text):
                window=text[pos:end]; floor=max(1,limit//2)
                for pat in (r'[\u3002\uff01\uff1f.!?;\uff1b\n]+',r'[\uff0c\u3001\uff1a,:]+',r'\s+'):
                    choices=[m.end() for m in re.finditer(pat,window) if m.end()>=floor]
                    if choices:
                        end=pos+choices[-1];break
            chunk=text[pos:end].strip()
            if chunk: out.append(chunk)
            pos=end
        ensure(bool(out),'No speakable text chunks')
        return out


    def audio_chunks(source, folder, seconds, speech_speed=1.0):
        """Decode once as raw PCM; bounded RAM, frame-aligned quiet cuts.
        A one-second lookahead prevents an invalid sub-second final segment.
        Works beyond RIFF/WAV's aggregate 4GiB limit. Every sample is retained.
        """
        import numpy as np
        ensure(type(seconds) is int and 10<=seconds<=600,'DITTO_SEGMENT_SECONDS must be 10-600')
        info=s1.read_media(source)
        tracks=[x for x in info['streams'] if x['codec_type']=='audio']
        ensure(len(tracks)==1,'Exactly one audio track required')
        duration=s1.stream_duration(tracks[0],info)
        speed=s1.validate_speech_speed(speech_speed)
        ensure(duration>=1,'Narration must be at least one second')
        maximum=seconds*RATE; lookahead=min(10*RATE,maximum//3)
        folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)
        count=0; total=0; records=[]; buf=b''; eof=False
        with tempfile.TemporaryFile() as err:
            proc=subprocess.Popen(['ffmpeg','-nostdin','-v','error','-xerror','-i',str(source),'-map','0:a:0','-vn','-ac','1','-ar',str(RATE),*s1.speech_speed_filter(speed),'-f','s16le','pipe:1'],stdout=subprocess.PIPE,stderr=err)
            try:
                while True:
                    want=2*(maximum+lookahead)-len(buf)
                    if not eof and want>0:
                        part=proc.stdout.read(want)
                        buf+=part
                        if len(part)<want: eof=True
                    ensure(len(buf)%2==0,'Unaligned PCM stream')
                    n=len(buf)//2
                    if n==0 and eof:break
                    if eof and n<=maximum:
                        ensure(n>=RATE,'Decoded narration is shorter than one second')
                        end=n
                    else:
                        hi=min(maximum,n-lookahead)//FRAME_SAMPLES*FRAME_SAMPLES
                        lo=max(RATE,hi-5*RATE);lo=(lo+FRAME_SAMPLES-1)//FRAME_SAMPLES*FRAME_SAMPLES
                        ensure(lo<=hi,'Cannot partition audio with a valid tail')
                        signal=np.frombuffer(buf,dtype='<i2')
                        def energy(i):
                            x=signal[max(0,i-4*FRAME_SAMPLES):min(n,i+4*FRAME_SAMPLES)].astype(np.float64)
                            return (float(np.mean(x*x)),-i)
                        end=min(range(lo,hi+1,FRAME_SAMPLES),key=energy)
                    count+=1;total+=end
                    path=folder/f'part_{count:08d}.wav'
                    with wave.open(str(path),'wb') as w:
                        w.setnchannels(1);w.setsampwidth(2);w.setframerate(RATE);w.writeframes(buf[:2*end])
                    records.append((path,end));buf=buf[2*end:]
                ensure(proc.wait()==0,'FFmpeg audio decode failed; source may be incomplete')
            finally:
                if proc.poll() is None:
                    proc.kill();proc.wait()
                proc.stdout.close()
        ensure(records and total>=RATE and all(n%FRAME_SAMPLES==0 for _,n in records[:-1]),'Invalid audio partition')
        if speed != 1.0:
            print('SPEECH_SPEED_APPLIED:',speed,'INPUT_SECONDS:',duration,'OUTPUT_SECONDS:',total/RATE,'SCOPE: continuous input before partition',flush=True)
        print('AUDIO_PLAN:',count,'SEGMENTS, EXACT_PCM_SAMPLES:',total,'SECONDS:',total/RATE,flush=True)
        return records


    def input_paths(mode, language, demo_portrait, demo_audio, portrait, audio, script, reference_audio, reference_text, input_root):
        ensure(mode in ('demo_audio','demo_text','audio','text'),'Invalid INPUT_MODE')
        if mode.startswith('demo_'):
            root,has_long=verify_demo_mount(input_root)
            ensure(demo_portrait in ('portrait_01.png','portrait_02.png'),'Invalid DEMO_PORTRAIT')
            portrait=root/demo_portrait
            if mode=='demo_audio':
                ensure(demo_audio in DEMO_AUDIO_FILES,'Invalid DEMO_AUDIO')
                ensure(demo_audio!='native_blink_long_test.wav' or has_long,'Long demo audio not mounted')
                audio=root/demo_audio;mode='audio'
            else:
                ensure(language in ('English','Chinese'),'Demo text language must be English or Chinese')
                script=root/('script_zh.txt' if language=='Chinese' else 'script_en.txt')
                reference_audio=root/'reference.wav';reference_text=root/'reference.txt';mode='text'
        paths={'portrait':s1.input_file(portrait)}
        s1.inspect_photo(paths['portrait'])
        if mode=='audio': paths['audio']=s1.input_file(audio)
        else:
            ensure(language in ('English','Chinese','Auto'),'Invalid LANGUAGE')
            paths.update(script=s1.input_file(script),reference_audio=s1.input_file(reference_audio))
            s1.inspect_audio(paths['reference_audio'],30)
            if reference_text: paths['reference_text']=s1.input_file(reference_text)
        return mode,paths


    def make_plan(folder, *, mode, paths, language, blink_mode, expression, seed, open_frames, chunk_chars, segment_seconds, text_segment_chars=2400, warmup_seconds=1.2, smooth_seams=True, speech_speed=1.0):
        use_lp,_=resolve_workflow(blink_mode,open_frames,True)
        speed=s1.validate_speech_speed(speech_speed)
        print('SPEECH_SPEED:',speed,'MODE:',mode,flush=True)
        ensure(type(seed) is int and 0<=seed<2**32,'Invalid SEED')
        ensure(type(chunk_chars) is int and 80<=chunk_chars<=800,'TTS_CHUNK_CHARACTERS must be 80-800')
        ensure(type(text_segment_chars) is int and max(600,chunk_chars)<=text_segment_chars<=4000,'TEXT_SEGMENT_CHARACTERS must be 600-4000 and >= TTS_CHUNK_CHARACTERS')
        ensure(type(warmup_seconds) in (int,float) and math.isfinite(warmup_seconds) and 0<=warmup_seconds<=3,'DITTO_WARMUP_SECONDS must be 0-3')
        ensure(type(smooth_seams) is bool,'SMOOTH_SEAMS must be True or False')
        ensure(type(segment_seconds) is int and 10<=segment_seconds<=600,'DITTO_SEGMENT_SECONDS must be 10-600')
        folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)
        config=dict(mode=mode,language=language if mode=='text' else None,blink_mode=blink_mode,expression=expression,seed=seed,open_frames=open_frames,chunk_chars=chunk_chars,segment_seconds=segment_seconds,text_segment_chars=text_segment_chars,warmup_seconds=warmup_seconds,smooth_seams=smooth_seams,speech_speed=speed)
        inputs={k:artifact_record(v) for k,v in paths.items()}
        # The plan identity contains source hashes and behavior settings, never paths or Secrets.
        base=dict(engine_sha=ENGINE_SHA,scheduler_sha=SCHEDULER_SHA,inputs=inputs,config=config)
        base_id=sha_json(base)
        if mode=='text':
            text=paths['script'].read_text(encoding='utf-8-sig')
            chunks=text_chunks(text,text_segment_chars)
            print('SCRIPT_CHARACTERS:',len(text.strip()),'TASK_SEGMENTS:',len(chunks),'TTS_TARGET_CHARACTERS:',chunk_chars,flush=True)
            parts=[]
            for i,chunk in enumerate(chunks,1):
                p=folder/f'part_{i:08d}.txt';p.write_text(chunk,encoding='utf-8');parts.append((p,None))
        else:
            parts=audio_chunks(paths['audio'],folder,segment_seconds,speech_speed=speed)
        jobs=[];units=[]
        for i,(part,samples) in enumerate(parts,1):
            # The old runtime's duration/character limits apply ONLY to this small unit.
            opts=dict(portrait=paths['portrait'],language=language,seed=(seed if mode=='text' else (seed+i-1)%2**32),expression=expression,blink_enabled=use_lp,
                      max_script_characters=text_segment_chars,tts_chunk_characters=chunk_chars,max_video_seconds=(1200 if mode=='text' else 600),ditto_segment_seconds=segment_seconds,ditto_warmup_seconds=warmup_seconds,smooth_seams=smooth_seams,speech_speed=(speed if mode=='text' else 1.0))
            if mode=='text':
                job=oas.prepare_job(mode='text',script=part,reference_audio=paths['reference_audio'],reference_text=paths.get('reference_text'),**opts)
            else:
                job=oas.prepare_job(mode='audio',audio=part,**opts)
            job=configure_eye_mode(job,blink_mode,open_frames)
            job['config']['blink']['seed']=(seed+i-1)%2**32
            # For outer AUDIO partitions, use true adjacent audio as inference context.
            # Context is hashed with the unit; it is NOT included in the final narration.
            if mode=='audio' and warmup_seconds>0:
                pre=round(warmup_seconds*FPS)*FRAME_SAMPLES;post=min(round(warmup_seconds*FPS),12)*FRAME_SAMPLES
                for label,neighbor,amount,tail in [('context_before',i-2,pre,True),('context_after',i,post,False)]:
                    if amount and 0<=neighbor<len(parts):
                        with wave.open(str(parts[neighbor][0]),'rb') as w:
                            n=min(amount,w.getnframes())
                            if tail:w.setpos(w.getnframes()-n)
                            data=w.readframes(n)
                        ensure(len(data)==2*n and n>0,'Missing adjacent PCM context')
                        extra=folder/f'{label}_{i:08d}.pcm';extra.write_bytes(data);job['inputs'][label]=extra
            revisions=dict(job['identity']['revisions'],long_task_source=base_id,long_task_index=i)
            job['identity']=cp.make_identity(job['inputs'],job['config'],revisions)
            job['task_id']=cp.task_id(job['identity'])
            units.append(dict(index=i,unit_id=job['task_id'],samples=samples,source=artifact_record(part)))
            jobs.append(job)
        ensure({k:artifact_record(v) for k,v in paths.items()}==inputs,'Source changed while planning')
        plan=dict(base=base,units=units,use_lp=use_lp)
        return plan,jobs


    def expected_version(stage,index,n):
        return 1+STAGES_V1.index(stage)*n+index


    def artifact_name(stage,index):
        return f'{stage}_{index:08d}.bin'


    def route(plan):
        return STAGES_V1 if plan['use_lp'] else ('stage1',)


    def validate_progress(p, ref=None, expected_id='', expected_plan=None):
        ensure(isinstance(p,dict) and set(p)=={'schema','dataset','task_id','plan','version','records','final'},'Unrecognized V1.0 progress index')
        ensure(p['schema']==SCHEMA_V1 and p['dataset']==ref,'Wrong Dataset or journal schema')
        plan=p['plan'];tid=p['task_id']
        ensure(isinstance(plan,dict) and set(plan)=={'base','units','use_lp'} and ishex(tid) and sha_json(plan)==tid,'Task identity mismatch')
        ensure(not expected_id or tid==expected_id,'EXPECTED_TASK_ID mismatch')
        ensure(expected_plan is None or plan==expected_plan,'REMOTE_TASK_MISMATCH: source/config/segmentation differs; use original settings or a new Dataset')
        base=plan['base']; cfg=base['config']
        ensure(base['engine_sha']==ENGINE_SHA and base['scheduler_sha']==SCHEDULER_SHA,'Use the matching Notebook build (V1.0 or V1.1 RC checkpoints need their original notebook)')
        s1.validate_speech_speed(cfg['speech_speed'])
        ensure(type(plan['use_lp']) is bool and (cfg['blink_mode']=='liveportrait')==plan['use_lp'],'Invalid route')
        units=plan['units'];n=len(units)
        ensure(isinstance(units,list) and n>0,'Empty plan')
        for i,u in enumerate(units,1):
            ensure(isinstance(u,dict) and set(u)=={'index','unit_id','samples','source'} and u['index']==i and ishex(u['unit_id']),'Invalid segment plan')
            check_record(u['source'])
            if cfg['mode']=='audio':
                ensure(type(u['samples']) is int and u['samples']>=RATE and (i==n or u['samples']%FRAME_SAMPLES==0),'Invalid audio sample partition')
            else: ensure(cfg['mode']=='text' and u['samples'] is None,'Invalid text plan')
        ensure(len({u['unit_id'] for u in units})==n,'Duplicate unit identity')
        records=p['records'];ensure(isinstance(records,dict) and set(records)==set(STAGES_V1),'Invalid stage index')
        count=0
        for stage in STAGES_V1:
            entries=records[stage]
            ensure(isinstance(entries,list) and len(entries)<=n,'Invalid stage count')
            ensure(stage in route(plan) or not entries,'Unexpected stage for direct mode')
            if stage!='stage1' and entries:
                ensure(len(records[STAGES_V1[STAGES_V1.index(stage)-1]])==n,'Stage order violated')
            for i,e in enumerate(entries,1):
                ensure(isinstance(e,dict) and set(e)=={'version','name','record','unit_id'} and e['version']==expected_version(stage,i,n) and e['name']==artifact_name(stage,i) and e['unit_id']==units[i-1]['unit_id'],'Invalid history record')
                check_record(e['record'])
            count+=len(entries)
        if p['final'] is not None:
            f=p['final'];ensure(all(len(records[s])==n for s in route(plan)),'Final published before all segments completed')
            ensure(isinstance(f,dict) and set(f)=={'version','name','record','media'} and f['version']==count+2 and f['name']==f'oas_{tid}_final.mp4','Invalid final record')
            check_record(f['record']); ensure(isinstance(f['media'],dict),'Missing final media report')
        ensure(type(p['version']) is int and p['version']==1+count+(p['final'] is not None),'Invalid Dataset version sequence')
        return p


    def new_progress(plan,ref):
        p=dict(schema=SCHEMA_V1,dataset=ref,task_id=sha_json(plan),plan=plan,version=1,records={s:[] for s in STAGES_V1},final=None)
        return validate_progress(p,ref)


    def advance(p,stage,source,media=None):
        """Append one immutable artifact reference; never modify previous records."""
        validate_progress(p,p['dataset']);q=copy.deepcopy(p);n=len(q['plan']['units'])
        ensure(q['final'] is None,'Task already complete')
        q['version']+=1
        rec=artifact_record(source)
        if stage=='final':
            e=dict(version=q['version'],name=f"oas_{q['task_id']}_final.mp4",record=rec,media=media)
            q['final']=e
        else:
            ensure(stage in route(q['plan']),'Invalid stage')
            i=len(q['records'][stage])+1
            ensure(i<=n,'Stage already complete')
            e=dict(version=q['version'],name=artifact_name(stage,i),record=rec,unit_id=q['plan']['units'][i-1]['unit_id'])
            q['records'][stage].append(e)
        validate_progress(q,q['dataset'])
        return q,e


    def latest_entry(p):
        if p['final']: return p['final']
        for stage in reversed(STAGES_V1):
            if p['records'][stage]: return p['records'][stage][-1]
        return None


    def print_progress(p):
        n=len(p['plan']['units']); stages=route(p['plan'])
        nxt=next((f'{s}:{len(p["records"][s])+1}' for s in stages if len(p['records'][s])<n),'final' if p['final'] is None else 'complete')
        print('V1_PROGRESS:',p['task_id'],'VERSION:',p['version'],'TOTAL_SEGMENTS:',n,'DONE:',{s:len(p['records'][s]) for s in stages},'NEXT:',nxt,flush=True)


    class LocalJournal:
        """Local fallback. A complete journal directory is required for local recovery."""
        def __init__(self,folder,plan=None,expected_id=''):
            self.folder=Path(folder).resolve();self.folder.mkdir(parents=True,exist_ok=True)
            self.path=self.folder/PROGRESS_NAME
            if self.path.exists():self.p=validate_progress(read_json(self.path),None,expected_id,plan)
            else:
                ensure(plan is not None,'No local journal; specify remote Dataset or exact local progress path')
                self.p=new_progress(plan,None);ensure(not expected_id or self.p['task_id']==expected_id,'EXPECTED_TASK_ID mismatch');write_json(self.path,self.p)
            last=latest_entry(self.p)
            if last:self._verify(last)
            print_progress(self.p)
        def _path(self,e):return self.folder/'versions'/str(e['version'])/e['name']
        def _verify(self,e):
            p=self._path(e);ensure(artifact_record(p)==e['record'],'Missing or corrupt local checkpoint: '+str(p));return p
        def assert_current(self):ensure(read_json(self.path)==self.p,'Local journal changed; do not run concurrent writers')
        def commit(self,stage,source,media=None):
            self.assert_current();q,e=advance(self.p,stage,source,media)
            dst=self._path(e);dst.parent.mkdir(parents=True,exist_ok=True)
            # If killed after file copy but before index commit, retry from that last verified index.
            tmp=dst.with_name(dst.name+'.partial');shutil.copyfile(source,tmp)
            ensure(artifact_record(tmp)==e['record'],'Local copy changed');os.replace(tmp,dst)
            write_json(self.path,q);self.p=q;print_progress(q)
            return e
        def fetch(self,e,dest):
            shutil.copyfile(self._verify(e),dest);ensure(artifact_record(dest)==e['record'],'Local readback mismatch')
            return Path(dest)
        def close(self):pass


    class RemoteJournal(PrivateDatasetStore):
        """One private Dataset, append-only Kaggle versions, rolling payloads.
        Prior versions are retained; latest index points to all historical artifacts.
        No cumulative copying of old media on each commit. Single writer only.
        """
        manifest_name=PROGRESS_NAME
        def __init__(self,ref,token,folder,plan=None,expected_id='',wait_seconds=600,command_seconds=3600,**kw):
            super().__init__(ref,'0'*64,token,wait_seconds=wait_seconds,command_seconds=command_seconds,**kw)
            self.folder=Path(folder);self.folder.mkdir(parents=True,exist_ok=True);self.path=self.folder/PROGRESS_NAME
            try:
                try:self._private()
                except PersistenceError as exc:
                    # A metadata command failure does NOT prove absence. Only attempt create-only.
                    if plan is None or not str(exc).startswith('KAGGLE_API_COMMAND_FAILED: metadata'):raise
                    self.p=new_progress(plan,ref);self.task_id=self.p['task_id']
                    ensure(not expected_id or self.task_id==expected_id,'EXPECTED_TASK_ID mismatch')
                    self._write_index(self.p,None,None,create=True)
                else:
                    p,rec=self._read_latest_manifest();validate_progress(p,ref,expected_id,plan)
                    self.task_id=p['task_id']
                    self._download_record(p['version'],self.manifest_name,rec)
                    last=latest_entry(p)
                    if last:self._download_record(last['version'],last['name'],last['record'])
                    self._private();self._assert_latest(rec)
                    self.p=p;self.version=p['version'];self._manifest=rec;write_json(self.path,p)
                ensure(not expected_id or self.p['task_id']==expected_id,'EXPECTED_TASK_ID mismatch')
                print('V1_REMOTE_CONNECTED:',self.ref,'VERSION:',self.p['version'],flush=True);print_progress(self.p)
            except BaseException:
                self.close();raise
        @classmethod
        def open_secret(cls,ref,folder,plan=None,expected_id='',wait_seconds=600,command_seconds=3600):
            validate_destination(ref)
            try:
                from kaggle_secrets import UserSecretsClient
                token=UserSecretsClient().get_secret('KAGGLE_API_TOKEN')
            except Exception:raise PersistenceError('Add KAGGLE_API_TOKEN in Secrets and grant this PRIVATE Notebook access.') from None
            return cls(ref,token,folder,plan,expected_id,wait_seconds,command_seconds)
        def _read_latest_manifest(self):
            # No fixed total task-index size cap. Only a regular index, or a one-file ZIP wrapper.
            with tempfile.TemporaryDirectory(prefix='oas-v1-latest-') as d:
                self.command(['download','-d',self.ref,'-f',self.manifest_name,'-p',d,'-o','-q'])
                p=Path(d)/self.manifest_name
                if p.is_file() and not p.is_symlink():data=p.read_bytes()
                else:
                    try:
                        with zipfile.ZipFile(Path(d)/(self.manifest_name+'.zip')) as z:
                            items=z.infolist()
                            if len(items)!=1 or items[0].filename!=self.manifest_name:raise PersistenceError('Unexpected progress ZIP wrapper')
                            data=z.read(items[0])
                    except (OSError,zipfile.BadZipFile):raise PersistenceError('Cannot read latest progress index') from None
                try:return json.loads(data),{'bytes':len(data),'sha256':hashlib.sha256(data).hexdigest()}
                except ValueError:raise PersistenceError('Malformed progress index') from None
        def _write_index(self,q,e,source,create=False):
            validate_progress(q,self.ref)
            if not create:self.assert_current()
            with tempfile.TemporaryDirectory(prefix='oas-v1-upload-') as d:
                d=Path(d);write_json(d/self.manifest_name,q);mrec=artifact_record(d/self.manifest_name);records={}
                if e is not None:
                    shutil.copyfile(source,d/e['name']);ensure(artifact_record(d/e['name'])==e['record'],'Artifact changed during upload staging');records[e['name']]=e['record']
                write_json(d/'dataset-metadata.json',{'id':self.ref,'title':self.ref.split('/')[1],'licenses':[{'name':'other'}],'description':'Private Open Avatar Studio V1.1 long-task recovery. Keep historical versions until final output is verified.'})
                print('V1_REMOTE_SAVE:',self.ref,'VERSION:',q['version'],'ARTIFACT:',e['name'] if e else 'plan',flush=True)
                if create:self.command(['create','-p',str(d),'-t','-r','skip','-q'])
                else:
                    # Explicitly never pass --delete-old-versions. Historical versions contain the media.
                    self.assert_current();self.command(['version','-p',str(d),'-m',f'OAS V1.1 {q["version"]} {e["name"]}','-t','-r','skip','-q'])
                self._verify(q['version'],mrec,records)
            self.p=q;self.version=q['version'];self._manifest=mrec;write_json(self.path,q)
            print('V1_REMOTE_VERIFIED:',f'https://www.kaggle.com/datasets/{self.ref}/versions/{self.version}',flush=True);print_progress(q)
        def commit(self,stage,source,media=None):
            q,e=advance(self.p,stage,source,media);self._write_index(q,e,source);return e
        def fetch(self,e,dest):
            self._private();self._download_record(e['version'],e['name'],e['record'],destination=dest);return Path(dest)

    class SegmentWorker:
        """Reuse one subprocess per stage; each stage has its own fresh cache.
        The frozen runtime's cross-stage GPU guard remains active inside each cache.
        """
        def __init__(self,stage,root,folder,input_root,accept,check_network,monitor):
            self.stage=stage;self.folder=Path(folder);self.folder.mkdir(parents=True,exist_ok=True)
            self.work=self.folder/'work';self.cache=self.folder/'cache';self.cache.mkdir()
            self.result=self.folder/'result.json'
            script=self.folder/'worker.py';script.write_text(SEGMENT_WORKER_SOURCE,encoding='utf-8')
            spec=self.folder/'spec.json';write_json(spec,dict(work=str(self.work),cache=str(self.cache),result=str(self.result),input_root=str(input_root),accept=accept,check_network=check_network,monitor=monitor))
            self.proc=subprocess.Popen([sys.executable,'-B',str(script),str(root),str(spec),stage],stdin=subprocess.PIPE,text=True,bufsize=1,env=em.worker_env(self.cache,cpu=stage=='stage3'),start_new_session=True)
        def __enter__(self):return self
        def __exit__(self,*exc):self.close()
        def run(self,unit_id,job=None,archive=None):
            self.result.unlink(missing_ok=True)
            cmd={'unit_id':unit_id}
            if job is not None:cmd['job']=dict(job,inputs={k:str(v) for k,v in job['inputs'].items()})
            if archive is not None:cmd['archive']=str(archive)
            self.proc.stdin.write(json.dumps(cmd)+'\n');self.proc.stdin.flush()
            while not self.result.is_file():
                ensure(self.proc.poll() is None,'Segment worker exited before commit; resume from last verified index')
                time.sleep(.25)
            r=read_json(self.result)
            ensure(r.get('status')=='OK' and r.get('unit_id')==unit_id and r.get('stage')==self.stage,'Segment inference failed; previous commits remain valid')
            result=Path(r['archive']).resolve()
            ensure(result.is_relative_to(self.work.resolve()) and result.is_file() and not result.is_symlink(),'Unexpected segment output')
            return result
        def release(self,unit_id,archive):
            # Only after journal commit. Clean this worker's own scratch media, never the journal.
            ensure(ishex(unit_id),'Invalid unit ID')
            shutil.rmtree(self.work/'tasks'/unit_id)
            Path(archive).unlink()
        def close(self):
            if self.proc.poll() is None:
                try:self.proc.stdin.write('{"stop":true}\n');self.proc.stdin.flush();self.proc.wait(timeout=10)
                except (OSError,subprocess.TimeoutExpired):
                    import signal
                    try:os.killpg(self.proc.pid,signal.SIGTERM)
                    except ProcessLookupError:pass
                    try:self.proc.wait(timeout=10)
                    except subprocess.TimeoutExpired:
                        try:os.killpg(self.proc.pid,signal.SIGKILL)
                        except ProcessLookupError:pass
                        self.proc.wait()
            if self.proc.stdin:self.proc.stdin.close()


    def stage_ready(journal,stage):
        p=journal.p;n=len(p['plan']['units'])
        if stage not in route(p['plan']):return False
        if stage=='stage1':return True
        previous=STAGES_V1[STAGES_V1.index(stage)-1]
        if len(p['records'][previous])!=n:
            print('V1_STAGE_BLOCKED:',stage,'WAITING_FOR:',previous,'COMPLETED:',len(p['records'][previous]),'OF',n,flush=True)
            return False
        return True


    def execute_stage(journal,stage,*,root,work,input_root,jobs=None,accept=False,check_network=True,monitor=True,max_new=0,max_seconds=0,worker_cls=SegmentWorker):
        ensure(type(max_new) is int and max_new>=0,'MAX_NEW_SEGMENTS must be nonnegative; 0 means unlimited')
        ensure(type(max_seconds) is int and max_seconds>=0,'MAX_STAGE_RUN_SECONDS must be nonnegative; 0 means unlimited')
        if not stage_ready(journal,stage):return 'BLOCKED'
        p=journal.p;n=len(p['plan']['units']);done=len(p['records'][stage])
        if done==n:
            print('V1_STAGE_ALREADY_COMPLETE:',stage,'SKIPPED_SEGMENTS:',n,flush=True);return 'COMPLETE'
        if stage in ('stage1','stage2'):ensure(accept is True,'Explicit model terms acceptance required before inference')
        if stage=='stage1':ensure(jobs is not None and len(jobs)==n,'Stage 1 recovery requires original source and settings')
        print('V1_STAGE_START:',stage,'SKIPPED_COMMITTED:',done,'REMAINING:',n-done,flush=True)
        start=time.monotonic();processed=0
        Path(work).mkdir(parents=True,exist_ok=True)
        with tempfile.TemporaryDirectory(prefix='oas-v1-'+stage+'-',dir=work) as tmp:
            with worker_cls(stage,root,tmp,input_root,accept,check_network,monitor) as worker:
                for i in range(done+1,n+1):
                    if processed and ((max_new and processed>=max_new) or (max_seconds and time.monotonic()-start>=max_seconds)):
                        print('V1_PAUSED:',stage,'NEXT_SEGMENT:',i,'REMOTE_OR_LOCAL_VERSION:',journal.p['version'],flush=True);return 'PAUSED'
                    journal.assert_current()
                    uid=p['plan']['units'][i-1]['unit_id'];prev=None
                    print('V1_SEGMENT_START:',stage,i,'/',n,flush=True)
                    if stage!='stage1':
                        previous=STAGES_V1[STAGES_V1.index(stage)-1];e=journal.p['records'][previous][i-1]
                        prev=journal.fetch(e,Path(tmp)/'incoming.bin')
                    try:
                        out=worker.run(uid,job=jobs[i-1] if stage=='stage1' else None,archive=prev)
                        # Verify full archive with the original checkpoint manager before the journal is advanced.
                        verifier=oas.Studio(work=worker.work,input_root=input_root)
                        ensure(verifier.store.completed(uid)==list(STAGES_V1[:STAGES_V1.index(stage)+1]),'Wrong segment checkpoint scope')
                        verifier.verify_archive_prefix(out,uid,stage)
                        journal.commit(stage,out)
                        worker.release(uid,out)
                        processed+=1
                        print('V1_SEGMENT_COMMITTED:',stage,i,'/',n,flush=True)
                    finally:
                        if prev is not None:Path(prev).unlink(missing_ok=True)
        print('V1_STAGE_COMPLETE:',stage,'SEGMENTS:',n,flush=True);return 'COMPLETE'


    def run_ffmpeg(args):
        # Long-task assembly is not constrained by an aggregate wall-clock timeout.
        subprocess.run(['ffmpeg','-nostdin','-hide_banner','-loglevel','error','-xerror','-y',*map(str,args)],check=True)


    def media_video_info(path):
        info=s1.read_media(path);videos=[s for s in info['streams'] if s['codec_type']=='video'];audios=[s for s in info['streams'] if s['codec_type']=='audio']
        ensure(len(videos)==1,'Expected one video stream');v=videos[0]
        count=v.get('nb_frames')
        if not str(count).isdigit():
            count=subprocess.check_output(['ffprobe','-v','error','-count_frames','-select_streams','v:0','-show_entries','stream=nb_read_frames','-of','default=nw=1:nk=1',str(path)],text=True).strip()
        return dict(frames=int(count),fps=Fraction(v['avg_frame_rate']),width=int(v['width']),height=int(v['height']),video_duration=s1.stream_duration(v,info),audio_duration=s1.stream_duration(audios[0],info) if len(audios)==1 else None,audio_tracks=len(audios),codec=v.get('codec_name'),pix_fmt=v.get('pix_fmt'),sar=v.get('sample_aspect_ratio'))


    def prepare_assembly_clip(video,narration,dest,raw_audio,geometry=None,expected_samples=None,copy_video=False):
        """Per-segment video CFR normalization + exact PCM timeline.
        Each segment ends on a frame boundary. Only <40ms silence padding may be added.
        Never concatenate independently encoded AAC packets (encoder delays accumulate).
        """
        with wave.open(str(narration),'rb') as w:
            ensure((w.getnchannels(),w.getsampwidth(),w.getframerate(),w.getcomptype())==(1,2,RATE,'NONE'),'Invalid normalized narration')
            samples=w.getnframes();ensure(samples>=RATE,'Invalid narration samples')
            if expected_samples is not None:ensure(samples==expected_samples,'Audio source sample count changed')
            frames=(samples+FRAME_SAMPLES-1)//FRAME_SAMPLES;pad=frames*FRAME_SAMPLES-samples
            v=media_video_info(video);g=(v['width'],v['height'])
            ensure(v['fps']==FPS and v['audio_tracks']==1 and all(x%2==0 for x in g),'Unexpected segment geometry/fps/audio')
            ensure(geometry is None or g==geometry,'Segment geometry changed; cannot assemble')
            ensure(abs(v['frames']-frames)<=5 and abs(v['video_duration']-samples/RATE)<=.5 and abs(v['audio_duration']-samples/RATE)<=.5,'Segment is truncated or out of sync')
            if copy_video and v['frames']==frames and v['codec']=='h264' and v['pix_fmt']=='yuv420p' and v['sar'] in (None,'1:1'):
                # One unit: preserve the already aligned H.264 picture, remux audio once.
                run_ffmpeg(['-i',video,'-map','0:v:0','-an','-c:v','copy','-video_track_timescale','25000',dest])
                print('V1_SINGLE_CLIP_VIDEO_COPY: no additional image re-encode',flush=True)
            else:
                run_ffmpeg(['-threads','1','-i',video,'-map','0:v:0','-an','-vf',f'fps={FPS},tpad=stop_mode=clone:stop_duration=0.24,trim=end_frame={frames},setpts=N/({FPS}*TB),setsar=1','-r',str(FPS),'-vsync','cfr','-c:v','libx264','-threads','2','-preset','veryfast','-crf','17','-pix_fmt','yuv420p','-video_track_timescale','25000',dest])
            aligned=media_video_info(dest)
            ensure(aligned['frames']==frames and aligned['fps']==FPS and (aligned['width'],aligned['height'])==g and aligned['audio_tracks']==0,'Aligned clip verification failed')
            copied=0
            while True:
                b=w.readframes(65536)
                if not b:break
                ensure(len(b)%2==0,'Invalid PCM');raw_audio.write(b);copied+=len(b)//2
            ensure(copied==samples,'Narration PCM truncated')
            raw_audio.write(b'\0'*(2*pad))
        return frames,samples,pad,g


    def publish_final(journal,*,work,output_root,input_root='/kaggle/input'):
        p=journal.p;n=len(p['plan']['units']);stage=route(p['plan'])[-1]
        if len(p['records'][stage])!=n:
            print('V1_FINAL_BLOCKED: complete all segments of',stage,'first',flush=True);return None
        output_root=Path(output_root);output_root.mkdir(parents=True,exist_ok=True)
        target=output_root/f'oas_{p["task_id"]}_final.mp4'
        if p['final'] is not None:
            if target.is_file() and not target.is_symlink() and artifact_record(target)==p['final']['record']:
                print('V1_FINAL_REUSED:',target,flush=True);return target
            with tempfile.TemporaryDirectory(prefix='oas-v1-final-fetch-',dir=output_root) as tmp:
                f=journal.fetch(p['final'],Path(tmp)/'final.mp4');os.replace(f,target)
            print('V1_FINAL_RESTORED:',target,flush=True);return target
        Path(work).mkdir(parents=True,exist_ok=True)
        with tempfile.TemporaryDirectory(prefix='oas-v1-assembly-',dir=work) as tmp:
            d=Path(tmp);geometry=None;frames=samples=padding=0;clips=[];frame_counts=[]
            with (d/'narration.s16le').open('wb') as raw:
                for i,e in enumerate(p['records'][stage],1):
                    journal.assert_current()
                    archive=journal.fetch(e,d/'incoming.bin');uid=e['unit_id']
                    studio=oas.Studio(work=d/'restore',input_root=input_root)
                    studio.restore(archive,uid)
                    ensure(studio.store.completed(uid)==list(STAGES_V1[:STAGES_V1.index(stage)+1]),'Wrong assembly checkpoint scope')
                    unit=studio.store.path(uid)
                    video=unit/stage/('final_video.mp4' if stage=='stage3' else 'ditto_video.mp4')
                    dest=d/f'clip_{i:08d}.mp4'
                    nf,ns,npad,geometry=prepare_assembly_clip(video,unit/'stage1'/'narration.wav',dest,raw,geometry,p['plan']['units'][i-1]['samples'],copy_video=(n==1))
                    frames+=nf;samples+=ns;padding+=npad;clips.append(dest.name);frame_counts.append(nf)
                    shutil.rmtree(unit);archive.unlink()
                    print('V1_ASSEMBLY_CLIP:',i,'/',n,'FRAMES:',nf,'PAD_SAMPLES:',npad,flush=True)
            ensure(frames*FRAME_SAMPLES==samples+padding and (d/'narration.s16le').stat().st_size==2*(samples+padding),'Final PCM timeline mismatch')
            seams=cont.smooth_joins([d/x for x in clips],frame_counts,d/'narration.s16le',d/'seam_patches',enabled=p['plan']['base']['config']['smooth_seams'])
            (d/'clips.ffconcat').write_text('ffconcat version 1.0\n'+''.join('file '+x+'\n' for x in clips),encoding='utf-8')
            # MP4 output is assembled using disk streams rather than storing all frames/samples in RAM.
            staged=d/'joined.mp4'
            run_ffmpeg(['-f','concat','-safe','1','-i',d/'clips.ffconcat','-f','s16le','-ar',str(RATE),'-ac','1','-i',d/'narration.s16le','-map','0:v:0','-map','1:a:0','-c:v','copy','-c:a','aac','-b:a','160k','-movflags','+faststart',staged])
            v=media_video_info(staged);duration=frames/FPS
            ensure(v['frames']==frames and v['fps']==FPS and (v['width'],v['height'])==geometry and v['audio_tracks']==1,'Final video frame/geometry verification failed')
            ensure(abs(v['video_duration']-duration)<=.05 and abs(v['audio_duration']-duration)<=.1,'Final audio/video timeline verification failed')
            # Full decode validates media integrity. No fixed aggregate duration limit.
            run_ffmpeg(['-threads','1','-i',staged,'-map','0:v:0','-map','0:a:0','-f','null','-'])
            media=dict(segments=n,seams=seams,speech_speed=p['plan']['base']['config']['speech_speed'],frames=frames,fps=FPS,width=geometry[0],height=geometry[1],original_pcm_samples=samples,padding_pcm_samples=padding,duration_seconds=duration,video_duration=v['video_duration'],audio_duration=v['audio_duration'])
            os.replace(staged,target)
            journal.commit('final',target,media)
            write_json(output_root/f'oas_{p["task_id"]}_final_report.json',media)
        print('V1_COMPLETE:',target,'DURATION:',duration,'SHA256:',artifact_record(target)['sha256'],flush=True)
        return target


    def assert_prefix(old,new):
        ensure(old['task_id']==new['task_id'] and new['version']>=old['version'],'Remote/local journal is older than the selected checkpoint')
        for stage in STAGES_V1:
            ensure(new['records'][stage][:len(old['records'][stage])]==old['records'][stage],'Journal history differs from selected checkpoint')
        ensure(old['final'] is None or new['final']==old['final'],'Final record history differs')


    def open_journal(work,ref='',plan=None,expected_id='',restore_manifest='',wait_seconds=600,command_seconds=3600):
        ensure(not expected_id or ishex(expected_id),'EXPECTED_TASK_ID must be blank or 64 hex characters')
        ensure(type(wait_seconds) is int and wait_seconds>0 and type(command_seconds) is int and command_seconds>0,'Remote timeouts must be positive integers')
        work=Path(work).resolve();work.mkdir(parents=True,exist_ok=True)
        old=None
        if restore_manifest:
            path=Path(restore_manifest).resolve(strict=True);old=read_json(path)
            validate_progress(old,old.get('dataset'),expected_id,plan)
            expected_id=old['task_id']
            if old['dataset']:
                ensure(not ref or ref==old['dataset'],'Restore manifest and PERSIST_DATASET disagree');ref=old['dataset']
            else:
                ensure(not ref,'Local journal cannot be promoted to remote automatically; use original local journal')
                ensure(path.name==PROGRESS_NAME,'Use exact oas-progress.json filename')
                dest=work/'journals'/expected_id
                if path.parent!=dest:
                    dest.mkdir(parents=True,exist_ok=True);marker=dest/PROGRESS_NAME
                    if marker.is_file():
                        present=validate_progress(read_json(marker),None,expected_id,plan);assert_prefix(old,present)
                    else:
                        for entry in [e for st in STAGES_V1 for e in old['records'][st]]+([old['final']] if old['final'] else []):
                            rel=Path('versions')/str(entry['version'])/entry['name'];src=path.parent/rel;dst=dest/rel
                            ensure(artifact_record(src)==entry['record'],'Local restore artifact missing or corrupt: '+str(rel))
                            dst.parent.mkdir(parents=True,exist_ok=True);shutil.copyfile(src,dst)
                            ensure(artifact_record(dst)==entry['record'],'Local restore copy changed')
                        write_json(marker,old)
                    return LocalJournal(dest,plan,expected_id)
                return LocalJournal(path.parent,plan,expected_id)
        if ref:
            validate_destination(ref)
            # Local snapshot is not authoritative; a newly started Session always reads latest remote index.
            folder=work/'remote-snapshots'/ref.replace('/','__')
            j=RemoteJournal.open_secret(ref,folder,plan,expected_id,wait_seconds,command_seconds)
            try:
                if old is not None:assert_prefix(old,j.p)
            except BaseException:
                j.close();raise
            return j
        if plan is None:
            ensure(expected_id,'Cold local recovery needs exact RESTORE_MANIFEST or EXPECTED_TASK_ID')
            folder=work/'journals'/expected_id
        else:
            tid=sha_json(plan);ensure(not expected_id or expected_id==tid,'EXPECTED_TASK_ID mismatch');folder=work/'journals'/tid
        print('V1_LOCAL_ONLY: /kaggle/working is NOT durable across full container loss. Configure PERSIST_DATASET for long jobs.',flush=True)
        return LocalJournal(folder,plan,expected_id)


    def choose_journal_for_cell(stage,work,ref,restore_manifest,expected_id,wait_seconds,command_seconds,active_ref='',active_manifest='',active_id=''):
        # Explicit selection is authoritative; never silently mix two jobs.
        if active_id:
            ensure(not expected_id or expected_id==active_id,'Active task and expected task differ')
            if ref and active_ref:ensure(ref==active_ref,'Active task and persistence destinations differ')
            if ref and active_manifest and not active_ref:raise RuntimeError('Cannot silently switch active local task to a new remote Dataset')
        return open_journal(work,ref or active_ref,expected_id=expected_id or active_id,restore_manifest=restore_manifest or (active_manifest if not (ref or active_ref) else ''),wait_seconds=wait_seconds,command_seconds=command_seconds)



    ensure(type(STAGE3_ASSEMBLE_FINAL) is bool,'STAGE3_ASSEMBLE_FINAL must be True or False')
    _ctx=globals().get('_OAS_V1_CONTEXT') or {}
    _work=Path(STAGE3_WORK_ROOT or _ctx.get('work') or globals().get('WORK_ROOT','/kaggle/working/open-avatar-studio-v11')).resolve()
    _ref=STAGE3_PERSIST_DATASET or _ctx.get('dataset') or globals().get('PERSIST_DATASET','')
    _j=choose_journal_for_cell('stage3',_work,_ref,STAGE3_RESTORE_MANIFEST,STAGE3_EXPECTED_TASK_ID,STAGE3_REMOTE_VERIFY_WAIT_SECONDS,STAGE3_REMOTE_COMMAND_TIMEOUT_SECONDS,_ctx.get('dataset',''),_ctx.get('manifest',''),_ctx.get('task_id',''))
    try:
        TASK_ID=_j.p['task_id'];_OAS_V1_CONTEXT=dict(task_id=TASK_ID,dataset=_j.p['dataset'] or '',manifest=str(_j.path),work=str(_work))
        print('V1_PROGRESS_PATH:',_j.path,flush=True)
        if _j.p['plan']['use_lp']:
            _state=execute_stage(_j,'stage3',root=_OAS_ROOT,work=_work,input_root='/kaggle/input',monitor=STAGE3_MONITOR_RESOURCES,max_new=STAGE3_MAX_NEW_SEGMENTS,max_seconds=STAGE3_MAX_RUN_SECONDS)
        else:print('V1_STAGE3_LP_SKIPPED: direct Ditto route',flush=True);_state='COMPLETE'
        if _state=='COMPLETE' and STAGE3_ASSEMBLE_FINAL:
            publish_final(_j,work=_work,output_root='/kaggle/working')
        run_status=copy.deepcopy(_j.p)
    finally:_j.close()

else:
    print('V1_STAGE3_SKIPPED: RUN_STAGE_3=False',flush=True)


RUNTIME_BUNDLE_VERIFIED: f0ddb13b6d236d1724f5bd9b8137a561405a3a22b9abb34758c0f5a771f3c6cc
RUNTIME_FILES_VERIFIED: 23
V1_REMOTE_CONNECTED: elleea/oas-checkpoints-avatar-checkpoint-continous-07 VERSION: 3
V1_PROGRESS: c930780bad42318800a803f8037d1618494c8636455fc4f572d6d11ea547507e VERSION: 3 TOTAL_SEGMENTS: 1 DONE: {'stage1': 1, 'stage2': 1, 'stage3': 0} NEXT: stage3:1
V1_PROGRESS_PATH: /kaggle/working/open-avatar-studio-v11/remote-snapshots/elleea__oas-checkpoints-avatar-checkpoint-continous-07/oas-progress.json
V1_STAGE_START: stage3 SKIPPED_COMMITTED: 0 REMAINING: 1
V1_SEGMENT_START: stage3 1 / 1
[RESOURCE] 05:04:08 stage3 RAM_GiB: 3.11 WORKING_GiB: 1.76 LIMIT_GiB: 30.0 GPU_MiB: [] EVENTS: low 0; high 0; max 0; oom 0; oom_kill 0; oom_group_kill 0
CHECKPOINT_RESTORED: c81b3b2ec90f8d7e89c93496e200d26bd01262062cbda07a00da22993e6a3fab
TASK_ID: c81b3b2ec90f8d7e89c93496e200d26bd01262062cbda07a00da22993e6a3fab
COMPLETED_STAGES: ['stage1', 'stage2']
NEXT_STAGE: stage3
START: lp_encode
[RESOU